# Synaptic Bouton Analysis Pipeline


This notebook walks through the post-processing workflow used to characterize glutamatergic parallel fiber boutons.
Each chapter tracks a distinct analytical theme, from data curation to mutant comparisons and stability assays.


## Chapter A – Data Foundations

Chapter A assembles the datasets, cleans fluorescence traces, and prepares pooled tables that will feed every later analysis.


### A Prelude – Toolkit Orientation

The opening steps prepare the computational environment and shared constants that support every subsequent analysis task in this notebook.


### A.1 Library Imports

This cell assembles the analytical toolbox needed for the bouton study. Core scientific libraries such as NumPy, pandas, SciPy, and scikit-learn support numerical modeling, clustering, and statistical testing, while Matplotlib and Seaborn provide the visualization backbone. By centralizing these imports we ensure every downstream analysis stage—trace preprocessing, dimensionality reduction, and classification—can access the same well-defined computational environment.


In [ ]:
## Standard library imports. PCA, Clustering, Ellipse/boundary, Alpha-shape, k-NN boundary, Stability/plasticity

# Standard library imports
import json
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

# Scientific computing and data analysis
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage, set_link_color_palette
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Data visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import ListedColormap, to_hex
from matplotlib.patches import Ellipse
from matplotlib.widgets import Button

# Geometric and statistical analysis tools
import alphashape
from shapely.geometry import MultiPolygon, Point, Polygon as ShapelyPolygon
from shapely.affinity import scale as shp_scale
from shapely.prepared import prep
from statannotations.Annotator import Annotator

### A.2 Data Source Configuration

Here we define the file system layout for the complete dataset, including the Excel workbooks that store PCA features, per-trial failure metrics, and target cell annotations. Establishing these paths ensures that subsequent routines retrieve raw traces and metadata consistently, keeping the analysis reproducible regardless of where the notebook is executed.


In [ ]:
## File paths and directory structure. Failures/reliability, PCA, Clustering, Extracellular Ca²⁺, Stability/plasticity, Temporal traces

# File paths and directory structure
BASE_DIR = Path('C:/Users/Antoine.Valera/Desktop/PPR_DATA_FINAL')
PPR_FILENAME = 'summary.xlsx'           # PCA features file
PPR_TRIALS_FILENAME = 'summary_trials.xlsx'  # Per-trial failure data
TARGET_MAP_FILENAME = 'Target_WT_pooled.xlsx'  # Bouton target identity mapping
OUTPUT_DIR = BASE_DIR / 'output'

# Data filtering and analysis parameters
EXCEPTIONAL_CONDITIONS = ['Stability_After_05', 'Stability_Before_05', 'Theo_1_5Ca', 'Theo_4Ca', 'WT_Theo']
PCA_DROP_COLS = [f'AMP{i}' for i in range(3, 11)] + ['measurement', 'Condition', 'ID', 'Target', '%Fail3']
N_CLUSTERS = 5                          # Number of clusters for analysis

# Trace processing parameters
STIM_SHIFT = 0.5                        # Stimulus time offset (seconds)
CROP_END = 2.0                          # Trace duration to keep (seconds)
SAMPLE_RATE = 1000                      # Target sampling rate (Hz)
N_SAMPLES = int(CROP_END * SAMPLE_RATE) + 1
COMMON_TIME = np.linspace(0, CROP_END, N_SAMPLES)  # Standardized time vector

### A Prelude – Data Discovery Roadmap

We next organize the raw recordings and metadata that feed the bouton analysis pipeline, ensuring every condition is accounted for before processing.


### A.3 Output Logistics and Helpers

This block creates the output directory structure and introduces helper utilities for filtering valid trace files. By curating which spreadsheets qualify as bouton recordings, we avoid ingesting metadata or temporary files and maintain a clean provenance for every trace that enters the processing workflow.


In [ ]:
## Create output directory and define helper functions. Temporal traces

# Create output directory and define helper functions
OUTPUT_DIR.mkdir(exist_ok=True)

def is_bouton_file(file_path: Path) -> bool:
    """Check if Excel file contains bouton trace data (excludes metadata files)"""
    metadata_files = {PPR_FILENAME.lower(), PPR_TRIALS_FILENAME.lower(), TARGET_MAP_FILENAME.lower()}
    return (file_path.suffix.lower() == '.xlsx' and 
            not file_path.name.startswith('~$') and 
            file_path.name.lower() not in metadata_files)

def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')

### A.4 Experimental Inventory

The loop catalogues each experimental condition present in the raw data repository and enumerates the bouton trace files found within. This systematic survey provides immediate feedback on data availability and builds the foundation for condition-specific preprocessing that follows.


In [ ]:
## Discover experimental conditions and load bouton trace files. Ellipse/boundary, Temporal traces

experimental_conditions = [dir_path.name for dir_path in sorted(BASE_DIR.iterdir()) if dir_path.is_dir()]
raw_traces_data = []

for condition_name in experimental_conditions:
    condition_dir = BASE_DIR / condition_name
    bouton_files = [file_path for file_path in condition_dir.glob('*.xlsx') if is_bouton_file(file_path)]
    print(f"Processing {condition_name}: {len(bouton_files)} files")
    
    for xlsx_file in sorted(bouton_files):
        bouton_id = clean_bouton_id(xlsx_file.stem)
        trace_data = pd.read_excel(xlsx_file).apply(pd.to_numeric, errors='coerce')
        
        if trace_data.shape[1] >= 2:  # Need at least time and average columns
            time_column = trace_data.columns[-1]     # Time is last column
            average_column = trace_data.columns[-2]  # Average trace is second-to-last
            
            raw_traces_data.append({
                'ID': bouton_id,
                'Condition': condition_name,
                'Time': trace_data[time_column].tolist(),
                'Avg': trace_data[average_column].tolist()
            })

RAW_TRACES_DF = pd.DataFrame(raw_traces_data)

### A.5 Feature Matrix Assembly

This cell loads the multi-sheet Excel workbook containing precomputed bouton features and aligns them with the discovered experimental conditions. By harmonizing the feature matrices across conditions, we prepare a coherent dataset that can support pooled analyses as well as condition-specific comparisons.


In [ ]:
## Load PCA features from multi-sheet Excel file. AMP1/strength, PCA, Feature correlations

# Load PCA features from multi-sheet Excel file
pca_features_file = BASE_DIR / PPR_FILENAME
excel_data = pd.ExcelFile(pca_features_file)

# Only process conditions that have corresponding feature sheets
available_conditions = [cond for cond in experimental_conditions if cond in excel_data.sheet_names]
CONDITIONS = available_conditions

feature_dataframes = []
for condition_name in CONDITIONS:
    condition_features = pd.read_excel(pca_features_file, sheet_name=condition_name)
    
    # Standardize ID column name (handle various naming conventions)
    id_column_names = ['id', 'bouton', 'bouton_id', 'name']
    for column in condition_features.columns:
        if str(column).strip().lower() in id_column_names:
            condition_features = condition_features.rename(columns={column: 'ID'})
            break
    
    # Clean bouton IDs and add condition label
    condition_features['ID'] = condition_features['ID'].apply(
        lambda x: clean_bouton_id(str(x)) if pd.notnull(x) else x
    )
    condition_features['Condition'] = condition_name
    feature_dataframes.append(condition_features)

excel_data.close()
FEATURES_DATAFRAME = pd.concat(feature_dataframes, ignore_index=True)

# Remove specified amplitude columns from analysis
amplitude_columns_to_drop = [col for col in PCA_DROP_COLS[:8] if col in FEATURES_DATAFRAME.columns]
if amplitude_columns_to_drop:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(columns=amplitude_columns_to_drop)

### A.6 Target Identity Integration

Target identity information (Purkinje cell, interneuron, or unclassified) is imported and standardized in this step. Cleaning identifiers and storing them as categorical labels allows later projections and clustering analyses to be biologically interpretable, connecting statistical patterns back to synaptic targets.


In [ ]:
## Load bouton target identity mapping (PC/IN/UN classification). Extracellular Ca²⁺, Temporal traces

# Load bouton target identity mapping (PC/IN/UN classification)
target_mapping_file = BASE_DIR / TARGET_MAP_FILENAME
target_identity_data = pd.read_excel(target_mapping_file).iloc[:, :2].copy()
target_identity_data.columns = ['ID', 'Target']

# Clean and standardize target data
target_identity_data['ID'] = target_identity_data['ID'].apply(lambda x: clean_bouton_id(str(x).strip()))
target_identity_data['Target'] = target_identity_data['Target'].astype(str).str.strip().str.upper()
# Set invalid targets to 'UN' (undefined)
valid_targets = ['PC', 'IN', 'UN']
target_identity_data['Target'] = target_identity_data['Target'].where(
    target_identity_data['Target'].isin(valid_targets), 'UN'
)

# Merge target identities into feature data
FEATURES_DATAFRAME['ID'] = FEATURES_DATAFRAME['ID'].astype(str).str.strip()
FEATURES_DATAFRAME = FEATURES_DATAFRAME.merge(target_identity_data, on='ID', how='left')
FEATURES_DATAFRAME['Target'] = FEATURES_DATAFRAME['Target'].fillna('UN')  # Missing targets → undefined

# Data loading summary
print(f"\n=== DATA LOADING SUMMARY ===")
print(f"Conditions: {len(CONDITIONS)} ({', '.join(CONDITIONS)})")
print(f"Traces: {len(RAW_TRACES_DF)} | Features: {len(FEATURES_DATAFRAME)} | Target mappings: {len(target_identity_data)}")
print(f"Target distribution: {dict(FEATURES_DATAFRAME['Target'].value_counts())}")
print(f"✓ Successfully processed {len(raw_traces_data)} bouton files")

### A Prelude – Trace Harmonization Goals

Before modeling, we align, pad, and normalize fluorescence traces so that recordings collected with different acquisition schemes can be compared on equal footing.


### A.7 Photobleaching and Normalization Utilities

This section defines the signal-processing toolkit for fluorescence traces. Functions handle photobleaching correction, baseline stabilization, exponential fitting, and smoothing, ensuring that raw optical signals are transformed into comparable, biologically meaningful time courses before higher-level analysis.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def correct_photobleaching(
    fluorescence_signal: np.ndarray,
    time_points: np.ndarray,
    *,
    stim_window=(1.0, 1.7),
    ref_points=20,
    return_info: bool = False,
    plot: bool = False
) -> np.ndarray | tuple[np.ndarray, dict]:
    """
    Photobleaching correction with bi-exponential fitting.
    Always applies correction.
    """
    y = fluorescence_signal.copy()
    t = time_points
    
    valid = ~np.isnan(y)
    if valid.sum() < 10:
        return (y, {}) if return_info else y

    # Baseline is 10th percentile of ALL valid data
    baseline = np.nanpercentile(y[valid], 10)
    
    # Get pre-stim data (all non-NaN before stimulus window)
    stim_start = stim_window[0] 
    pre_stim_mask = (t < stim_start) & valid
    pre_stim_idx = np.where(pre_stim_mask)[0]
    
    if len(pre_stim_idx) < 5:
        return (y, {"applied": False}) if return_info else y
        
    # Get end points (last ref_points)
    valid_idx = np.where(valid)[0]
    end_idx = valid_idx[-ref_points:] if len(valid_idx) >= ref_points else valid_idx[-5:]
    
    # Decide whether to use end points
    pre_median = np.nanmedian(y[pre_stim_idx])
    end_median = np.nanmedian(y[end_idx]) 
    use_end = end_median < pre_median
    
    # Build fitting data
    if use_end:
        fit_idx = np.concatenate([pre_stim_idx, end_idx])
    else:
        fit_idx = pre_stim_idx
        
    fit_idx = np.unique(fit_idx)
    t_fit = t[fit_idx]
    y_fit_raw = y[fit_idx]
    y_fit = y_fit_raw - baseline
    
    # Bi-exponential decay: A1*exp(k1*t) + A2*exp(k2*t)
    def biexp_decay(t, A1, k1, A2, k2):
        return A1 * np.exp(k1 * t) + A2 * np.exp(k2 * t)
    
    try:
        # Initial guess for bi-exponential
        A_total = np.max(y_fit) if np.max(y_fit) > 0 else 1.0
        A1_guess = A_total * 0.7
        A2_guess = A_total * 0.3
        k1_guess = -0.1  # Fast component
        k2_guess = -0.01  # Slow component
        
        # Fit bi-exponential with constraints
        popt, pcov = curve_fit(
            biexp_decay, 
            t_fit, 
            y_fit,
            p0=[A1_guess, k1_guess, A2_guess, k2_guess],
            bounds=([0.01, -10, 0.01, -10], [1000, 0.1, 1000, 0.1]),
            maxfev=3000
        )
        A1_fit, k1_fit, A2_fit, k2_fit = popt
        
        # Check fit quality
        y_pred = biexp_decay(t_fit, A1_fit, k1_fit, A2_fit, k2_fit)
        ss_res = np.sum((y_fit - y_pred)**2)
        ss_tot = np.sum((y_fit - np.mean(y_fit))**2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
    except Exception:
        # Fallback to single exponential
        def exp_decay(t, A, k):
            return A * np.exp(k * t)
            
        A_guess = np.max(y_fit) if np.max(y_fit) > 0 else 1.0
        k_guess = -0.01
        
        try:
            popt, pcov = curve_fit(
                exp_decay, 
                t_fit, 
                y_fit,
                p0=[A_guess, k_guess],
                bounds=([0.1, -10], [1000, 0.1]),
                maxfev=2000
            )
            A_fit, k_fit = popt
            
            # Convert to bi-exp format for consistency
            A1_fit, k1_fit, A2_fit, k2_fit = A_fit, k_fit, 0, 0
            y_pred = exp_decay(t_fit, A_fit, k_fit)
            ss_res = np.sum((y_fit - y_pred)**2)
            ss_tot = np.sum((y_fit - np.mean(y_fit))**2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            
        except Exception:
            return (y, {"applied": False, "error": "Both fits failed"}) if return_info else y
    
    # Always apply correction - subtraction method
    stim_start_time = stim_window[0]
    decay_at_stim = biexp_decay(stim_start_time, A1_fit, k1_fit, A2_fit, k2_fit)
    decay_curve = biexp_decay(t, A1_fit, k1_fit, A2_fit, k2_fit)
    
    y_corrected = y.copy()
    # Subtraction correction: subtract decay trend, normalized to stimulus start
    y_corrected[valid] = y[valid] - (decay_curve[valid] - decay_at_stim)
    
    # Gain correction (commented out):
    # correction_factor = decay_curve / decay_at_stim
    # y_corrected[valid] = (y[valid] - baseline) / correction_factor[valid] + baseline
    
    if plot:
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        
        # Original trace in red (background)
        ax.plot(t, y, 'r-', alpha=0.7, linewidth=0.5, label='Original')
        
        # Corrected trace in black
        ax.plot(t, y_corrected, 'k-', linewidth=0.5, label='Corrected')
        
        # Fit of original trace in blue
        t_plot = t[valid]
        y_fit_plot = baseline + biexp_decay(t_plot, A1_fit, k1_fit, A2_fit, k2_fit)
        ax.plot(t_plot, y_fit_plot, 'b-', linewidth=1, label=f'Bi-exp fit (R²={r_squared:.3f})')
        
        ax.fill_betweenx(ax.get_ylim(), stim_window[0], stim_window[1], 
                        alpha=0.2, color='yellow')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Fluorescence')
        ax.legend(fontsize=8)
        
        plt.tight_layout()
        plt.show()
    
    info = {
        "applied": True, "A1": A1_fit, "k1": k1_fit, "A2": A2_fit, "k2": k2_fit,
        "baseline": baseline, "use_end": use_end, "r_squared": r_squared, 
        "n_fit_points": len(fit_idx)
    }
    return (y_corrected, info) if return_info else y_corrected

def process_single_trace(trace_data) -> dict:
    """
    Process individual bouton trace: time alignment, resampling, bleaching correction, ΔF/F normalization.
    Handles NaN values throughout the pipeline.
    """
    original_time = np.array(trace_data['Time'])
    original_signal = np.array(trace_data['Avg'])
    condition_name = trace_data['Condition']
    
    # Remove any infinite values and replace with NaN
    original_signal = np.where(np.isfinite(original_signal), original_signal, np.nan)
    
    # Apply time shift for specific experimental conditions
    if condition_name in EXCEPTIONAL_CONDITIONS:
        adjusted_time = original_time + STIM_SHIFT
        
        # Pad signal with NaN values if time shift creates gap at beginning
        points_before_start = np.sum(COMMON_TIME < adjusted_time[0])
        if points_before_start > 0:
            padded_signal = np.concatenate([np.full(points_before_start, np.nan), original_signal])
            adjusted_time = np.concatenate([COMMON_TIME[:points_before_start], adjusted_time])
        else:
            padded_signal = original_signal
    else:
        adjusted_time = original_time
        padded_signal = original_signal
    
    # Resample to common time base using linear interpolation
    # Only interpolate where we have valid data
    valid_mask = np.isfinite(padded_signal) & np.isfinite(adjusted_time)
    
    if np.sum(valid_mask) < 2:  # Need at least 2 points for interpolation
        resampled_signal = np.full(N_SAMPLES, np.nan)
    else:
        try:
            interpolator = interp1d(adjusted_time[valid_mask], padded_signal[valid_mask], 
                                  kind='linear', bounds_error=False, fill_value=np.nan)
            resampled_signal = interpolator(COMMON_TIME)
        except ValueError:
            # If interpolation fails, fill with NaN
            resampled_signal = np.full(N_SAMPLES, np.nan)
    
    # Ensure exact length (crop or pad to N_SAMPLES)
    if len(resampled_signal) < N_SAMPLES:
        padding_needed = N_SAMPLES - len(resampled_signal)
        resampled_signal = np.concatenate([resampled_signal, np.full(padding_needed, np.nan)])
    else:
        resampled_signal = resampled_signal[:N_SAMPLES]
    
    # Apply photobleaching correction (now handles NaN values)
    bleach_corrected_signal = correct_photobleaching(resampled_signal, COMMON_TIME)
    
    # Calculate ΔF/F using pre-stimulus baseline (0-1s)
    baseline_period_mask = (COMMON_TIME >= 0) & (COMMON_TIME < 1.0)
    baseline_fluorescence = np.nanmean(bleach_corrected_signal[baseline_period_mask])
    
    if not np.isnan(baseline_fluorescence) and baseline_fluorescence != 0:
        delta_f_over_f = (bleach_corrected_signal - baseline_fluorescence) / baseline_fluorescence
    else:
        delta_f_over_f = np.full_like(bleach_corrected_signal, np.nan)
    
    return {
        'ID': trace_data['ID'],
        'Condition': condition_name,
        'Time': COMMON_TIME,
        'Avg': delta_f_over_f
    }

### A.8 Trace Processing Pipeline

With the utilities in place, this code iterates through all boutons, applies bleaching correction, normalizes baselines, and stores metadata describing each trace. The result is a harmonized collection of time series that captures synaptic responses across experimental conditions while preserving the contextual information needed for downstream grouping.


In [ ]:
## Process all bouton traces and organize by condition. Temporal traces

all_processed_traces = []
traces_by_condition = {}

for condition_name in RAW_TRACES_DF['Condition'].unique():
    condition_traces = RAW_TRACES_DF[RAW_TRACES_DF['Condition'] == condition_name]
    
    condition_all_processed_traces = []
    for _, single_trace in condition_traces.iterrows():
        processed_trace = process_single_trace(single_trace)
        all_processed_traces.append(processed_trace)
        condition_all_processed_traces.append(processed_trace['Avg'])
    
    traces_by_condition[condition_name] = condition_all_processed_traces
    print(f"Processed {len(condition_all_processed_traces)} traces for condition: {condition_name}")

print(f"✓ Processed {len(all_processed_traces)} traces across {len(traces_by_condition)} conditions")

### A.9 Condition-Level Trace Visualization

Processed traces are visualized for every condition to sanity-check preprocessing outcomes. Overlaying individual responses reveals whether normalization, alignment, and denoising preserved the stereotyped waveform dynamics expected for each experimental group.


In [ ]:
## Plot processed traces by condition. Extracellular Ca²⁺, Temporal traces

n_conditions = len(traces_by_condition)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for plot_idx, (condition_name, condition_traces) in enumerate(traces_by_condition.items()):
    if plot_idx >= len(axes):
        break
        
    ax = axes[plot_idx]
    
    # Plot individual traces (transparent gray)
    for single_trace in condition_traces:
        ax.plot(COMMON_TIME, single_trace, alpha=0.2, color='gray', linewidth=0.5)
    
    # Plot condition average (bold black)
    condition_average = np.nanmean(condition_traces, axis=0)
    ax.plot(COMMON_TIME, condition_average, color='black', linewidth=2, label='Average')
    
    # Format subplot
    ax.set_title(f'{condition_name}\n(n={len(condition_traces)})', fontsize=10, pad=10)
    ax.set_xlim(0, CROP_END)
    ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)  # Zero line
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, linewidth=1)  # Stimulus onset
    
    # Add axis labels for edge subplots
    if plot_idx >= 5:  # Bottom row
        ax.set_xlabel('Time (s)')
    if plot_idx % 5 == 0:  # Leftmost column
        ax.set_ylabel('ΔF/F')

# Remove unused subplots
for empty_idx in range(n_conditions, len(axes)):
    fig.delaxes(axes[empty_idx])

plt.tight_layout()
plt.suptitle('Processed Bouton Traces: Aligned, Resampled, and Normalized by Condition', 
             y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Create final processed traces dataframe
NORM_TRACES_DATAFRAME = pd.DataFrame(all_processed_traces)

print(f"Time range: {COMMON_TIME[0]:.2f} - {COMMON_TIME[-1]:.2f}s ({len(COMMON_TIME)} points)")
print(f"Exceptional conditions (shifted by {STIM_SHIFT}s): {', '.join(EXCEPTIONAL_CONDITIONS)}")

### A Prelude – DataFrame Assembly Goals

The final phase of Chapter A consolidates cleaned traces and associated metadata into pooled tables that downstream statistical models can consume.


### A.10 Condition Pooling

Here we aggregate related experimental conditions into pooled datasets that reflect meaningful biological groupings. This pooling step increases statistical power for later PCA and clustering stages while retaining the ability to trace results back to the original acquisition cohorts.


In [ ]:
## Pool related experimental conditions for analysis. Extracellular Ca²⁺, Stability/plasticity, Synapsin-II / genotype

CONDITION_POOLS = {
    'WT_pooled': ['WT_Theo', 'WT_Anthime', 'WT_Theo_1scd'],
    'stability_before': ['Stability_Before', 'Stability_Before_05'],
    'stability_after': ['Stability_After', 'Stability_After_05']
}

# Add pooled conditions to features dataframe
existing_conditions = set(FEATURES_DATAFRAME['Condition'].unique())
for pool_name, source_conditions in CONDITION_POOLS.items():
    if pool_name not in existing_conditions:
        available_sources = [c for c in source_conditions if c in existing_conditions]
        if available_sources:
            pooled_data = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'].isin(available_sources)].copy()
            pooled_data['Condition'] = pool_name
            FEATURES_DATAFRAME = pd.concat([FEATURES_DATAFRAME, pooled_data], ignore_index=True)

# Extract condition-specific dataframes (keeping original names for downstream compatibility)
PCA_Data_WT_Pooled = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_pooled'].copy()
PCA_Data_WT_Theo = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_Theo'].copy()
PCA_Data_WT_Anthime = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_Anthime'].copy()
PCA_Data_SynII = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'SynII'].copy()
PCA_Data_WT_Low_Ca = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_1_5Ca'].copy()
PCA_Data_WT_High_Ca = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_4Ca'].copy()
PCA_Data_Stability_Before = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'stability_before'].copy()
PCA_Data_Stability_After = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'stability_after'].copy()

print(f"✓ {len(FEATURES_DATAFRAME)} boutons across {len(FEATURES_DATAFRAME['Condition'].unique())} conditions")
print(f"Key datasets: WT_pooled({len(PCA_Data_WT_Pooled)}), SynII({len(PCA_Data_SynII)}), 1.5Ca({len(PCA_Data_WT_Low_Ca)}), 4Ca({len(PCA_Data_WT_High_Ca)})")

The PCA will use only the columns AMP1, AMP2, all PPRs and %Fail1 and 2. All the other columns will be un-selected.

## Chapter B – Baseline Comparisons

Chapter B evaluates whether foundational amplitude and plasticity metrics align across experimental cohorts before dimensionality reduction.


### B Prelude – Baseline Comparability Questions

Chapter B interrogates whether foundational amplitude and plasticity measurements agree across cohorts prior to dimensionality reduction.


#### B Focus – Distribution Diagnostics

We begin by contrasting amplitude, failure, and variability distributions between experimental groups to flag any gross disparities.


### B.1 Distribution Comparison Setup

To test whether key synaptic parameters are comparable across conditions, we assemble a dictionary of metrics—amplitudes, failure rates, and plasticity indices—for visualization. Organizing the data in this way supports consistent histograms and boxplots for each feature.


In [ ]:
# Parameter comparison setup: histograms and boxplots
comparison_datasets = {
    'WT_Theo': PCA_Data_WT_Theo, 
    'WT_Anthime': PCA_Data_WT_Anthime, 
    'SynII': PCA_Data_SynII
}

dataset_colors = ['#1f77b4', '#2ca02c', '#ff7f0e']
color_palette = dict(zip(comparison_datasets.keys(), dataset_colors))

# Parameters to analyze and their reference values
analysis_parameters = ['PPR2/1', 'AMP1', 'AMP2', 'STD_baseline', '%Fail1']
reference_values = {'PPR2/1': 1.0}  # Theoretical no-facilitation line

# Statistical comparisons to perform
pairwise_comparisons = [("WT_Theo", "SynII"), ("WT_Theo", "WT_Anthime"), ("WT_Anthime", "SynII")]

### B.2 Amplitude and Plasticity Diagnostics

Using the prepared datasets, this cell renders paired histograms and boxplots that contrast amplitude, paired-pulse ratios, and failure fractions across groups. The accompanying statistical annotations help determine whether experimental cohorts can be legitimately compared or require condition-specific treatment.


In [ ]:
# Create parameter comparison plots with statistical analysis
fig, axes = plt.subplots(5, 2, figsize=(15, 20))
statistical_results = []

for param_idx, parameter_name in enumerate(analysis_parameters):
    # Extract valid data for each dataset
    parameter_data = {}
    for dataset_name, dataset_df in comparison_datasets.items():
        if parameter_name in dataset_df.columns:
            clean_data = dataset_df[parameter_name].dropna()
            if len(clean_data) > 0:
                parameter_data[dataset_name] = clean_data
    
    # Skip if insufficient data
    if len(parameter_data) < 2 or sum(len(data) for data in parameter_data.values()) < 10:
        statistical_results.append(f"[SKIP] {parameter_name}: insufficient data")
        axes[param_idx, 0].text(0.5, 0.5, f'No data for {parameter_name}', 
                               ha='center', va='center', transform=axes[param_idx, 0].transAxes)
        axes[param_idx, 1].axis('off')
        continue
    
    # Create histogram (left panel)
    ax_histogram = axes[param_idx, 0]
    all_parameter_values = np.concatenate([data.values for data in parameter_data.values()])
    histogram_bins = np.linspace(all_parameter_values.min(), all_parameter_values.max(), 21)
    
    for dataset_name, dataset_values in parameter_data.items():
        transparency = 0.35 if dataset_name == 'SynII' else 0.65
        ax_histogram.hist(dataset_values, bins=histogram_bins, alpha=transparency, 
                         label=dataset_name, color=color_palette[dataset_name], 
                         density=True, edgecolor='black')
    
    ax_histogram.set_xlabel(parameter_name)
    ax_histogram.set_ylabel('Probability Density')
    ax_histogram.set_title(f'{parameter_name} Distribution')
    ax_histogram.legend(fontsize=8)
    
    # Add reference line if specified
    if parameter_name in reference_values:
        ax_histogram.axvline(reference_values[parameter_name], color='gray', linestyle='--', alpha=0.7)
    
    # Create boxplot with individual points (right panel)
    combined_data_list = []
    for dataset_name, dataset_values in parameter_data.items():
        combined_data_list.append(pd.DataFrame({
            parameter_name: dataset_values, 
            'Condition': dataset_name
        }))
    combined_parameter_df = pd.concat(combined_data_list, ignore_index=True)
    
    ax_boxplot = axes[param_idx, 1]
    sns.boxplot(data=combined_parameter_df, x='Condition', y=parameter_name, 
            ax=ax_boxplot, hue='Condition', palette=color_palette, legend=False)
    sns.stripplot(data=combined_parameter_df, x='Condition', y=parameter_name, 
                 ax=ax_boxplot, color='black', size=3, alpha=0.6)
    
    ax_boxplot.set_title(f'{parameter_name} by Condition')
    ax_boxplot.tick_params(axis='x', rotation=30)
    
    # Add reference line to boxplot
    if parameter_name in reference_values:
        ax_boxplot.axhline(reference_values[parameter_name], color='gray', linestyle='--', alpha=0.7)
    
    # Statistical annotations and tests
    try:
        # Add pairwise comparison annotations
        stats_annotator = Annotator(ax_boxplot, pairwise_comparisons, 
                                   data=combined_parameter_df, x='Condition', y=parameter_name)
        stats_annotator.configure(test='Mann-Whitney', text_format='star', loc='outside',
                                 comparisons_correction='bonferroni', show_test_name=False)
        stats_annotator.apply_and_annotate()
        
        # Overall group comparison (Kruskal-Wallis)
        group_data = [parameter_data[name] for name in comparison_datasets.keys() if name in parameter_data]
        kruskal_h, kruskal_p = stats.kruskal(*group_data)
        statistical_results.append(f"{parameter_name}: Kruskal-Wallis H={kruskal_h:.3f}, p={kruskal_p:.4g}")
        
    except Exception as error:
        statistical_results.append(f"[ERROR] {parameter_name}: {error}")

plt.tight_layout()
plt.show()

# Save results
output_filename_base = OUTPUT_DIR / "parameter_comparison"
plt.savefig(f"{output_filename_base}.pdf", dpi=300, bbox_inches='tight')

# Save statistical summary
with open(f"{output_filename_base}_statistics.txt", "w") as stats_file:
    stats_file.write("Parameter Comparison Statistical Results\n")
    stats_file.write("=" * 50 + "\n\n")
    for result in statistical_results:
        stats_file.write(f"{result}\n")
        print(result)

print(f"✓ Saved analysis to {output_filename_base}")

#### B Focus – Paired-Pulse Averages

Comparing mean PPR waveforms reveals whether facilitation and depression motifs are conserved or condition-specific before clustering.


### B.3 Paired-Pulse Response Profiles

Here we overlay the averaged PPR trajectories for each condition to inspect facilitation and depression patterns across ten stimulus pulses. This comparison highlights whether short-term plasticity motifs differ significantly between cohorts before entering dimensionality reduction.


In [ ]:
# PPR profile analysis: compare facilitation patterns across conditions

def extract_ppr_profile(condition_dataframe, max_pulse_number=10):
    """Extract PPR profile with means and standard errors for plotting."""
    # Find available PPR columns (PPR2/1, PPR3/1, etc.)
    ppr_column_names = [f'PPR{pulse_num}/1' for pulse_num in range(2, max_pulse_number+1) 
                        if f'PPR{pulse_num}/1' in condition_dataframe.columns]
    
    # PPR1/1 = 1.0 by definition, then calculate means for other ratios
    ppr_means = [1.0] + condition_dataframe[ppr_column_names].mean().tolist()
    ppr_standard_errors = [0.0] + condition_dataframe[ppr_column_names].sem().tolist()
    total_pulses = len(ppr_column_names) + 1
    
    return ppr_means, ppr_standard_errors, total_pulses

# Configure conditions for comparison
condition_configs = [
    ('WT_Theo', PCA_Data_WT_Theo, '#1f77b4', 'o'),        # Blue circles
    ('WT_Anthime', PCA_Data_WT_Anthime, '#2ca02c', 's'),  # Green squares  
    ('WT_pooled', PCA_Data_WT_Pooled, '#d62728', 'D'),    # Red diamonds
    ('SynII', PCA_Data_SynII, '#ff7f0e', '^')             # Orange triangles
]

plt.figure(figsize=(10, 6))

# Plot PPR profiles for each condition
for condition_name, condition_data, plot_color, marker_style in condition_configs:
    if condition_data.empty:
        continue
    
    ppr_means, ppr_errors, num_pulses = extract_ppr_profile(condition_data)
    pulse_numbers = list(range(1, num_pulses + 1))
    
    # Plot mean trajectory with error bands
    plt.plot(pulse_numbers, ppr_means, marker=marker_style, 
             label=f'{condition_name} (n={len(condition_data)})', 
             color=plot_color, linewidth=2, markersize=6)
    
    # Add standard error shading
    plt.fill_between(pulse_numbers, 
                     np.array(ppr_means) - np.array(ppr_errors),
                     np.array(ppr_means) + np.array(ppr_errors),
                     alpha=0.2, color=plot_color)

# Format plot
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.7, linewidth=1, 
            label='No facilitation')
plt.xlabel('Pulse Number')
plt.ylabel('PPR (A_n/A_1)')
plt.title('Paired-Pulse Ratio Profiles Across Conditions')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "ppr_profiles_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("PPR Profile Summary:")
print("-" * 50)
for condition_name, condition_data, _, _ in condition_configs:
    if condition_data.empty:
        continue
    profile_means, _, num_pulses = extract_ppr_profile(condition_data)
    
    summary_text = f"{condition_name:12} (n={len(condition_data):2d}): PPR2/1={profile_means[1]:.3f}"
    if len(profile_means) >= 10:  # Include PPR10/1 if available
        summary_text += f", PPR10/1={profile_means[9]:.3f}"
    print(summary_text)

print(f"✓ Saved PPR profile comparison to {output_file}")

#### B Focus – Normalized Trace Check

Visualizing normalized traces verifies that preprocessing preserved biologically meaningful kinetics despite differing acquisition speeds and sampling rates.


## Chapter C – PCA Preparation

Chapter C standardizes features and constructs a unified PCA model that anchors all subsequent visualizations and statistics.


### C Prelude – Feature Preparation Agenda

With baseline comparability established, Chapter C standardizes feature matrices and builds the shared PCA model used throughout the analysis.


### 1.E.a Normalization

Normalization is essential before applying PCA or UMAP because these algorithms are sensitive to the scale of the variables. If features have different units or variances, those with larger scales can dominate the analysis and distort the results. By standardizing all features to have zero mean and unit variance, we ensure that each variable contributes equally to the dimensionality reduction and clustering steps.

### C.1 Feature Sanitization for PCA

Before dimensionality reduction, we strip away non-numeric identifiers, harmonize column names, and scale each feature to unit variance. This normalization ensures that PCA captures true covariation among synaptic properties rather than artifacts of measurement scale.


In [ ]:
# Prepare datasets for PCA analysis by removing non-feature columns and scaling

def apply_standard_drops(dataframe_dict):
    """Apply standard column drops for PCA analysis to multiple dataframes."""
    return {name: df.drop(columns=[col for col in PCA_DROP_COLS if col in df.columns], errors='ignore') 
            for name, df in dataframe_dict.items() if df is not None and hasattr(df, 'columns')}

# Organize all condition dataframes
condition_dfs = {
    'PCA_Data_WT_Pooled': PCA_Data_WT_Pooled, 'PCA_Data_WT_Theo': PCA_Data_WT_Theo, 'PCA_Data_WT_Anthime': PCA_Data_WT_Anthime,
    'PCA_Data_SynII': PCA_Data_SynII, 'PCA_Data_WT_Low_Ca': PCA_Data_WT_Low_Ca, 'PCA_Data_WT_High_Ca': PCA_Data_WT_High_Ca,
    'PCA_Data_Stability_Before': PCA_Data_Stability_Before, 'PCA_Data_Stability_After': PCA_Data_Stability_After
}

# Prepare reference dataset for PCA (remove metadata columns)
WT_pooled_for_pca = PCA_Data_WT_Pooled.drop(columns=[col for col in PCA_DROP_COLS if col in PCA_Data_WT_Pooled.columns])

# Fit StandardScaler on WT_pooled reference dataset
scaler = StandardScaler()
scaled_data = {}
scaled_data['WT_pooled'] = scaler.fit_transform(WT_pooled_for_pca)

# Transform all other datasets using same scaling parameters from WT_pooled
datasets_to_scale = {
    'WT_Theo': PCA_Data_WT_Theo, 'WT_Anthime': PCA_Data_WT_Anthime, 'SynII': PCA_Data_SynII,
    'WT_1_5Ca': PCA_Data_WT_Low_Ca, 'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before, 'stab_after': PCA_Data_Stability_After
}

for dataset_name, dataset_df in datasets_to_scale.items():
    pca_features = dataset_df.drop(columns=[col for col in PCA_DROP_COLS if col in dataset_df.columns])
    scaled_data[dataset_name] = scaler.transform(pca_features)

print(f"✓ Feature columns for PCA: {list(WT_pooled_for_pca.columns)}")
print(f"✓ Removed {len(PCA_DROP_COLS)} metadata columns from each dataset")  
print(f"✓ Scaled {len(scaled_data)} datasets using WT_pooled reference parameters")

### 1.E.b - PCA construction
PCA is performed on the entire WT_pooled dataset to capture the main axes of variation present in all samples. By using all available data, the principal components reflect the global structure and variability of the dataset, ensuring that the projection is representative and comparable across different experimental groups. PCA is a good choice because it reduces dimensionality while preserving as much variance as possible, making it easier to visualize complex data and identify patterns or clusters. Additionally, PCA provides orthogonal axes (principal components) that are uncorrelated, which simplifies downstream analyses and interpretation.

### C.2 Reference PCA Model

The WT pooled dataset defines the PCA axes used throughout the study. Fitting the decomposition here and projecting every condition into the shared space creates a consistent coordinate system for cross-condition comparisons and clustering.


In [ ]:
# Fit PCA on WT_pooled reference dataset and transform all conditions

# Fit PCA model using WT_pooled as reference
pca = PCA(n_components=2)
pca_data = {}
pca_data['WT_pooled'] = pca.fit_transform(scaled_data['WT_pooled'])

# Transform all other datasets using same PCA axes from WT_pooled
for dataset_name in datasets_to_scale.keys():
    pca_data[dataset_name] = pca.transform(scaled_data[dataset_name])

# Display PCA results summary
variance_pc1, variance_pc2 = pca.explained_variance_ratio_
print(f"✓ PCA transformation complete")
print(f"✓ PC1 explains {variance_pc1:.1%} of variance, PC2 explains {variance_pc2:.1%}")
print(f"✓ Total variance explained: {variance_pc1 + variance_pc2:.1%}")
print(f"✓ Transformed {len(pca_data)} datasets using WT_pooled PCA axes")

### 1.E.c Organisation

### C.3 PCA DataFrames

Projected coordinates are packaged into tidy DataFrames alongside metadata so that plotting and statistical routines can access principal components with clear provenance. This structure underpins every visualization and classification step built on top of the PCA embedding.


In [ ]:
# Convert PCA coordinates to DataFrames for analysis and plotting

# Create DataFrames for PCA-transformed coordinates
principal_component_columns = ['PC1', 'PC2']
pca_dfs = {}

for dataset_name, pca_coordinates in pca_data.items():
    pca_dfs[f'transformed_{dataset_name}'] = pd.DataFrame(pca_coordinates, columns=principal_component_columns)

# Create PCA components table showing feature contributions to each PC
feature_names = list(WT_pooled_for_pca.columns)
df_components = pd.DataFrame(pca.components_, columns=feature_names, index=['PC1', 'PC2'])

# Display top feature contributors for each principal component
print("PCA Components Analysis (top 3 contributors per PC):")
print("-" * 50)
for pc_name in ['PC1', 'PC2']:
    top_contributing_features = df_components.loc[pc_name].abs().nlargest(3)
    feature_contributions = [f'{feature_name}({contribution:.3f})' 
                           for feature_name, contribution in top_contributing_features.items()]
    print(f"  {pc_name}: {', '.join(feature_contributions)}")

print(f"\n✓ Created {len(pca_dfs)} PCA coordinate DataFrames")
print(f"✓ PCA components table shape: {df_components.shape}")

### 1.E.d Few checks

This code segment performs correlation analysis between PCA components and original features to understand what biological characteristics drive the principal component patterns. The analysis follows a systematic approach to combine, analyze, and interpret dimensionality reduction results.

The first step creates a unified dataset by concatenating PCA-transformed coordinates with the original feature data. The pd.concat() operation joins pca_dfs['transformed_WT_pooled'] (which contains the PC1, PC2 coordinates for each sample) with PCA_Data_WT_Pooled (the original biological features) along columns (axis=1). Both DataFrames have their indices reset to ensure proper alignment during concatenation.

For interpretability, the code identifies and displays the strongest correlations for each principal component. It uses abs().nlargest(3) to find the three features with the highest absolute correlation values, regardless of positive or negative direction. The results are formatted into readable strings showing feature names with their correlation coefficients rounded to three decimal places.

### C.4 PCA–Feature Correlation Mapping

By concatenating PCA coordinates with the original feature measurements, we compute correlation coefficients that reveal which biophysical parameters drive each principal component. Identifying the strongest contributors translates abstract PCA axes back into mechanistic synaptic descriptors.


In [ ]:
# Combine PCA coordinates with original features for correlation analysis

# Create combined dataset: PCA coordinates + original features
combined_pca_FEATURES_DATAFRAME = pd.concat([
    pca_dfs['transformed_WT_pooled'].reset_index(drop=True),
    PCA_Data_WT_Pooled.reset_index(drop=True)
], axis=1)

# Calculate correlation matrix between principal components and original features
full_correlation_matrix = combined_pca_FEATURES_DATAFRAME.corr(numeric_only=True)
pc_feature_correlations = full_correlation_matrix.iloc[:2, 2:]  # Extract PC1,PC2 vs features

# Display strongest correlations for interpretability
print("Strongest PC-Feature Correlations:")
print("-" * 40)
for pc_name in ['PC1', 'PC2']:
    strongest_correlations = pc_feature_correlations.loc[pc_name].abs().nlargest(3)
    correlation_strings = [f'{feature_name}({correlation_value:.3f})' 
                          for feature_name, correlation_value in strongest_correlations.items()]
    print(f"  {pc_name}: {', '.join(correlation_strings)}")

# Store comprehensive PCA results for downstream analysis
PCA_RESULTS = {
    'pca_model': pca,                                    # Fitted PCA transformer
    'scaler': scaler,                                    # Fitted StandardScaler
    'pca_dataframes': pca_dfs,                          # PCA coordinates for all datasets
    'components': df_components,                         # Feature contributions to PCs
    'correlations': pc_feature_correlations,            # PC-feature correlation matrix
    'explained_variance': pca.explained_variance_ratio_  # Variance explained by each PC
}

print(f"\n✓ Correlation analysis complete")
print(f"✓ PCA results stored in PCA_RESULTS dictionary with {len(PCA_RESULTS)} components")

## Chapter D – Clustering and Release Phenotypes

Chapter D leverages the PCA embedding to interrogate hierarchical clusters, relate them to biological targets, and describe release properties.


### D Prelude – Visual Analytics Strategy

Chapter D leverages the PCA embedding to expose clustering structure, interpret loadings, and relate spatial patterns to biological phenotypes.


### 2.A.a - Hierarchical clustering on PCA
By performing hierarchical clustering (HC) on the PCA embedding, we can identify clusters that may not be visible in the linear PCA projection. The resulting cluster labels are then projected back onto the PCA space to visualize how these non-linear clusters are distributed in the principal component axes.

### D.1 Cluster Plot Utilities

We import specialized plotting utilities that annotate PCA scatter plots with explained variance and cluster information. These helpers streamline the visualization of complex clustering results throughout the rest of the chapter.


In [ ]:
# load plot_pca_clusters_3cm from plot_pca_nice.py, which is in the same directory
from plot_pca_nice import plot_pca_with_variance_labels

# Perform hierarchical clustering on PCA-transformed WT_pooled data

# Extract PCA coordinates for clustering
pca_coordinates = pca_data['WT_pooled']  # Shape: (n_samples, 2)

# Perform hierarchical clustering using Ward linkage method
linkage_matrix = linkage(pca_coordinates, method='ward')
cluster_assignments = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')

# Add cluster labels to original dataframe
PCA_Data_WT_Pooled_clustered = PCA_Data_WT_Pooled.copy()
PCA_Data_WT_Pooled_clustered['HC_Cluster'] = cluster_assignments

# Visualize clusters in PCA space using plot_pca_nice
pc1_variance = PCA_RESULTS["explained_variance"][0]
pc2_variance = PCA_RESULTS["explained_variance"][1]

fig, ax = plot_pca_with_variance_labels(
    x_data=pca_coordinates[:, 0],
    y_data=pca_coordinates[:, 1],
    cluster_labels=cluster_assignments,
    pc1_variance=pc1_variance,
    pc2_variance=pc2_variance,
    output_dir=OUTPUT_DIR,
    filename="hierarchical_clustering_pca"
)

plt.show()

# Display cluster distribution statistics
print("Hierarchical Clustering Results:")
print("-" * 40)
total_samples = len(cluster_assignments)

for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_size = np.sum(cluster_assignments == cluster_id)
    cluster_percentage = (cluster_size / total_samples) * 100
    print(f"Cluster {cluster_id}: {cluster_size:2d} samples ({cluster_percentage:4.1f}%)")

print(f"\nTotal: {total_samples} samples distributed across {N_CLUSTERS} clusters")
print(f"✓ Saved clustering visualization using plot_pca_nice function")

### 2.A.b Dendrogram 

A dendrogram is plotted here to visualize the hierarchical relationships between samples based on their similarity in the consensus UMAP embedding. This tree-like diagram allows us to see how clusters are formed at different levels of similarity, identify distinct groups, and assess the structure and separation of clusters. It provides an intuitive way to explore the results of hierarchical clustering and supports the selection of cluster boundaries for downstream analysis.

### D.2 Hierarchical Dendrogram

Using the PCA coordinates, this code reconstructs the hierarchical clustering tree to visualize how boutons group across linkage distances. The dendrogram exposes nested relationships among boutons that complement the 2D PCA projection.


In [ ]:
# Create dendrogram visualization of hierarchical clustering

# Set up colormap for dendrogram branches (updated for matplotlib compatibility)
dendrogram_colormap = plt.get_cmap('Set2')
cluster_hex_colors = [to_hex(dendrogram_colormap(i)) for i in range(N_CLUSTERS)]
set_link_color_palette(cluster_hex_colors)

# Create sample labels for dendrogram leaves
try:
    sample_labels = PCA_Data_WT_Pooled.index.tolist()
except AttributeError:
    sample_labels = [f'Sample_{i+1}' for i in range(len(pca_coordinates))]

# Calculate clustering threshold for specified number of clusters
clustering_threshold = linkage_matrix[-N_CLUSTERS+1, 2]

# Generate dendrogram plot
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, 
          color_threshold=clustering_threshold,
          labels=sample_labels,
          leaf_rotation=90,
          leaf_font_size=8)

plt.title(f'Hierarchical Clustering Dendrogram (k={N_CLUSTERS})')
plt.xlabel('Samples')
plt.ylabel('Ward Distance')
plt.tight_layout()

# Save dendrogram
output_file = OUTPUT_DIR / "hierarchical_clustering_dendrogram.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display clustering information
print(f"Clustering threshold for {N_CLUSTERS} clusters: {clustering_threshold:.3f}")
print(f"Dendrogram branches colored by cluster membership")
print(f"✓ Saved dendrogram to {output_file}")

#### D Insight – Why Use a Biplot?

The forthcoming biplot synthesizes sample positions and variable loadings in PCA space, allowing us to interpret how individual features sculpt the observed bouton distribution.


### D.3 PCA Correlation Circle

A correlation circle is generated to display how each feature loads onto the first two principal components. This biplot view clarifies which synaptic properties pull samples along specific PCA axes and aids in interpreting cluster separation.


In [ ]:
# Create PCA correlation circle (biplot) showing feature contributions to principal components

# Extract correlation coefficients between PCs and original features
feature_pc_correlations = PCA_RESULTS['correlations'].T.values  # Shape: (n_features, 2)
scaling_factor = 1.0
correlation_vectors = feature_pc_correlations * scaling_factor

# Create correlation circle visualization
fig, ax = plt.subplots(figsize=(8, 8))

# Add reference elements: axes and unit circle
ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
unit_circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='-', alpha=0.5)
ax.add_patch(unit_circle)

# Plot correlation vectors for each feature
feature_names = list(WT_pooled_for_pca.columns)
for feature_idx, feature_name in enumerate(feature_names):
    pc1_correlation, pc2_correlation = correlation_vectors[feature_idx, 0], correlation_vectors[feature_idx, 1]
    
    # Draw correlation vector as arrow
    ax.arrow(0, 0, pc1_correlation, pc2_correlation, 
             color='darkred', alpha=0.8, 
             head_width=0.03, head_length=0.05, 
             length_includes_head=True, linewidth=1.5)
    
    # Position feature label outside the arrow tip
    label_x_position = pc1_correlation * 1.1
    label_y_position = pc2_correlation * 1.1
    ax.text(label_x_position, label_y_position, feature_name, 
            ha='center', va='center', fontsize=10, weight='bold', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Format plot with variance information
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
total_variance = pc1_variance + pc2_variance

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.set_title(f'PCA Correlation Circle\n({total_variance:.1%} total variance explained)')
ax.grid(True, alpha=0.2)

plt.tight_layout()

# Save correlation circle
output_file = OUTPUT_DIR / "pca_correlation_circle.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display strongest feature correlations for biological interpretation
print("Strongest Feature-PC Correlations:")
print("=" * 45)
pc_feature_correlations = PCA_RESULTS['correlations']

for pc_name in ['PC1', 'PC2']:
    strongest_features = pc_feature_correlations.loc[pc_name].abs().nlargest(5)
    print(f"\n{pc_name} (strongest contributors):")
    
    for feature_name, correlation_magnitude in strongest_features.items():
        correlation_value = pc_feature_correlations.loc[pc_name, feature_name]
        correlation_direction = "+" if correlation_value > 0 else "-"
        print(f"  {correlation_direction} {feature_name}: {correlation_magnitude:.3f}")

print(f"\n✓ Saved correlation circle to {output_file}")

#### D Control – WT Consistency Check

Overlaying Anthime's recordings on the WT manifold serves as a control to ensure cross-laboratory datasets align within the common PCA frame.


### D.4 WT Reference Check

To validate that Anthime's WT dataset aligns with the pooled WT reference, we overlay both cohorts in PCA space. This control confirms that lab-to-lab differences do not distort the shared coordinate system.


In [ ]:
# Compare WT Pooled and WT Anthime datasets in PCA space

# Extract PCA coordinates for comparison datasets
wt_pooled_coordinates = np.asarray(pca_data['WT_pooled'])
wt_anthime_coordinates = np.asarray(pca_data['WT_Anthime'])

# Create PCA comparison plot
plt.figure(figsize=(8, 6))

# Plot WT Pooled dataset
plt.scatter(wt_pooled_coordinates[:, 0], wt_pooled_coordinates[:, 1], 
           marker='o', s=50, c='lightblue', alpha=0.7, 
           label=f'WT Pooled (n={len(wt_pooled_coordinates)})')

# Plot WT Anthime dataset  
plt.scatter(wt_anthime_coordinates[:, 0], wt_anthime_coordinates[:, 1], 
           marker='d', s=50, c='magenta', alpha=0.7,
           label=f'WT Anthime (n={len(wt_anthime_coordinates)})')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.title('PCA Projection: WT Pooled vs WT Anthime')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "pca_wt_pooled_vs_anthime.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ PCA comparison plot saved to {output_file}")
print(f"✓ WT Pooled: {len(wt_pooled_coordinates)} samples")
print(f"✓ WT Anthime: {len(wt_anthime_coordinates)} samples")

#### D Overlay – Target Labels in PCA

Color-coding Purkinje and interneuron boutons highlights whether target identity alone can explain structure within the PCA projection.


### D.5 Target Projection

By coloring the PCA scatter with Purkinje versus interneuron labels, we inspect whether synaptic target identity explains variance captured by the first components. This biological overlay links statistical clusters back to anatomical classes.


In [ ]:
# Visualize PCA space colored by target cell type (Purkinje Cells vs Interneurons)

# Extract target cell type labels from dataframe
target_cell_types = PCA_Data_WT_Pooled['Target'].values
pc_cell_mask = target_cell_types == 'PC'  # Purkinje Cells
in_cell_mask = target_cell_types == 'IN'  # Interneurons
un_cell_mask = target_cell_types == 'UN'  # Undefined

plt.figure(figsize=(8, 6))

# Plot different target types with distinct markers and colors
if np.any(pc_cell_mask):
    plt.scatter(pca_coordinates[pc_cell_mask, 0], pca_coordinates[pc_cell_mask, 1], 
               c='red', marker='o', s=100, alpha=0.8,
               label=f'Purkinje Cells (n={np.sum(pc_cell_mask)})')

if np.any(in_cell_mask):
    plt.scatter(pca_coordinates[in_cell_mask, 0], pca_coordinates[in_cell_mask, 1], 
               c='blue', marker='^', s=100, alpha=0.8,
               label=f'Interneurons (n={np.sum(in_cell_mask)})')

if np.any(un_cell_mask):
    plt.scatter(pca_coordinates[un_cell_mask, 0], pca_coordinates[un_cell_mask, 1], 
               c='gray', marker='s', s=60, alpha=0.6,
               label=f'Undefined (n={np.sum(un_cell_mask)})')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.title('PCA Space by Target Cell Type')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "pca_target_cell_types.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("Target Cell Type Distribution:")
print("-" * 35)
print(f"Purkinje Cells (PC): {np.sum(pc_cell_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_cell_mask):2d} samples") 
print(f"Undefined (UN):      {np.sum(un_cell_mask):2d} samples")
print(f"Total:               {len(target_cell_types):2d} samples")
print(f"✓ Saved target visualization to {output_file}")

### D.6 Target-Aligned Trace Summaries

Average traces for Purkinje and interneuron boutons are contrasted here with consistent color assignments. Visualizing response dynamics per target type reveals how physiology underpins spatial separation in PCA space.


In [ ]:
# PC and IN trace plotting with fixed target colors (PC=red, IN=blue)

def plot_target_traces_fixed_colors(target_data, target_name, target_color, trace_lookup, common_time):
    """Plot traces for specific target type with fixed color scheme."""
    
    if len(target_data) == 0:
        print(f"No {target_name} data available")
        return
    
    # Collect all traces for this target type
    all_traces = []
    valid_traces = 0
    
    for _, row in target_data.iterrows():
        bouton_id = row['ID']
        if bouton_id in trace_lookup:
            trace_values = trace_lookup[bouton_id]['Avg']
            all_traces.append(trace_values)
            valid_traces += 1
    
    if not all_traces:
        print(f"No trace data found for {target_name}")
        return
    
    # Create plot
    plt.figure(figsize=(10, 6))
    
    # Plot individual traces in target color with transparency
    for trace_values in all_traces:
        plt.plot(common_time, trace_values, color=target_color, 
                alpha=0.3, linewidth=0.8)
    
    # Plot overall average in same color but bold
    overall_average = np.nanmean(all_traces, axis=0)
    plt.plot(common_time, overall_average, color=target_color, linewidth=4, 
            label=f'{target_name} Average (n={len(all_traces)})')
    
    # Format plot
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
    plt.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus onset')
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.title(f'{target_name} Traces (Individual + Average)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save plot
    output_file = OUTPUT_DIR / f"{target_name.lower()}_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ {target_name} traces: {len(all_traces)} boutons")
    return len(all_traces)

# Create trace lookup from resampled data
resampled_trace_lookup = {}
for _, trace_row in NORM_TRACES_DATAFRAME.iterrows():
    resampled_trace_lookup[trace_row['ID']] = {
        'Time': trace_row['Time'],
        'Avg': trace_row['Avg']
    }

# Extract PC and IN data
pc_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'PC'].copy()
in_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'IN'].copy()

# Plot with fixed colors
pc_count = plot_target_traces_fixed_colors(pc_data, 'PC', 'red', resampled_trace_lookup, COMMON_TIME)
in_count = plot_target_traces_fixed_colors(in_data, 'IN', 'blue', resampled_trace_lookup, COMMON_TIME)

### D.7 Target Distance Metrics

Beyond visualization, we compute the median PCA distance between Purkinje and interneuron groups. Quantifying this separation gauges how distinctly the two synapse types occupy the reduced-dimensional manifold.


In [ ]:
# Enhanced PCA visualization with median distance between PC and IN groups

# Extract target cell type labels and coordinates
target_cell_types = PCA_Data_WT_Pooled['Target'].values
pc_cell_mask = target_cell_types == 'PC'
in_cell_mask = target_cell_types == 'IN'
un_cell_mask = target_cell_types == 'UN'

# Calculate medians for PC and IN groups
pc_coordinates = pca_coordinates[pc_cell_mask]
in_coordinates = pca_coordinates[in_cell_mask]

pc_median = np.median(pc_coordinates, axis=0) if len(pc_coordinates) > 0 else None
in_median = np.median(in_coordinates, axis=0) if len(in_coordinates) > 0 else None

# Calculate distance between medians
if pc_median is not None and in_median is not None:
    median_distance = np.linalg.norm(pc_median - in_median)
else:
    median_distance = None

plt.figure(figsize=(10, 8))

# Plot individual data points
if np.any(pc_cell_mask):
    plt.scatter(pca_coordinates[pc_cell_mask, 0], pca_coordinates[pc_cell_mask, 1], 
               c='red', marker='o', s=80, alpha=0.7, edgecolors='darkred', linewidth=0.5,
               label=f'Purkinje Cells (n={np.sum(pc_cell_mask)})')

if np.any(in_cell_mask):
    plt.scatter(pca_coordinates[in_cell_mask, 0], pca_coordinates[in_cell_mask, 1], 
               c='blue', marker='^', s=80, alpha=0.7, edgecolors='darkblue', linewidth=0.5,
               label=f'Interneurons (n={np.sum(in_cell_mask)})')

if np.any(un_cell_mask):
    plt.scatter(pca_coordinates[un_cell_mask, 0], pca_coordinates[un_cell_mask, 1], 
               c='gray', marker='s', s=60, alpha=0.6, edgecolors='black', linewidth=0.5,
               label=f'Undefined (n={np.sum(un_cell_mask)})')

# Plot medians and distance line
if pc_median is not None and in_median is not None:
    # Plot median points
    plt.scatter(pc_median[0], pc_median[1], c='darkred', marker='X', s=200, 
               label='PC Median', edgecolors='black', linewidth=2)
    plt.scatter(in_median[0], in_median[1], c='darkblue', marker='X', s=200, 
               label='IN Median', edgecolors='black', linewidth=2)
    
    # Draw line between medians
    plt.plot([pc_median[0], in_median[0]], [pc_median[1], in_median[1]], 
             color='black', linestyle='--', linewidth=2, alpha=0.8, 
             label=f'Median Distance: {median_distance:.3f}')
    
    # Annotate distance
    midpoint_x = (pc_median[0] + in_median[0]) / 2
    midpoint_y = (pc_median[1] + in_median[1]) / 2
    plt.annotate(f'd = {median_distance:.3f}', 
                xy=(midpoint_x, midpoint_y), 
                xytext=(midpoint_x + 0.5, midpoint_y + 0.5),
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.title('PCA Space: PC vs IN with Median Distance Analysis')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "pca_pc_in_median_distance.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("=== PC vs IN SEPARATION ANALYSIS ===")
print(f"Purkinje Cells (PC): {np.sum(pc_cell_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_cell_mask):2d} samples")
print(f"Undefined (UN):      {np.sum(un_cell_mask):2d} samples")



if pc_median is not None and in_median is not None:
    print(f"PC median coordinates: ({pc_median[0]:.3f}, {pc_median[1]:.3f})")
    print(f"IN median coordinates: ({in_median[0]:.3f}, {in_median[1]:.3f})")
    print(f"Median-to-median distance: {median_distance:.3f}")
else:
    print("Cannot calculate median distance - insufficient data")

print(f"✓ Saved enhanced PCA plot to {output_file}")

#### D Highlight – High-Amplitude Boutons

Spotlighting the strongest WT boutons reveals whether exceptional release strength defines a distinct region of PCA space.


### D.8 High-Amplitude Bouton Identification

This cell pinpoints WT boutons whose initial EPSC amplitudes exceed the largest SynII response. Flagging these outliers enables targeted inspection of whether unusually strong WT boutons cluster together or remain dispersed.


In [ ]:
# Compare WT and SynII amplitude profiles

# Extract SynII trace data and find amplitude threshold
synii_trace_data = []
synii_max_amplitudes = []

# loop through NORM_TRACES_DATAFRAME that have 'Condition' == 'SynII'

for _, row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == 'SynII'].iterrows():
    bouton_id = row['ID']
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        synii_trace_data.append((bouton_id, trace_values))
        
        # Find maximum absolute amplitude for this trace
        if len(trace_values) > 0:
            max_amp = np.nanmax(np.abs(trace_values))
            synii_max_amplitudes.append(max_amp)

# Calculate SynII amplitude threshold (second highest to avoid outliers)
if len(synii_max_amplitudes) >= 2:
    synii_amplitude_threshold = pd.Series(synii_max_amplitudes).nlargest(2).iloc[-1]
    print(f"SynII amplitude threshold (2nd highest): {synii_amplitude_threshold:.3f}")
else:
    synii_amplitude_threshold = np.max(synii_max_amplitudes) if synii_max_amplitudes else 0
    print(f"SynII amplitude threshold (max): {synii_amplitude_threshold:.3f}")

# Extract WT pooled trace data and identify high-amplitude traces
wt_trace_data = []
wt_high_amplitude_traces = []
wt_regular_traces = []

# loop through NORM_TRACES_DATAFRAME that have 'Condition' == 'WT_Anthime' or 'WT_Theo' or 'WT_Theo_1scd'
for _, row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'].isin(['WT_Anthime', 'WT_Theo', 'WT_Theo_1scd'])].iterrows():
    bouton_id = row['ID']
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        wt_trace_data.append((bouton_id, trace_values))
        
        # Check if this trace exceeds SynII threshold
        if len(trace_values) > 0:
            max_amp = np.nanmax(np.abs(trace_values))
            if max_amp > synii_amplitude_threshold:
                wt_high_amplitude_traces.append((bouton_id, trace_values, max_amp))
            else:
                wt_regular_traces.append((bouton_id, trace_values))

print(f"WT traces above SynII threshold: {len(wt_high_amplitude_traces)}/{len(wt_trace_data)} ({len(wt_high_amplitude_traces)/len(wt_trace_data)*100:.1f}%)")

# Visualization with x-axis clipped to focus on relevant time period (0.5-2s)
print(f"Creating visualization with clipped time axis and threshold: {synii_amplitude_threshold:.3f}")

# Calculate common y-axis limits for both panels
all_trace_values = []

# Collect all SynII trace values
for _, trace_values in synii_trace_data:
    all_trace_values.extend(trace_values[~np.isnan(trace_values)])

# Collect all WT trace values
for _, trace_values in wt_trace_data:
    all_trace_values.extend(trace_values[~np.isnan(trace_values)])

# Set common y-limits with some padding
y_min = np.min(all_trace_values) * 1.1
y_max = np.max(all_trace_values) * 1.1

# Create the dual-panel plot with clipped x-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: SynII traces
ax1.set_title(f'SynII Traces (n={len(synii_trace_data)})', fontweight='bold')

synii_traces_for_avg = []
for bouton_id, trace_values in synii_trace_data:
    ax1.plot(COMMON_TIME, trace_values, color='orange', alpha=0.6, linewidth=0.8)
    synii_traces_for_avg.append(trace_values)

# SynII average
if synii_traces_for_avg:
    synii_average = np.nanmean(synii_traces_for_avg, axis=0)
    ax1.plot(COMMON_TIME, synii_average, color='darkorange', linewidth=3, label='SynII Average')

# Threshold lines
ax1.axhline(synii_amplitude_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'Threshold: {synii_amplitude_threshold:.3f}')
ax1.axhline(-synii_amplitude_threshold, color='red', linestyle='--', linewidth=2)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('ΔF/F')
ax1.set_xlim(0.5, 2.0)
ax1.set_ylim(y_min, y_max)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title('SynII Traces (Individual + Average)')

# Panel 2: WT traces with enhanced legend
ax2.set_title(f'WT Traces: {len(wt_high_amplitude_traces)}/{len(wt_trace_data)} above threshold', fontweight='bold')

# Plot WT traces below threshold
for bouton_id, trace_values in wt_regular_traces:
    ax2.plot(COMMON_TIME, trace_values, color='black', alpha=0.3, linewidth=0.8)

# Plot WT traces above threshold  
for bouton_id, trace_values, max_amp in wt_high_amplitude_traces:
    ax2.plot(COMMON_TIME, trace_values, color='red', alpha=0.7, linewidth=1.0)

# WT average
if wt_trace_data:
    wt_traces_for_avg = [trace_values for _, trace_values in wt_trace_data]
    wt_average = np.nanmean(wt_traces_for_avg, axis=0)
    ax2.plot(COMMON_TIME, wt_average, color='darkblue', linewidth=3, label='WT Average')

# Threshold lines
ax2.axhline(synii_amplitude_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'SynII Threshold')
ax2.axhline(-synii_amplitude_threshold, color='red', linestyle='--', linewidth=2)

# Add legend entries for trace types
ax2.plot([], [], color='black', alpha=0.6, linewidth=2, label=f'Below threshold (n={len(wt_regular_traces)})')
ax2.plot([], [], color='red', alpha=0.7, linewidth=2, label=f'Above threshold (n={len(wt_high_amplitude_traces)})')

ax2.set_xlabel('Time (s)')
ax2.set_ylabel('ΔF/F')
ax2.set_xlim(0.5, 2.0)
ax2.set_ylim(y_min, y_max)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axvline(1.0, color='blue', linestyle=':', alpha=0.7, linewidth=2)

plt.tight_layout()

output_file = OUTPUT_DIR / "synii_vs_wt_amplitude_comparison_clipped.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved clipped amplitude comparison to {output_file}")
print(f"✓ Time axis focused on 0.5-2.0s stimulus response period")
print(f"✓ WT superiority: {len(wt_high_amplitude_traces)}/{len(wt_trace_data)} traces ({len(wt_high_amplitude_traces)/len(wt_trace_data)*100:.1f}%) exceed SynII maximum")

### D.9 High-Amplitude Bouton Mapping

The previously identified high-amplitude WT boutons are highlighted within PCA space to see whether they define a coherent region or scatter across clusters. Their locations inform hypotheses about the mechanisms supporting exceptionally strong synapses.


In [ ]:
# Show PCA locations of high-amplitude WT traces identified in the previous cell (ignoring target type)
if wt_high_amplitude_traces:
    high_amp_ids = [bouton_id for bouton_id, _, _ in wt_high_amplitude_traces]
    high_amp_mask = PCA_Data_WT_Pooled_clustered['ID'].isin(high_amp_ids)
    high_amp_coordinates = pca_coordinates[high_amp_mask]   

    plt.figure(figsize=(8, 6)) 
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c='lightgray', marker='o', s=50, alpha=0.5, label='All WT Pooled')
    plt.scatter(high_amp_coordinates[:, 0], high_amp_coordinates[:, 1], 
               c='red', marker='o', s=80, alpha=0.9, label=f'High-Amplitude WT (n={len(high_amp_coordinates)})')
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('PCA Locations of High-Amplitude WT Traces')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    output_file = OUTPUT_DIR / "pca_high_amplitude_wt_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()





### D Prelude – Clustered Trace Summaries

Having mapped PCA space, we now translate cluster assignments back into temporal response motifs to interpret functional signatures.


### D.10 Cluster-Averaged Traces

Mean fluorescence traces are computed for each hierarchical cluster, with side-by-side comparisons for Purkinje and interneuron members. These profiles translate abstract clusters into recognizable temporal response motifs.


In [ ]:
# Generate average trace profiles by hierarchical cluster with PC/IN side-by-side comparison

# Check if required data is available
if 'PCA_Data_WT_Pooled_clustered' not in globals() or 'Target' not in PCA_Data_WT_Pooled_clustered.columns:
    print('[CLUSTER PROFILES][SKIP] Required clustered data or Target column missing.')
else:
    # Extract PC and IN data with cluster assignments
    pc_clustered_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'PC'].copy()
    in_clustered_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'IN'].copy()
    
    # Organize traces by cluster number for each target type
    pc_cluster_traces = {}
    in_cluster_traces = {}
    
    # Process PC traces by cluster
    for _, row in pc_clustered_data.iterrows():
        bouton_id = row['ID']
        cluster_id = row['HC_Cluster']
        
        if bouton_id in resampled_trace_lookup:
            trace_values = resampled_trace_lookup[bouton_id]['Avg']
            
            if cluster_id not in pc_cluster_traces:
                pc_cluster_traces[cluster_id] = []
            pc_cluster_traces[cluster_id].append(trace_values)
    
    # Process IN traces by cluster
    for _, row in in_clustered_data.iterrows():
        bouton_id = row['ID']
        cluster_id = row['HC_Cluster']
        
        if bouton_id in resampled_trace_lookup:
            trace_values = resampled_trace_lookup[bouton_id]['Avg']
            
            if cluster_id not in in_cluster_traces:
                in_cluster_traces[cluster_id] = []
            in_cluster_traces[cluster_id].append(trace_values)
    
    # Determine all cluster numbers present
    all_clusters = sorted(set(pc_cluster_traces.keys()).union(set(in_cluster_traces.keys())))
    
    if not all_clusters:
        print('[CLUSTER PROFILES][SKIP] No cluster data found.')
    else:
        # Calculate statistics for each cluster and target type
        def calculate_cluster_stats(trace_list):
            if len(trace_list) > 0:
                trace_matrix = np.column_stack(trace_list)
                mean_trace = np.nanmean(trace_matrix, axis=1)
                sem_trace = np.nanstd(trace_matrix, axis=1, ddof=1) / np.sqrt(trace_matrix.shape[1])
                return mean_trace, sem_trace, len(trace_list)
            return None, None, 0
        
        # Set up side-by-side subplot layout
        n_clusters = len(all_clusters)
        fig, axes = plt.subplots(n_clusters, 2, figsize=(16, 3*n_clusters), sharex=True, sharey=True)
        
        if n_clusters == 1:
            axes = axes.reshape(1, -1)
        
        # Calculate global y-limits for consistent scaling
        all_trace_values = []
        for cluster_id in all_clusters:
            if cluster_id in pc_cluster_traces:
                for trace in pc_cluster_traces[cluster_id]:
                    all_trace_values.extend(trace[np.isfinite(trace)])
            if cluster_id in in_cluster_traces:
                for trace in in_cluster_traces[cluster_id]:
                    all_trace_values.extend(trace[np.isfinite(trace)])
        
        if all_trace_values:
            y_padding = 0.1
            y_min = np.min(all_trace_values) - y_padding
            y_max = np.max(all_trace_values) + y_padding
        else:
            y_min, y_max = -0.5, 0.5
        
        # Plot each cluster row
        for row_idx, cluster_id in enumerate(all_clusters):
            # Left column: PC traces
            ax_pc = axes[row_idx, 0]
            if cluster_id in pc_cluster_traces:
                pc_mean, pc_sem, pc_count = calculate_cluster_stats(pc_cluster_traces[cluster_id])
                
                if pc_mean is not None:
                    valid_indices = np.isfinite(pc_mean) & np.isfinite(pc_sem)
                    
                    if np.any(valid_indices):
                        ax_pc.plot(COMMON_TIME[valid_indices], pc_mean[valid_indices], 
                                  color='red', linewidth=2.5, 
                                  label=f'PC Cluster {cluster_id} (n={pc_count})')
                        ax_pc.fill_between(COMMON_TIME[valid_indices], 
                                          (pc_mean - pc_sem)[valid_indices], 
                                          (pc_mean + pc_sem)[valid_indices], 
                                          color='red', alpha=0.2)
            
            ax_pc.set_title(f'PC Cluster {cluster_id}', fontweight='bold')
            ax_pc.set_ylabel('ΔF/F')
            ax_pc.axhline(0, color='gray', linestyle='dotted', linewidth=1, alpha=0.7)
            ax_pc.axvline(1.0, color='black', linestyle='--', alpha=0.5, linewidth=1)
            ax_pc.grid(True, alpha=0.3)
            ax_pc.set_ylim(y_min, y_max)
            
            # Add sample count annotation
            if cluster_id in pc_cluster_traces:
                ax_pc.text(0.02, 0.95, f'n={len(pc_cluster_traces[cluster_id])}', 
                          transform=ax_pc.transAxes, verticalalignment='top',
                          bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            else:
                ax_pc.text(0.5, 0.5, 'No PC data', transform=ax_pc.transAxes, 
                          ha='center', va='center', style='italic', color='gray')
            
            # Right column: IN traces
            ax_in = axes[row_idx, 1]
            if cluster_id in in_cluster_traces:
                in_mean, in_sem, in_count = calculate_cluster_stats(in_cluster_traces[cluster_id])
                
                if in_mean is not None:
                    valid_indices = np.isfinite(in_mean) & np.isfinite(in_sem)
                    
                    if np.any(valid_indices):
                        ax_in.plot(COMMON_TIME[valid_indices], in_mean[valid_indices], 
                                  color='blue', linewidth=2.5, 
                                  label=f'IN Cluster {cluster_id} (n={in_count})')
                        ax_in.fill_between(COMMON_TIME[valid_indices], 
                                          (in_mean - in_sem)[valid_indices], 
                                          (in_mean + in_sem)[valid_indices], 
                                          color='blue', alpha=0.2)
            
            ax_in.set_title(f'IN Cluster {cluster_id}', fontweight='bold')
            ax_in.axhline(0, color='gray', linestyle='dotted', linewidth=1, alpha=0.7)
            ax_in.axvline(1.0, color='black', linestyle='--', alpha=0.5, linewidth=1)
            ax_in.grid(True, alpha=0.3)
            ax_in.set_ylim(y_min, y_max)
            
            # Add sample count annotation
            if cluster_id in in_cluster_traces:
                ax_in.text(0.02, 0.95, f'n={len(in_cluster_traces[cluster_id])}', 
                          transform=ax_in.transAxes, verticalalignment='top',
                          bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            else:
                ax_in.text(0.5, 0.5, 'No IN data', transform=ax_in.transAxes, 
                          ha='center', va='center', style='italic', color='gray')
        
        # Format final plot
        for ax in axes[-1, :]:  # Bottom row x-axis labels
            ax.set_xlabel('Time (s)')
            ax.set_xlim(0, CROP_END)
        
        plt.tight_layout()
        
        # Save plot
        output_file = OUTPUT_DIR / "cluster_profiles_side_by_side.pdf"
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.show()
        
        # Summary statistics
        print(f"=== CLUSTER PROFILE COMPARISON ===")
        print(f"Clusters analyzed: {', '.join(map(str, all_clusters))}")
        
        for cluster_id in all_clusters:
            pc_count = len(pc_cluster_traces.get(cluster_id, []))
            in_count = len(in_cluster_traces.get(cluster_id, []))
            print(f"Cluster {cluster_id}: PC={pc_count} traces, IN={in_count} traces")
        
        print(f"✓ Saved side-by-side cluster profiles to {output_file}")

### D Prelude – Release Property Survey

Beyond traces, we examine amplitude and plasticity metrics across clusters to understand how release phenotypes differ among bouton classes.


#### D Metric – PPR Trajectories

Cluster-level PPR curves expose how facilitation dynamics contribute to the identities of each bouton class.


### D.11 Cluster PPR Trajectories

Paired-pulse response curves are assembled per cluster to evaluate how facilitation or depression patterns differ among bouton classes. Linking temporal plasticity to cluster identity refines our interpretation of each group.


In [ ]:
# PPR profiles by hierarchical cluster
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
x_pulses = list(range(1, len(ppr_cols)+2))
clusters = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].unique())
cmap = plt.get_cmap('Set2')

plt.figure(figsize=(10, 6))
for idx, cluster in enumerate(clusters):
    cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster]
    means = [1] + cluster_data[ppr_cols].mean().tolist()
    sems = [0] + cluster_data[ppr_cols].sem().tolist()
    color = cmap(idx)
    
    plt.plot(x_pulses, means, marker='o', label=f'Cluster {cluster} (n={len(cluster_data)})', color=color, linewidth=2)
    plt.fill_between(x_pulses, np.array(means)-np.array(sems), np.array(means)+np.array(sems), alpha=0.2, color=color)

plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.ylabel('Mean PPR (A_n/A_1)')
plt.xlabel('Pulse Number')
plt.xticks(x_pulses)
plt.title('PPR Profiles by Hierarchical Cluster')
plt.legend(loc='upper right')
plt.tight_layout()

output_file = OUTPUT_DIR / "ppr_profiles_by_cluster.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

#### D Metric – PPR2 Ratio

Comparing the second-to-first pulse amplitude ratio across clusters highlights early facilitation differences that may underpin distinct release behaviors.


### D.12 Cluster Metric Utility

Reusable helper functions are defined to produce boxplots and statistical annotations for any scalar feature across clusters. This modularity supports consistent reporting of amplitude and plasticity differences.


In [ ]:
# Reusable function for cluster boxplot analysis
def cluster_boxplot_analysis(data, column, title, output_prefix):
    """Create boxplot by cluster with statistical analysis."""
    from scipy.stats import kruskal, mannwhitneyu
    from itertools import combinations
    
    plt.figure(figsize=(8, 5))
    ax = sns.boxplot(x='HC_Cluster', y=column, hue='HC_Cluster', data=data, palette='Set2', legend=False)
    sns.stripplot(x='HC_Cluster', y=column, data=data, color='k', size=3, alpha=0.5, ax=ax)
    
    if 'PPR' in column:
        plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    
    plt.ylabel(column)
    plt.title(title)
    
    # Clean styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.get_xaxis().set_visible(False)
    
    # Create legend
    n_clusters = data['HC_Cluster'].nunique()
    handles = [plt.Line2D([0], [0], color=plt.get_cmap('Set2')(i), lw=4) for i in range(n_clusters)]
    labels = [f'Cluster {i+1}' for i in range(n_clusters)]
    plt.legend(handles, labels, loc='upper right')
    
    # Statistical tests
    clusters = sorted(data['HC_Cluster'].unique())
    data_per_cluster = {c: data.loc[data['HC_Cluster'] == c, column].dropna().values for c in clusters}
    
    try:
        kw_stat, kw_p = kruskal(*data_per_cluster.values())
    except ValueError:
        kw_stat, kw_p = float('nan'), float('nan')
    
    pairs = list(combinations(clusters, 2))
    results = []
    for a, b in pairs:
        x, y = data_per_cluster[a], data_per_cluster[b]
        try:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), float('nan')
        results.append({
            'C_A': a, 'C_B': b, 'n_A': len(x), 'n_B': len(y),
            'U_stat': stat, 'p_raw': p, 'p_corr': min(p * len(pairs), 1.0) if not np.isnan(p) else np.nan
        })
    
    results.sort(key=lambda d: d['p_corr'] if not np.isnan(d['p_corr']) else 1)
    
    # Save results
    stats_file = OUTPUT_DIR / f"{output_prefix}_statistics.txt"
    with open(stats_file, "w") as f:
        f.write(f"{column} Statistical Analysis by Cluster\n")
        f.write(f"Kruskal-Wallis: H = {kw_stat:.4f}, p = {kw_p:.6g}\n")
        f.write(f"Bonferroni correction: {len(pairs)}\n\n")
        f.write("C_A\tC_B\tnA\tnB\tU_stat\tp_raw\tp_corr\n")
        for r in results:
            f.write(f"{r['C_A']}\t{r['C_B']}\t{r['n_A']}\t{r['n_B']}\t"
                    f"{r['U_stat']:.4f}\t{r['p_raw']:.6g}\t{r['p_corr']:.6g}\n")
    
    plt.tight_layout()
    output_file = OUTPUT_DIR / f"{output_prefix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved {column} analysis to {output_file} and {stats_file}")

### D.13 Cluster-Level PPR2/1 Analysis

Applying the reusable plotting routine, we examine how the second pulse amplitude relative to the first varies across clusters. This metric probes whether early facilitation distinguishes bouton groups discovered by hierarchical clustering.


In [ ]:
# PPR2/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR2/1', 'PPR2/1 by Cluster', 'ppr2_1_cluster')

# PPR3/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR3/1', 'PPR3/1 by Cluster', 'ppr3_1_cluster')

# AMP1 analysis  
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'AMP1', 'AMP1 by Cluster', 'amp1_cluster')

# %Fail1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, '%Fail1', '%Fail1 by Cluster', 'fail1_cluster')

## Chapter E – SynII Integration and Fiber Organization

Chapter E projects SynII boutons into the WT-defined manifold, contrasts their properties, and examines how bouton classes distribute along axonal fibers.


### E Prelude – SynII in PCA Space

Chapter E introduces SynII boutons into the WT reference frame to observe how the mutant population occupies the shared manifold.


### E.1 SynII Projection

SynII bouton features are projected into the WT-derived PCA space to assess how the mutant dataset occupies the existing manifold. This shared embedding enables direct comparison between SynII and WT boutons.


In [ ]:
# Project SynII data onto WT-trained PCA space
plt.figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
           c=cluster_assignments, alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')

# Plot SynII projected data
synii_pca_coords = pca_data['SynII']
plt.scatter(synii_pca_coords[:, 0], synii_pca_coords[:, 1], 
           marker='*', s=100, c='black', alpha=0.8, 
           edgecolors='white', linewidth=1, label=f'SynII KO (n={len(synii_pca_coords)})')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.title('SynII KO Projection onto WT PCA Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "synii_pca_projection.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ Saved to {output_file}")

### E Prelude – SynII Cluster Composition

We quantify how SynII boutons distribute across WT-derived clusters to pinpoint phenotypes that expand or contract in the mutant.


### 3.B.a Barplot WT vs SynII

This analysis explores the distribution of SynII knockout data points across the clusters previously identified in wild-type (WT) samples. 

The visualization in the previous cell reveals that SynII knockout boutons have a distinct distribution pattern among the five functional classes identified in WT boutons. This suggests that the absence of Synapsin II alters specific aspects of synaptic transmission, potentially affecting some functional clusters more than others.

The bar chart compares the percentage of boutons belonging to each cluster between WT and SynII samples, highlighting shifts in the functional profile caused by the Synapsin II knockout. This comparison helps identify which specific functional classes of synaptic transmission are most affected by the loss of Synapsin II.

#### E Insight – Assigning New Boutons

Cluster attribution for SynII boutons relies on nearest-neighbor logic within PCA space, mirroring the linkage structure learned from WT data and respecting elongated or multimodal cluster shapes.


### E.2 Cluster Attribution for SynII

Using distances in PCA space, SynII boutons are assigned to the nearest WT-derived clusters. This provides a principled way to translate the WT clustering schema onto the mutant data.


In [ ]:
# Assign SynII samples to WT-derived clusters using PCA space
def assign_clusters_robust(X_existing, labels_existing, X_new, method='centroid'):
    """Assign new samples to existing clusters using specified linkage method."""
    unique_labels = np.unique(labels_existing)
    
    if method == 'centroid':
        centroids = np.vstack([X_existing[labels_existing == lbl].mean(axis=0) for lbl in unique_labels])
        distances = cdist(X_new, centroids)
        return unique_labels[np.argmin(distances, axis=1)]
    
    elif method == 'single':
        dist_matrix = np.empty((X_new.shape[0], len(unique_labels)))
        for j, lbl in enumerate(unique_labels):
            cluster_points = X_existing[labels_existing == lbl]
            dist_matrix[:, j] = np.min(cdist(X_new, cluster_points), axis=1)
        return unique_labels[np.argmin(dist_matrix, axis=1)]
    
    else:
        raise ValueError("method must be 'centroid' or 'single'")

# Assign SynII samples to WT clusters
synii_cluster_assignments = assign_clusters_robust(pca_coordinates, cluster_assignments, 
                                                   pca_data['SynII'], method='single')

print(f"✓ Assigned {len(synii_cluster_assignments)} SynII samples to WT clusters")
print(f"SynII cluster distribution: {dict(pd.Series(synii_cluster_assignments).value_counts().sort_index())}")

#### E Focus – Visual Customization

Customizing the cluster distribution plots clarifies SynII versus WT contrasts and highlights shifts in bouton class prevalence.


### E.3 Cluster Composition Comparison

Cluster membership counts for WT and SynII boutons are contrasted to reveal which bouton phenotypes expand or diminish in the mutant. The resulting bar plots offer a population-level view of SynII remodeling.


In [ ]:
# Compare cluster distributions between WT and SynII
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
synii_cluster_counts = pd.Series(synii_cluster_assignments).value_counts()

# Ensure all clusters represented
all_clusters = sorted(wt_cluster_counts.index)
synii_complete = pd.Series([synii_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)
synii_percentages = 100 * synii_complete / len(synii_cluster_assignments)

# Stacked bar plot
fig, ax = plt.subplots(figsize=(8, 6))
cluster_colormap = plt.get_cmap('Set2')
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_wt = bottom_synii = 0
for cluster_id in all_clusters:
    color = cluster_colormap(cluster_id - 1)
    wt_pct = wt_percentages.iloc[cluster_id - 1]
    synii_pct = synii_percentages.iloc[cluster_id - 1]
    
    # WT bar
    ax.bar(0, wt_pct, bar_width, bottom=bottom_wt, color=color, 
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    if wt_pct > 3:  # Only label if segment is large enough
        ax.text(0, bottom_wt + wt_pct/2, f"{wt_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9)
    bottom_wt += wt_pct
    
    # SynII bar
    ax.bar(1, synii_pct, bar_width, bottom=bottom_synii, color=color, alpha=0.7)
    if synii_pct > 3:
        ax.text(1, bottom_synii + synii_pct/2, f"{synii_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9)
    bottom_synii += synii_pct

# Add sample counts
ax.text(0, 102, f"n={len(cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(1, 102, f"n={len(synii_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: WT vs SynII')
ax.set_xticks([0, 1])
ax.set_xticklabels(['WT', 'SynII'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=cluster_colormap(i)) for i in range(len(all_clusters))]
ax.legend(handles, [f'Cluster {c}' for c in all_clusters], 
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "wt_vs_synii_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")

### E Prelude – SynII Summary Metrics

Descriptive statistics contextualize SynII amplitudes and failure probabilities before deeper comparative analyses.


### E.4 SynII Descriptive Statistics

Summary statistics for SynII amplitudes and failure rates are computed to contextualize the mutant population. Reporting mean and standard deviation for key metrics helps quantify how SynII boutons diverge from WT benchmarks.


In [ ]:
# Calculate mean ± SD for Amp1, Amp2, and %Fail1 in SynII data
print("Mean ± SD for SynII data:")
print(f"AMP1: {PCA_Data_SynII['AMP1'].mean():.3f} ± {PCA_Data_SynII['AMP1'].std():.3f}")
print(f"AMP2: {PCA_Data_SynII['AMP2'].mean():.3f} ± {PCA_Data_SynII['AMP2'].std():.3f}")
print(f"%Fail1: {PCA_Data_SynII['%Fail1'].mean():.3f} ± {PCA_Data_SynII['%Fail1'].std():.3f}")

### E Prelude – SynII vs WT Comparisons

We compare SynII bouton behavior against WT cluster archetypes to understand which synaptic motifs are retained or altered.


#### E Focus – PPR Alignment

Overlaying SynII PPR profiles with dominant WT clusters reveals how short-term plasticity is reshaped in the mutant population.


### E.5 SynII-Enriched Clusters

This analysis identifies clusters disproportionately populated by SynII boutons and extracts the associated traces. Spotlighting these groups isolates the synaptic phenotypes most affected by the SynII mutation.


In [ ]:
# Identify and analyze SynII-enriched clusters
enriched_clusters = []
for cluster_id in all_clusters:
    wt_pct = wt_percentages[cluster_id]
    synii_pct = synii_percentages[cluster_id]
    if synii_pct > wt_pct:
        enriched_clusters.append(cluster_id)
        print(f"Cluster {cluster_id}: WT={wt_pct:.1f}%, SynII={synii_pct:.1f}% (enriched)")

if enriched_clusters:
    # PPR profile comparison for enriched clusters
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
    x_pulses = list(range(1, len(ppr_cols)+2))
    
    plt.figure(figsize=(10, 6))
    
    # Plot enriched WT clusters
    for cluster_id in enriched_clusters:
        cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_id]
        means = [1] + cluster_data[ppr_cols].mean().tolist()
        sems = [0] + cluster_data[ppr_cols].sem().tolist()
        color = cluster_colormap(cluster_id-1)
        
        plt.plot(x_pulses, means, marker='o', label=f'WT Cluster {cluster_id} (n={len(cluster_data)})', 
                color=color, linewidth=2)
        plt.fill_between(x_pulses, np.array(means)-np.array(sems), np.array(means)+np.array(sems), 
                        alpha=0.2, color=color)
    
    # Add SynII profile
    synii_means = [1] + PCA_Data_SynII[ppr_cols].mean().tolist()
    synii_sems = [0] + PCA_Data_SynII[ppr_cols].sem().tolist()
    plt.plot(x_pulses, synii_means, marker='s', label=f'SynII KO (n={len(PCA_Data_SynII)})', 
            color='red', linewidth=3)
    plt.fill_between(x_pulses, np.array(synii_means)-np.array(synii_sems), 
                    np.array(synii_means)+np.array(synii_sems), alpha=0.2, color='red')
    
    plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    plt.ylabel('Mean PPR (A_n/A_1)')
    plt.xlabel('Pulse Number')
    plt.xticks(x_pulses)
    plt.title(f'PPR Profiles: SynII-Enriched WT Clusters vs SynII KO')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "synii_enriched_clusters_ppr_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved enriched clusters comparison to {output_file}")
else:
    print("No SynII-enriched clusters found")

# SynII summary statistics
print(f"\nSynII Summary Statistics:")
for param in ['AMP1', 'AMP2', '%Fail1']:
    if param in PCA_Data_SynII.columns:
        mean_val = PCA_Data_SynII[param].mean()
        std_val = PCA_Data_SynII[param].std()
        print(f"{param}: {mean_val:.3f} ± {std_val:.3f}")

#### E Focus – Trace Comparisons

Direct comparisons of SynII and WT mean traces expose kinetic differences that accompany shifts in cluster membership.


### E.6 SynII vs WT Trace Overlays

Mean traces from SynII-enriched clusters are compared against their WT counterparts to visualize how response kinetics shift in the mutant. These overlays tie population statistics back to time-domain dynamics.


In [ ]:
# Compare mean traces between SynII and WT with cluster assignments

# Ensure SynII has cluster assignments
if 'cluster_synII' not in PCA_Data_SynII.columns:
    PCA_Data_SynII['cluster_synII'] = synii_cluster_assignments

# Extract SynII and WT traces from resampled data
synii_traces = [resampled_trace_lookup[bid]['Avg'] for bid in PCA_Data_SynII['ID'] if bid in resampled_trace_lookup]
wt_traces = []

for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    bouton_id = row['ID']
    if bouton_id in resampled_trace_lookup:
        wt_traces.append(resampled_trace_lookup[bouton_id]['Avg'])

if len(synii_traces) > 0 and len(wt_traces) > 0:
    # Calculate mean and SEM for each group
    synii_matrix = np.column_stack(synii_traces)
    wt_matrix = np.column_stack(wt_traces)
    
    synii_mean = np.nanmean(synii_matrix, axis=1)
    synii_sem = np.nanstd(synii_matrix, axis=1, ddof=1) / np.sqrt(synii_matrix.shape[1])
    
    wt_mean = np.nanmean(wt_matrix, axis=1)
    wt_sem = np.nanstd(wt_matrix, axis=1, ddof=1) / np.sqrt(wt_matrix.shape[1])
    
    # Calculate y-limits for consistent scaling
    all_values = np.concatenate([
        synii_mean - synii_sem, synii_mean + synii_sem,
        wt_mean - wt_sem, wt_mean + wt_sem
    ])
    y_min, y_max = np.nanmin(all_values), np.nanmax(all_values)
    y_padding = (y_max - y_min) * 0.05
    
    # Create comparison plot with cropped time axis
    fig, (ax_synii, ax_wt) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    
    # SynII panel
    ax_synii.plot(COMMON_TIME, synii_mean, color='red', linewidth=2, 
                  label=f'SynII KO (n={len(synii_traces)})')
    ax_synii.fill_between(COMMON_TIME, synii_mean - synii_sem, synii_mean + synii_sem, 
                          color='red', alpha=0.25)
    
    ax_synii.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_synii.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_synii.set_xlim(0.5, 2.0)  # Cropped time axis
    ax_synii.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_synii.set_title('SynII KO - Average Response')
    ax_synii.set_ylabel('ΔF/F')
    ax_synii.set_xlabel('Time (s)')
    ax_synii.legend()
    ax_synii.grid(True, alpha=0.3)
    
    # WT panel
    ax_wt.plot(COMMON_TIME, wt_mean, color='black', linewidth=2, 
               label=f'WT (n={len(wt_traces)})')
    ax_wt.fill_between(COMMON_TIME, wt_mean - wt_sem, wt_mean + wt_sem, 
                       color='black', alpha=0.25)
    ax_wt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_wt.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_wt.set_xlim(0.5, 2.0)  # Cropped time axis
    ax_wt.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_wt.set_title('WT - Average Response')
    ax_wt.set_xlabel('Time (s)')
    ax_wt.legend()
    ax_wt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "synii_vs_wt_mean_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Mean trace comparison: SynII (n={len(synii_traces)}) vs WT (n={len(wt_traces)})")
    print(f"✓ Time axis focused on stimulus response period (0.5-2.0s)")
    
    # Cluster distribution summary
    if 'enriched_clusters' in locals() and enriched_clusters:
        print(f"\nSynII distribution in enriched clusters {enriched_clusters}:")
        for cluster_id in enriched_clusters:
            count = np.sum(synii_cluster_assignments == cluster_id)
            percent = 100 * count / len(synii_cluster_assignments)
            print(f"  Cluster {cluster_id}: {count} samples ({percent:.1f}%)")
    
    print(f"✓ Saved to {output_file}")
    
else:
    print("No trace data available for comparison")

### E Prelude – SynII and Target Identity

We investigate how SynII bouton positions relate to Purkinje and interneuron territories within PCA space.


### 3.E.a Syn correspondance to nonPC synapses

### E.7 SynII Confidence Ellipses

We construct ellipses around SynII boutons in PCA space to delineate the region occupied by the mutant population. This geometric boundary provides an interpretable measure of SynII variability relative to WT clusters.


In [ ]:
# Simplified SynII ellipse analysis with controllable tightness
from scipy.stats import chi2

# Ellipse tightness control (0.50 = loose, 0.95 = tight, 0.99 = very tight)
ELLIPSE_CONFIDENCE = 0.5

def fit_confidence_ellipse(points, confidence=0.95):
    """Fit confidence ellipse around points and return parameters."""
    center = points.mean(axis=0)
    cov = np.cov(points.T)
    chi2_val = chi2.ppf(confidence, df=2)
    
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0]))
    width, height = 2 * np.sqrt(chi2_val * eigenvals)
    cov_inv = np.linalg.inv(cov)
    
    return center, cov_inv, chi2_val, (width, height, angle)

def classify_points_in_ellipse(points, center, cov_inv, chi2_threshold):
    """Return boolean mask for points inside ellipse."""
    inside_mask = []
    for pt in points:
        diff = pt - center
        mahal_dist_sq = diff @ cov_inv @ diff.T
        inside_mask.append(mahal_dist_sq <= chi2_threshold)
    return np.array(inside_mask)

def plot_pca_with_ellipse(target_coords, target_ids, target_name, target_color, ellipse_params, inside_mask, expansion_factor=0):
    """Plot PCA with ellipse and inside/outside classification."""
    center, cov_inv, chi2_val, (width, height, angle) = ellipse_params
    
    plt.figure(figsize=(8, 6))
    
    # Background WT data
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], c=cluster_assignments, alpha=0.3, s=20, label='WT background')
    
    # SynII points
    plt.scatter(pca_data['SynII'][:, 0], pca_data['SynII'][:, 1], 
               marker='*', s=80, c='cyan', edgecolors='blue', label='SynII KO')
    
    # Target points inside/outside
    outside_mask = ~inside_mask
    if np.any(inside_mask):
        plt.scatter(target_coords[inside_mask, 0], target_coords[inside_mask, 1], 
                   marker='o', s=60, c=target_color, edgecolors='black', 
                   label=f'{target_name} inside (n={np.sum(inside_mask)})')
    if np.any(outside_mask):
        plt.scatter(target_coords[outside_mask, 0], target_coords[outside_mask, 1], 
                   marker='s', s=40, c=target_color, alpha=0.4, edgecolors='gray',
                   label=f'{target_name} outside (n={np.sum(outside_mask)})')
    
    # Add ellipse
    ellipse = Ellipse(center, width, height, angle=angle, facecolor='none', 
                     edgecolor='cyan', linewidth=2, linestyle='--', 
                     label=f'SynII {ELLIPSE_CONFIDENCE:.0%} ellipse')
    plt.gca().add_patch(ellipse)
    
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    title = f'SynII Ellipse vs {target_name}'
    if expansion_factor > 0:
        title += f' (Expanded {expansion_factor*100:.0f}%)'
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / f"synii_ellipse_vs_{target_name.lower()}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

def compare_traces_inside_outside(target_ids, inside_mask, target_name, target_color):
    """Compare mean traces for inside vs outside ellipse groups."""
    inside_ids = [target_ids[i] for i in range(len(target_ids)) if inside_mask[i]]
    outside_ids = [target_ids[i] for i in range(len(target_ids)) if not inside_mask[i]]
    
    # Get traces for each group
    inside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in inside_ids if bid in resampled_trace_lookup]
    outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in outside_ids if bid in resampled_trace_lookup]
    synii_traces = [resampled_trace_lookup[bid]['Avg'] for bid in PCA_Data_SynII['ID'] if bid in resampled_trace_lookup]
    
    if not (inside_traces and outside_traces and synii_traces):
        print(f"Insufficient trace data for {target_name} comparison")
        return
    
    # Calculate means and SEMs
    inside_mean = np.nanmean(inside_traces, axis=0)
    inside_sem = np.nanstd(inside_traces, axis=0, ddof=1) / np.sqrt(len(inside_traces))
    
    outside_mean = np.nanmean(outside_traces, axis=0)
    outside_sem = np.nanstd(outside_traces, axis=0, ddof=1) / np.sqrt(len(outside_traces))
    
    synii_mean = np.nanmean(synii_traces, axis=0)
    synii_sem = np.nanstd(synii_traces, axis=0, ddof=1) / np.sqrt(len(synii_traces))
    
    # Plot comparison
    plt.figure(figsize=(10, 6))
    
    # SynII reference
    # Inside ellipse
    plt.plot(COMMON_TIME, inside_mean, color=target_color, linewidth=2, 
             label=f'{target_name} inside ellipse (n={len(inside_traces)}, {len(inside_traces)/len(target_ids)*100:.1f}%)')
    plt.fill_between(COMMON_TIME, inside_mean-inside_sem, inside_mean+inside_sem, color=target_color, alpha=0.3)
    
    # Outside ellipse
    plt.plot(COMMON_TIME, outside_mean, color=target_color, linewidth=2, linestyle=':', alpha=0.7,
             label=f'{target_name} outside ellipse (n={len(outside_traces)}, {len(outside_traces)/len(target_ids)*100:.1f}%)')
    plt.fill_between(COMMON_TIME, outside_mean-outside_sem, outside_mean+outside_sem, color=target_color, alpha=0.15)
    
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    plt.xlim(0.5, 2.0)
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.title(f'{target_name} Inside vs Outside SynII Ellipse')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / f"{target_name.lower()}_inside_outside_ellipse_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ {target_name} ellipse analysis:")
    print(f"  Inside: {len(inside_traces)} traces ({len(inside_traces)/len(target_ids)*100:.1f}%)")
    print(f"  Outside: {len(outside_traces)} traces ({len(outside_traces)/len(target_ids)*100:.1f}%)")
    
    return output_file

# Main analysis
print(f"SynII ellipse analysis (confidence: {ELLIPSE_CONFIDENCE:.0%})")

# Fit ellipse around SynII points
synii_points = pca_data['SynII']
ellipse_params = fit_confidence_ellipse(synii_points, ELLIPSE_CONFIDENCE)
center, cov_inv, chi2_val = ellipse_params[:3]

# Extract target coordinates and IDs
target_cell_types = PCA_Data_WT_Pooled['Target'].values
in_cell_mask = target_cell_types == 'IN'
pc_cell_mask = target_cell_types == 'PC'

in_coordinates = pca_coordinates[in_cell_mask]
in_ids = PCA_Data_WT_Pooled.loc[in_cell_mask, 'ID'].tolist()

pc_coordinates = pca_coordinates[pc_cell_mask]
pc_ids = PCA_Data_WT_Pooled.loc[pc_cell_mask, 'ID'].tolist()

# Classify points inside/outside ellipse
if len(in_ids) > 0:
    in_inside_mask = classify_points_in_ellipse(in_coordinates, center, cov_inv, chi2_val)
    plot_pca_with_ellipse(in_coordinates, in_ids, 'IN', 'red', ellipse_params, in_inside_mask)
    compare_traces_inside_outside(in_ids, in_inside_mask, 'IN', 'red')

if len(pc_ids) > 0:
    pc_inside_mask = classify_points_in_ellipse(pc_coordinates, center, cov_inv, chi2_val)
    plot_pca_with_ellipse(pc_coordinates, pc_ids, 'PC', 'mediumseagreen', ellipse_params, pc_inside_mask)
    compare_traces_inside_outside(pc_ids, pc_inside_mask, 'PC', 'mediumseagreen')

print(f"\n✓ Analysis complete. Adjust ELLIPSE_CONFIDENCE (currently {ELLIPSE_CONFIDENCE}) to control ellipse tightness.")

### E.8 Target Composition Within SynII Space

By examining which Purkinje and interneuron boutons fall inside or outside the SynII ellipse, we evaluate whether the mutant phenotype preferentially overlaps with specific target identities.


In [ ]:
# Four-panel comparison: IN/PC inside/outside SynII ellipse traces

def plot_group_traces(ax, trace_ids, group_name, color, show_individuals=True):
    """Plot individual traces + average for a group."""
    # Get traces for this group
    group_traces = [resampled_trace_lookup[bid]['Avg'] for bid in trace_ids if bid in resampled_trace_lookup]
    
    if not group_traces:
        ax.text(0.5, 0.5, f'No traces\navailable', ha='center', va='center', 
                transform=ax.transAxes, fontsize=12, color='gray')
        ax.set_title(f'{group_name}\n(n=0)')
        return
    
    # Plot individual traces
    if show_individuals:
        for trace in group_traces:
            ax.plot(COMMON_TIME, trace, color=color, alpha=0.2, linewidth=0.5)
    
    # Calculate and plot average
    group_mean = np.nanmean(group_traces, axis=0)
    group_sem = np.nanstd(group_traces, axis=0, ddof=1) / np.sqrt(len(group_traces))
    
    ax.plot(COMMON_TIME, group_mean, color=color, linewidth=3, 
            label=f'{group_name} avg')
    ax.fill_between(COMMON_TIME, group_mean-group_sem, group_mean+group_sem, 
                    color=color, alpha=0.3)
    
    # Format subplot
    ax.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlim(0.5, 2.0)
    ax.set_title(f'{group_name}\n(n={len(group_traces)})')
    ax.grid(True, alpha=0.3)
    
    return group_mean, group_sem

# Prepare trace groups based on ellipse classification
if 'in_inside_mask' in locals() and 'pc_inside_mask' in locals():
    # Get IDs for each group
    in_inside_ids = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
    in_outside_ids = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
    pc_inside_ids = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
    pc_outside_ids = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
    
    # Create 2x2 subplot layout
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
    
    # Plot each group
    plot_group_traces(axes[0,0], in_inside_ids, 'IN Inside Ellipse', 'red')
    plot_group_traces(axes[0,1], in_outside_ids, 'IN Outside Ellipse', 'darkred')
    plot_group_traces(axes[1,0], pc_inside_ids, 'PC Inside Ellipse', 'mediumseagreen')
    plot_group_traces(axes[1,1], pc_outside_ids, 'PC Outside Ellipse', 'darkgreen')
    
    # Add common labels
    for ax in axes[-1, :]:  # Bottom row
        ax.set_xlabel('Time (s)')
    for ax in axes[:, 0]:   # Left column
        ax.set_ylabel('ΔF/F')
    
    # Add overall title
    fig.suptitle(f'Trace Analysis: Inside vs Outside SynII {ELLIPSE_CONFIDENCE:.0%} Ellipse', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "four_panel_ellipse_trace_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Summary statistics
    print(f"=== ELLIPSE TRACE ANALYSIS SUMMARY ===")
    print(f"Ellipse confidence level: {ELLIPSE_CONFIDENCE:.0%}")
    print(f"IN inside ellipse:  {len(in_inside_ids):2d} traces ({len(in_inside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
    print(f"IN outside ellipse: {len(in_outside_ids):2d} traces ({len(in_outside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
    print(f"PC inside ellipse:  {len(pc_inside_ids):2d} traces ({len(pc_inside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
    print(f"PC outside ellipse: {len(pc_outside_ids):2d} traces ({len(pc_outside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
    
    print(f"\n✓ Saved four-panel comparison to {output_file}")
    
else:
    print("Run ellipse analysis first to generate inside/outside classifications")

### E Prelude – Fiber Diversity Agenda

The analysis pivots to axonal organization, examining how bouton classes distribute along individual fibers.


#### E Focus – Fiber Class Counts

Counting how many cluster types appear per fiber assesses whether anatomical bundles host diverse functional boutons.


### E.9 Fiber-Level Diversity

Boutons are regrouped by axonal fiber to quantify how many distinct clusters appear along individual fibers. Assessing intra-fiber heterogeneity sheds light on whether structural units host multiple functional bouton classes.


In [ ]:
# Analyze cluster diversity within individual fibers (boutons from same axon)

def extract_fiber_id(bouton_id):
    """Extract fiber ID from bouton ID (first 22 characters)."""
    return str(bouton_id)[:22]

def analyze_fiber_diversity(min_boutons_per_fiber=4):
    """Analyze how clusters are distributed within individual fibers."""
    
    # Extract fiber IDs and count boutons per fiber
    fiber_data = PCA_Data_WT_Pooled_clustered.copy()
    fiber_data['Fiber_ID'] = fiber_data['ID'].apply(extract_fiber_id)
    
    # Count boutons per fiber
    fiber_bouton_counts = fiber_data.groupby('Fiber_ID').size()
    
    # Filter fibers with sufficient boutons
    valid_fibers = fiber_bouton_counts[fiber_bouton_counts >= min_boutons_per_fiber].index
    filtered_data = fiber_data[fiber_data['Fiber_ID'].isin(valid_fibers)]
    
    print(f"Fiber analysis: {len(valid_fibers)} fibers with {min_boutons_per_fiber}+ boutons")
    print(f"Total boutons analyzed: {len(filtered_data)}")
    
    return filtered_data, valid_fibers

def calculate_cluster_diversity(filtered_data):
    """Calculate number of different clusters per fiber."""
    fiber_diversity = {}
    
    for fiber_id in filtered_data['Fiber_ID'].unique():
        fiber_boutons = filtered_data[filtered_data['Fiber_ID'] == fiber_id]
        unique_clusters = fiber_boutons['HC_Cluster'].nunique()
        total_boutons = len(fiber_boutons)
        cluster_distribution = fiber_boutons['HC_Cluster'].value_counts(normalize=True)
        
        fiber_diversity[fiber_id] = {
            'num_cluster_types': unique_clusters,
            'total_boutons': total_boutons,
            'cluster_props': cluster_distribution.to_dict()
        }
    
    return fiber_diversity

# Main analysis
filtered_data, valid_fibers = analyze_fiber_diversity(min_boutons_per_fiber=4)
fiber_diversity = calculate_cluster_diversity(filtered_data)

# Organize data by diversity level
diversity_categories = {'1': [], '2': [], '3': [], '4+': []}
for fiber_id, info in fiber_diversity.items():
    num_types = info['num_cluster_types']
    category = str(num_types) if num_types <= 3 else '4+'
    diversity_categories[category].append(info)

# Calculate average cluster proportions for each diversity category
cluster_colormap = plt.get_cmap('Set2')
diversity_means = {}
diversity_counts = {}

for category, fiber_list in diversity_categories.items():
    diversity_counts[category] = len(fiber_list)
    
    if len(fiber_list) > 0:
        # Calculate mean proportion for each cluster
        cluster_means = {}
        for cluster_id in range(1, N_CLUSTERS + 1):
            proportions = [fiber['cluster_props'].get(cluster_id, 0) for fiber in fiber_list]
            cluster_means[cluster_id] = np.mean(proportions)
        diversity_means[category] = cluster_means
    else:
        diversity_means[category] = {i: 0 for i in range(1, N_CLUSTERS + 1)}

# Create stacked bar plot
fig, ax = plt.subplots(figsize=(10, 6))

diversity_order = ['1', '2', '3', '4+']
x_positions = np.arange(len(diversity_order))
bottom_values = np.zeros(len(diversity_order))

total_fibers = len(valid_fibers)

# Plot stacked bars
for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_heights = []
    for category in diversity_order:
        # Height = (fibers in category / total fibers) * 100 * average cluster proportion
        fiber_percentage = (diversity_counts[category] / total_fibers) * 100
        cluster_proportion = diversity_means[category][cluster_id]
        height = fiber_percentage * cluster_proportion
        cluster_heights.append(height)
    
    ax.bar(x_positions, cluster_heights, bottom=bottom_values, 
           color=cluster_colormap(cluster_id-1), label=f'Cluster {cluster_id}', alpha=0.8)
    bottom_values += cluster_heights

# Add fiber count labels
for i, category in enumerate(diversity_order):
    count = diversity_counts[category]
    percentage = (count / total_fibers) * 100
    ax.text(i, percentage + 1, f'n={count}\n({percentage:.1f}%)', 
            ha='center', va='bottom', fontweight='bold', fontsize=9)

# Format plot
ax.set_xlabel('Number of Cluster Types per Fiber')
ax.set_ylabel('Percentage of Fibers (%)')
ax.set_title(f'Fiber Cluster Diversity (Fibers with 4+ Boutons)\nTotal: {total_fibers} fibers')
ax.set_xticks(x_positions)
ax.set_xticklabels(diversity_order)
ax.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()

output_file = OUTPUT_DIR / "fiber_cluster_diversity_analysis.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print(f"\n=== FIBER CLUSTER DIVERSITY SUMMARY ===")
for category in diversity_order:
    count = diversity_counts[category]
    if count > 0:
        percentage = (count / total_fibers) * 100
        print(f"\nFibers with {category} cluster type(s): {count} ({percentage:.1f}%)")
        
        # Show cluster composition
        for cluster_id in range(1, N_CLUSTERS + 1):
            prop = diversity_means[category][cluster_id]
            if prop > 0.05:  # Only show clusters with >5% average proportion
                print(f"  Cluster {cluster_id}: {prop:.1%} average proportion")

# Identify most diverse fibers
diverse_fibers = [fid for fid, info in fiber_diversity.items() if info['num_cluster_types'] >= 3]
if diverse_fibers:
    print(f"\nMost diverse fibers (3+ cluster types): {len(diverse_fibers)} fibers")
    for fiber_id in diverse_fibers[:5]:  # Show first 5
        info = fiber_diversity[fiber_id]
        clusters = list(info['cluster_props'].keys())
        print(f"  {fiber_id}: {info['num_cluster_types']} clusters ({clusters})")

print(f"\n✓ Saved fiber diversity analysis to {output_file}")

#### E Focus – Representative Fiber

Inspecting a representative fiber illustrates how raw and smoothed traces capture shared waveform features across boutons.


### E.10 Representative Fiber Dynamics

For a selected fiber, raw and smoothed traces are visualized to showcase how preprocessing captures the essential synaptic waveform while reducing noise. This example grounds the fiber-level analysis in concrete data.


In [ ]:
# Analyze traces from a specific fiber (raw vs smoothed)
from scipy.signal import savgol_filter

def analyze_single_fiber(fiber_prefix, window_length=9, poly_order=2):
    """Analyze all boutons from a specific fiber."""
    
    # Find boutons from this fiber
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(fiber_prefix)]
    
    if len(fiber_boutons) == 0:
        print(f"No boutons found with prefix '{fiber_prefix}'")
        return
    
    print(f"Found {len(fiber_boutons)} boutons from fiber '{fiber_prefix}'")
    
    # Get trace data for these boutons
    fiber_traces = {}
    for _, bouton in fiber_boutons.iterrows():
        bouton_id = bouton['ID']
        if bouton_id in resampled_trace_lookup:
            trace_data = resampled_trace_lookup[bouton_id]['Avg']
            cluster_id = bouton['HC_Cluster']
            fiber_traces[bouton_id] = {
                'raw': trace_data,
                'cluster': cluster_id
            }
    
    if not fiber_traces:
        print(f"No trace data found for fiber '{fiber_prefix}'")
        return
    
    # Apply smoothing
    for bouton_id in fiber_traces:
        raw_trace = fiber_traces[bouton_id]['raw']
        
        # Adjust window length if needed
        win_len = min(window_length, len(raw_trace))
        if win_len % 2 == 0:  # Must be odd
            win_len -= 1
        if win_len < 3:
            smoothed_trace = raw_trace.copy()
        else:
            smoothed_trace = savgol_filter(raw_trace, window_length=win_len, 
                                         polyorder=min(poly_order, win_len-1), mode='interp')
        
        fiber_traces[bouton_id]['smoothed'] = smoothed_trace
    
    return fiber_traces

def plot_fiber_traces(fiber_traces, fiber_prefix):
    """Plot raw vs smoothed traces for a fiber."""
    if not fiber_traces:
        return
    
    n_boutons = len(fiber_traces)
    cluster_colormap = plt.get_cmap('Set2')
    
    # Create subplot layout (2 columns: raw, smoothed)
    fig, axes = plt.subplots(n_boutons, 2, figsize=(10, min(20, 2*n_boutons)), 
                            sharex=True, sharey=True)
    
    if n_boutons == 1:
        axes = axes.reshape(1, -1)
    
    # Calculate common y-limits
    all_values = []
    for data in fiber_traces.values():
        all_values.extend(data['raw'][np.isfinite(data['raw'])])
        all_values.extend(data['smoothed'][np.isfinite(data['smoothed'])])
    
    if all_values:
        y_min, y_max = np.min(all_values), np.max(all_values)
        y_padding = (y_max - y_min) * 0.05
        y_lims = (y_min - y_padding, y_max + y_padding)
    else:
        y_lims = (-0.5, 0.5)
    
    # Plot each bouton
    for idx, (bouton_id, data) in enumerate(fiber_traces.items()):
        cluster_id = data['cluster']
        color = cluster_colormap(cluster_id - 1)
        
        # Raw trace (left)
        ax_raw = axes[idx, 0]
        ax_raw.plot(COMMON_TIME, data['raw'], color=color, linewidth=1.2)
        ax_raw.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_raw.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_raw.set_xlim(0.5, 2.0)
        ax_raw.set_ylim(y_lims)
        ax_raw.set_ylabel('ΔF/F', fontsize=8)
        ax_raw.set_title(f'{bouton_id} (Cluster {cluster_id})\nRaw', fontsize=9, loc='left')
        ax_raw.tick_params(labelsize=7)
        
        # Smoothed trace (right)
        ax_smooth = axes[idx, 1]
        ax_smooth.plot(COMMON_TIME, data['smoothed'], color=color, linewidth=1.2)
        ax_smooth.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_smooth.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_smooth.set_xlim(0.5, 2.0)
        ax_smooth.set_ylim(y_lims)
        ax_smooth.set_title('Savitzky-Golay Smoothed', fontsize=9, loc='left')
        ax_smooth.tick_params(labelsize=7)
    
    # Add x-axis labels to bottom row
    axes[-1, 0].set_xlabel('Time (s)', fontsize=8)
    axes[-1, 1].set_xlabel('Time (s)', fontsize=8)
    
    fig.suptitle(f'Single Fiber Analysis: {fiber_prefix} (n={n_boutons})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"fiber_traces_{safe_prefix}_raw_vs_smoothed.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

def plot_fiber_ppr_profiles(fiber_boutons, fiber_prefix):
    """Plot PPR profiles for boutons from the same fiber."""
    
    # Get PPR data
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in fiber_boutons.columns]
    if not ppr_cols:
        print("No PPR columns found")
        return
    
    pulse_numbers = list(range(1, len(ppr_cols) + 2))  # Include pulse 1
    cluster_colormap = plt.get_cmap('Set2')
    
    plt.figure(figsize=(8, 5))
    
    # Plot each bouton's PPR profile
    for _, bouton in fiber_boutons.iterrows():
        cluster_id = bouton['HC_Cluster']
        color = cluster_colormap(cluster_id - 1)
        
        # Get PPR values (start with 1.0 for pulse 1)
        ppr_values = [1.0] + bouton[ppr_cols].tolist()
        
        plt.plot(pulse_numbers, ppr_values, color=color, alpha=0.8, linewidth=2, 
                marker='o', markersize=4, label=f'Cluster {cluster_id}')
    
    plt.axhline(1.0, color='gray', linestyle='--', linewidth=1)
    plt.xlabel('Pulse Number')
    plt.ylabel('PPR (A_n/A_1)')
    plt.title(f'PPR Profiles: {fiber_prefix} (n={len(fiber_boutons)})')
    plt.grid(True, alpha=0.3)
    plt.xticks(pulse_numbers)
    
    # Only show legend if multiple clusters
    if fiber_boutons['HC_Cluster'].nunique() > 1:
        plt.legend()
    
    plt.tight_layout()
    
    # Save figure
    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"fiber_ppr_{safe_prefix}_profiles.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

# Example usage - change the prefix to analyze different fibers
FIBER_PREFIX = "241212_Fibre2_PortionA_"  # Change this to your fiber of interest

# Run analysis
fiber_traces = analyze_single_fiber(FIBER_PREFIX)

if fiber_traces:
    # Plot traces
    trace_output = plot_fiber_traces(fiber_traces, FIBER_PREFIX)
    
    # Plot PPR profiles
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(FIBER_PREFIX)]
    ppr_output = plot_fiber_ppr_profiles(fiber_boutons, FIBER_PREFIX)
    
    print(f"✓ Fiber analysis complete:")
    print(f"  Traces: {trace_output}")
    print(f"  PPR profiles: {ppr_output}")
    
    # Summary statistics
    cluster_distribution = fiber_boutons['HC_Cluster'].value_counts().sort_index()
    print(f"\nFiber cluster composition:")
    for cluster_id, count in cluster_distribution.items():
        print(f"  Cluster {cluster_id}: {count} boutons")

else:
    print(f"No analysis possible for fiber '{FIBER_PREFIX}'")
    
    # Show available fiber prefixes
    available_prefixes = PCA_Data_WT_Pooled_clustered['ID'].str[:25].value_counts()
    print("\nAvailable fiber prefixes (showing top 10):")
    for prefix, count in available_prefixes.head(10).items():
        if count >= 3:  # Only show fibers with multiple boutons
            print(f"  '{prefix}': {count} boutons")

#### E Focus – SynII-Like Fibers

Classifying fibers by their cluster composition tests whether SynII-like phenotypes aggregate along specific anatomical branches.


### E.11 Fiber Classification by Cluster Composition

Fibers containing at least four boutons are categorized based on whether they predominantly express SynII-associated clusters. This classification probes whether mutant-like phenotypes segregate along specific anatomical pathways.


In [ ]:
# Classify fibers based on cluster expression patterns (using fibers with 4+ boutons)

def classify_fibers_by_clusters(filtered_data, min_boutons=4):
    """Classify fibers into functional groups based on cluster expression."""
    
    # Group clusters into functional categories (adjust these groupings as needed)
    LOW_CLUSTERS = {1, 2, 3}     # Lower-numbered clusters
    HIGH_CLUSTERS = {4, 5}       # Higher-numbered clusters
    
    # Get cluster sets for each fiber
    fiber_cluster_sets = filtered_data.groupby('Fiber_ID')['HC_Cluster'].apply(lambda x: set(x.unique()))
    
    # Classify fibers
    low_only_fibers = fiber_cluster_sets[fiber_cluster_sets.apply(
        lambda s: s.issubset(LOW_CLUSTERS) and len(s) > 0)]
    
    high_only_fibers = fiber_cluster_sets[fiber_cluster_sets.apply(
        lambda s: s.issubset(HIGH_CLUSTERS) and len(s) > 0)]
    
    # Mixed fibers (contain both low and high clusters)
    all_fiber_ids = set(fiber_cluster_sets.index)
    exclusive_fiber_ids = set(low_only_fibers.index) | set(high_only_fibers.index)
    mixed_fiber_ids = all_fiber_ids - exclusive_fiber_ids
    
    return {
        'low_only': low_only_fibers,
        'high_only': high_only_fibers, 
        'mixed': mixed_fiber_ids,
        'all_sets': fiber_cluster_sets,
        'low_clusters': LOW_CLUSTERS,
        'high_clusters': HIGH_CLUSTERS
    }

# Run classification using our filtered data from previous analysis
if 'filtered_data' in locals():
    classification = classify_fibers_by_clusters(filtered_data)
    
    # Count fibers in each category
    count_low_only = len(classification['low_only'])
    count_high_only = len(classification['high_only'])
    count_mixed = len(classification['mixed'])
    total_fibers = len(classification['all_sets'])
    
    # Calculate percentages
    pct_low_only = (count_low_only / total_fibers) * 100
    pct_high_only = (count_high_only / total_fibers) * 100
    pct_mixed = (count_mixed / total_fibers) * 100
    
    # Create visualization
    categories = ['Low Clusters\n(1,2,3) Only', 'High Clusters\n(4,5) Only', 'Mixed\nFibers']
    counts = [count_low_only, count_high_only, count_mixed]
    percentages = [pct_low_only, pct_high_only, pct_mixed]
    colors = ['#6ea740', '#d40404', '#74278B']
    
    plt.figure(figsize=(8, 6))
    bars = plt.bar(categories, percentages, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
    
    # Add count and percentage labels
    for bar, count, pct in zip(bars, counts, percentages):
        plt.text(bar.get_x() + bar.get_width()/2, pct + 1, f'n={count}\n({pct:.1f}%)', 
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    plt.ylabel('Percentage (%)')
    plt.title(f'Fiber Classification by Cluster Expression\n(Fibers with 4+ boutons, n={total_fibers})')
    plt.ylim(0, 110)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "fiber_classification_by_cluster_type.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Detailed results
    print(f"=== FIBER CLASSIFICATION RESULTS ===")
    print(f"Total fibers analyzed: {total_fibers} (with 4+ boutons)")
    print(f"\nLow clusters only (1,2,3): {count_low_only} fibers ({pct_low_only:.1f}%)")
    print(f"High clusters only (4,5): {count_high_only} fibers ({pct_high_only:.1f}%)")
    print(f"Mixed fibers: {count_mixed} fibers ({pct_mixed:.1f}%)")
    
    # Show examples
    if count_low_only > 0:
        print(f"\nExamples of low-cluster-only fibers:")
        for i, (fiber_id, clusters) in enumerate(classification['low_only'].head(3).items()):
            print(f"  {fiber_id}: clusters {sorted(list(clusters))}")
    
    if count_high_only > 0:
        print(f"\nExamples of high-cluster-only fibers:")
        for i, (fiber_id, clusters) in enumerate(classification['high_only'].head(3).items()):
            print(f"  {fiber_id}: clusters {sorted(list(clusters))}")
    
    if count_mixed > 0:
        print(f"\nExamples of mixed fibers:")
        mixed_examples = list(classification['mixed'])[:3]
        for fiber_id in mixed_examples:
            clusters = classification['all_sets'][fiber_id]
            print(f"  {fiber_id}: clusters {sorted(list(clusters))}")
    
    # Verify totals
    assert count_low_only + count_high_only + count_mixed == total_fibers, "Classification count mismatch!"
    
    print(f"\n✓ Saved classification to {output_file}")
    
    # Store results for potential further analysis
    FIBER_CLASSIFICATION = classification
    
else:
    print("Run fiber diversity analysis first to generate filtered_data")

## Chapter F – Calcium Perturbations and Stability Experiments

Chapter F explores how extracellular calcium and longitudinal manipulations reshape bouton phenotypes within the PCA framework.


### F Prelude – Calcium Projection Goals

Chapter F evaluates how extracellular calcium levels reposition boutons within the PCA embedding.


### F.1 Calcium Modulation in PCA Space

High- and low-calcium conditions are projected into the WT PCA embedding to observe how extracellular calcium reshapes bouton distributions. Visualizing these shifts indicates whether calcium availability drives distinct synaptic states.


In [ ]:
# Analyze calcium concentration effects on bouton properties in PCA space

def plot_calcium_trajectories():
    """Plot how calcium concentration changes affect PCA positioning."""
    
    plt.figure(figsize=(10, 8))
    
    # Background: WT pooled (2.5mM Ca standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=cluster_assignments, alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')
    
    # Low calcium (1.5mM) - blue triangles pointing down
    low_ca_coords = pca_data['WT_1_5Ca']
    plt.scatter(low_ca_coords[:, 0], low_ca_coords[:, 1], 
               marker='v', s=60, c='blue', alpha=0.8, edgecolors='darkblue', linewidth=0.5,
               label=f'1.5mM Ca (n={len(low_ca_coords)})')
    
    # High calcium (4mM) - red triangles pointing up  
    high_ca_coords = pca_data['WT_4Ca']
    plt.scatter(high_ca_coords[:, 0], high_ca_coords[:, 1], 
               marker='^', s=60, c='red', alpha=0.8, edgecolors='darkred', linewidth=0.5,
               label=f'4mM Ca (n={len(high_ca_coords)})')
    
    # Calculate centroids
    center_pooled = np.mean(pca_coordinates, axis=0)
    center_low_ca = np.mean(low_ca_coords, axis=0)
    center_high_ca = np.mean(high_ca_coords, axis=0)
    
    # Plot centroids
    plt.scatter(center_low_ca[0], center_low_ca[1], marker='X', s=180, c='blue', 
               edgecolor='black', linewidth=2, label='1.5mM centroid')
    plt.scatter(center_high_ca[0], center_high_ca[1], marker='X', s=180, c='red', 
               edgecolor='black', linewidth=2, label='4mM centroid')
    
    # Draw arrows from pooled centroid to calcium condition centroids
    plt.arrow(center_pooled[0], center_pooled[1],
              center_low_ca[0] - center_pooled[0], center_low_ca[1] - center_pooled[1],
              color='blue', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
    plt.arrow(center_pooled[0], center_pooled[1],
              center_high_ca[0] - center_pooled[0], center_high_ca[1] - center_pooled[1],
              color='red', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
    
    # Annotate centroid coordinates
    plt.text(center_low_ca[0], center_low_ca[1], f'  1.5mM\n({center_low_ca[0]:.2f},{center_low_ca[1]:.2f})', 
             color='blue', fontsize=9, ha='left', va='center', fontweight='bold')
    plt.text(center_high_ca[0], center_high_ca[1], f'  4mM\n({center_high_ca[0]:.2f},{center_high_ca[1]:.2f})', 
             color='red', fontsize=9, ha='left', va='center', fontweight='bold')
    
    # Connect paired boutons between conditions (assuming matched order)
    n_pairs = min(len(low_ca_coords), len(high_ca_coords))
    if n_pairs > 0:
        for i in range(n_pairs):
            plt.plot([low_ca_coords[i, 0], high_ca_coords[i, 0]],
                     [low_ca_coords[i, 1], high_ca_coords[i, 1]],
                     color='gray', alpha=0.4, linewidth=1)
        print(f"Connected {n_pairs} bouton pairs between calcium conditions")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('Calcium Concentration Effects on Bouton Properties')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "calcium_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_low_ca, center_high_ca, n_pairs

def calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs):
    """Calculate movement statistics for calcium concentration changes."""
    
    # Distances from standard condition to each calcium level
    dist_to_low = np.linalg.norm(center_low_ca - center_pooled)
    dist_to_high = np.linalg.norm(center_high_ca - center_pooled)
    
    # Individual bouton movements
    low_ca_coords = pca_data['WT_1_5Ca']
    high_ca_coords = pca_data['WT_4Ca']
    
    # Movement from standard to low calcium
    movements_to_low = []
    n_low_comparisons = min(len(pca_coordinates), len(low_ca_coords))
    for i in range(n_low_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - low_ca_coords[i])
        movements_to_low.append(dist)
    
    # Movement from standard to high calcium
    movements_to_high = []
    n_high_comparisons = min(len(pca_coordinates), len(high_ca_coords))
    for i in range(n_high_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - high_ca_coords[i])
        movements_to_high.append(dist)
    
    # Movement between calcium conditions (paired boutons)
    calcium_range_movements = []
    for i in range(n_pairs):
        dist = np.linalg.norm(low_ca_coords[i] - high_ca_coords[i])
        calcium_range_movements.append(dist)
    
    return {
        'centroid_distances': {'low': dist_to_low, 'high': dist_to_high},
        'individual_movements': {
            'to_low': movements_to_low,
            'to_high': movements_to_high,
            'between_ca': calcium_range_movements
        }
    }

# Run analysis
center_pooled, center_low_ca, center_high_ca, n_pairs = plot_calcium_trajectories()
movement_stats = calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs)

# Display results
print(f"\n=== CALCIUM CONCENTRATION ANALYSIS ===")
print(f"Centroid coordinates:")
print(f"  WT pooled (2.5mM): ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  1.5mM Ca:        ({center_low_ca[0]:.3f}, {center_low_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['low']:.3f}")
print(f"  4mM Ca:          ({center_high_ca[0]:.3f}, {center_high_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['high']:.3f}")

print(f"\nIndividual bouton movements in PCA space:")
if movement_stats['individual_movements']['to_low']:
    low_moves = movement_stats['individual_movements']['to_low']
    print(f"2.5mM → 1.5mM Ca (n={len(low_moves)}): {np.mean(low_moves):.3f} ± {np.std(low_moves):.3f}")

if movement_stats['individual_movements']['to_high']:
    high_moves = movement_stats['individual_movements']['to_high']
    print(f"2.5mM → 4mM Ca (n={len(high_moves)}): {np.mean(high_moves):.3f} ± {np.std(high_moves):.3f}")

if movement_stats['individual_movements']['between_ca']:
    range_moves = movement_stats['individual_movements']['between_ca']
    print(f"1.5mM ↔ 4mM Ca (n={len(range_moves)}): {np.mean(range_moves):.3f} ± {np.std(range_moves):.3f}")

print(f"\n✓ Calcium trajectory analysis complete")

### F Prelude – Calcium Comparisons

We systematically compare amplitude and failure metrics between calcium conditions to quantify release modulation.


### F.2 Calcium-Dependent Amp1 Distributions

Amplitude distributions for the first stimulus are contrasted between calcium conditions to test how release probability responds to extracellular calcium changes.


In [ ]:
# Compare AMP1 distributions between calcium concentrations

def plot_calcium_amp1_comparison():
    """Compare AMP1 distributions between 2.5mM and 1.5mM calcium."""
    
    # Get AMP1 data for both conditions
    amp1_standard = PCA_Data_WT_Pooled['AMP1'].dropna()
    amp1_low_ca = PCA_Data_WT_Low_Ca['AMP1'].dropna()
    
    # Calculate common bins for fair comparison
    all_amp1_values = pd.concat([amp1_standard, amp1_low_ca])
    bin_edges = np.linspace(all_amp1_values.min(), all_amp1_values.max(), 61)
    
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Calculate weights for percentage display
    weights_standard = np.ones(len(amp1_standard)) * (100.0 / len(amp1_standard))
    weights_low_ca = np.ones(len(amp1_low_ca)) * (100.0 / len(amp1_low_ca))
    
    # Plot histograms
    plt.hist(amp1_standard, bins=bin_edges, alpha=0.7, color='gray', 
             weights=weights_standard, edgecolor='black', linewidth=0.5,
             label=f'WT 2.5mM Ca (n={len(amp1_standard)})')
    plt.hist(amp1_low_ca, bins=bin_edges, alpha=0.7, color='blue', 
             weights=weights_low_ca, edgecolor='darkblue', linewidth=0.5,
             label=f'WT 1.5mM Ca (n={len(amp1_low_ca)})')
    
    # Add vertical lines for means
    mean_standard = amp1_standard.mean()
    mean_low_ca = amp1_low_ca.mean()
    
    plt.axvline(mean_standard, color='black', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Mean 2.5mM: {mean_standard:.3f}')
    plt.axvline(mean_low_ca, color='blue', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Mean 1.5mM: {mean_low_ca:.3f}')
    
    # Format plot
    plt.xlabel('AMP1 (Amplitude)')
    plt.ylabel('Proportion (%)')
    plt.title('AMP1 Distribution: 2.5mM vs 1.5mM Calcium')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "amp1_histogram_calcium_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return amp1_standard, amp1_low_ca, output_file

# Run analysis
amp1_standard, amp1_low_ca, output_file = plot_calcium_amp1_comparison()

# Statistical comparison
from scipy.stats import mannwhitneyu, ttest_ind

# Perform statistical tests
mw_stat, mw_p = mannwhitneyu(amp1_standard, amp1_low_ca, alternative='two-sided')
t_stat, t_p = ttest_ind(amp1_standard, amp1_low_ca)

# Summary statistics
print(f"=== AMP1 CALCIUM COMPARISON ===")
print(f"2.5mM Ca (standard): {amp1_standard.mean():.3f} ± {amp1_standard.std():.3f} (n={len(amp1_standard)})")
print(f"1.5mM Ca (low):      {amp1_low_ca.mean():.3f} ± {amp1_low_ca.std():.3f} (n={len(amp1_low_ca)})")

print(f"\nStatistical tests:")
print(f"Mann-Whitney U test: U={mw_stat:.1f}, p={mw_p:.4g}")
print(f"T-test: t={t_stat:.3f}, p={t_p:.4g}")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(amp1_standard)-1)*amp1_standard.var() + (len(amp1_low_ca)-1)*amp1_low_ca.var()) / 
                     (len(amp1_standard) + len(amp1_low_ca) - 2))
cohens_d = (amp1_standard.mean() - amp1_low_ca.mean()) / pooled_std
print(f"Cohen's d (effect size): {cohens_d:.3f}")

# Percentage change
pct_change = ((amp1_low_ca.mean() - amp1_standard.mean()) / amp1_standard.mean()) * 100
print(f"Percentage change (1.5mM vs 2.5mM): {pct_change:+.1f}%")

print(f"\n✓ Saved comparison to {output_file}")

#### F Metric – Failure Distributions

Contrasting failure rates between calcium levels reveals how release reliability depends on extracellular calcium.


### F.3 Calcium-Dependent Failure Rates

Failure percentages are compared between calcium levels, revealing whether reduced calcium disproportionately increases synaptic failures.


In [ ]:
# Compare failure rates between calcium concentrations
fail1_standard = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_low_ca = PCA_Data_WT_Low_Ca['%Fail1'].dropna()

# Plot histograms
all_fail1 = pd.concat([fail1_standard, fail1_low_ca])
bins = np.linspace(all_fail1.min(), all_fail1.max(), 31)

plt.figure(figsize=(10, 6))
weights_standard = np.ones(len(fail1_standard)) / len(fail1_standard) * 100
weights_low_ca = np.ones(len(fail1_low_ca)) / len(fail1_low_ca) * 100

plt.hist(fail1_standard, bins=bins, alpha=0.7, color='gray', weights=weights_standard, 
         edgecolor='black', label=f'WT 2.5mM Ca (n={len(fail1_standard)})')
plt.hist(fail1_low_ca, bins=bins, alpha=0.7, color='blue', weights=weights_low_ca, 
         edgecolor='darkblue', label=f'WT 1.5mM Ca (n={len(fail1_low_ca)})')

plt.axvline(fail1_standard.mean(), color='black', linestyle='--', linewidth=2)
plt.axvline(fail1_low_ca.mean(), color='blue', linestyle='--', linewidth=2)

plt.xlabel('%Fail1')
plt.ylabel('Proportion (%)')
plt.title('Failure Rate: 2.5mM vs 1.5mM Calcium')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

output_file = OUTPUT_DIR / "fail1_calcium_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Stats
from scipy.stats import mannwhitneyu
_, p_value = mannwhitneyu(fail1_standard, fail1_low_ca)
print(f"2.5mM Ca: {fail1_standard.mean():.1f}% ± {fail1_standard.std():.1f}%")
print(f"1.5mM Ca: {fail1_low_ca.mean():.1f}% ± {fail1_low_ca.std():.1f}%")
print(f"Mann-Whitney p = {p_value:.4g}")

### F.4 Calcium Impact on Summary Metrics

Non-parametric tests and paired boxplots quantify how calcium concentration affects key amplitudes and plasticity measures, providing statistical backing for observed shifts.


In [ ]:
# Direct comparison between low and high calcium conditions
from scipy.stats import mannwhitneyu

# Prepare data for plotting
amp1_comparison = pd.DataFrame({
    'AMP1': pd.concat([PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

fail1_comparison = pd.DataFrame({
    '%Fail1': pd.concat([PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

# Create side-by-side boxplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# AMP1 comparison
sns.boxplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1, 
           palette=['blue', 'red'], showcaps=True, fliersize=0)
sns.stripplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1,
             color='black', size=3, alpha=0.6)
ax1.set_title('AMP1: Low vs High Calcium')

# %Fail1 comparison  
sns.boxplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
           palette=['blue', 'red'], showcaps=True, fliersize=0)
sns.stripplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
             color='black', size=3, alpha=0.6)
ax2.set_title('%Fail1: Low vs High Calcium')

plt.tight_layout()

output_file = OUTPUT_DIR / "calcium_direct_comparison_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Statistical tests
amp1_u, amp1_p = mannwhitneyu(PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1'])
fail1_u, fail1_p = mannwhitneyu(PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1'])

# Results
print("1.5mM vs 4mM Calcium Comparison:")
print("-" * 40)
print(f"AMP1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_Low_Ca['AMP1'].std():.3f}")
print(f"  4mM:   {PCA_Data_WT_High_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_High_Ca['AMP1'].std():.3f}")
print(f"  p = {amp1_p:.4g}")

print(f"%Fail1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_Low_Ca['%Fail1'].std():.1f}%")
print(f"  4mM:   {PCA_Data_WT_High_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_High_Ca['%Fail1'].std():.1f}%")
print(f"  p = {fail1_p:.4g}")

# Save stats
stats_file = OUTPUT_DIR / "calcium_comparison_statistics.txt"
with open(stats_file, 'w') as f:
    f.write("1.5mM vs 4mM Calcium Statistical Comparison\n")
    f.write("=" * 45 + "\n\n")
    f.write(f"AMP1: Mann-Whitney U={amp1_u:.1f}, p={amp1_p:.6g}\n")
    f.write(f"%Fail1: Mann-Whitney U={fail1_u:.1f}, p={fail1_p:.6g}\n")

print(f"✓ Saved to {output_file} and {stats_file}")

### F Prelude – Calcium PPR Profiles

Paired-pulse trajectories across calcium conditions show whether facilitation rules shift with release probability.


### F.5 Calcium PPR Trajectories

Average paired-pulse profiles are contrasted across calcium conditions to determine whether facilitation dynamics are calcium-sensitive.


In [ ]:
# Compare PPR profiles across calcium concentrations
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Low_Ca.columns]
pulse_numbers = list(range(1, len(ppr_cols) + 2))

# Calculate PPR profiles for each condition
def get_ppr_profile(data, condition_name):
    means = [1.0] + data[ppr_cols].mean().tolist()
    sems = [0.0] + data[ppr_cols].sem().tolist()
    return means, sems

# Get profiles for each calcium condition
means_low_ca, sems_low_ca = get_ppr_profile(PCA_Data_WT_Low_Ca, '1.5mM Ca')
means_high_ca, sems_high_ca = get_ppr_profile(PCA_Data_WT_High_Ca, '4mM Ca') 
means_standard, sems_standard = get_ppr_profile(PCA_Data_WT_Pooled, '2.5mM Ca')

# Plot PPR profiles
plt.figure(figsize=(8, 5))

# Low calcium (blue)
plt.plot(pulse_numbers, means_low_ca, marker='o', color='blue', linewidth=2,
         label=f'1.5mM Ca (n={len(PCA_Data_WT_Low_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_low_ca) - np.array(sems_low_ca),
                 np.array(means_low_ca) + np.array(sems_low_ca), color='blue', alpha=0.2)

# High calcium (red)
plt.plot(pulse_numbers, means_high_ca, marker='s', color='red', linewidth=2,
         label=f'4mM Ca (n={len(PCA_Data_WT_High_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_high_ca) - np.array(sems_high_ca),
                 np.array(means_high_ca) + np.array(sems_high_ca), color='red', alpha=0.2)

# Standard calcium (black)
plt.plot(pulse_numbers, means_standard, marker='D', color='black', linewidth=2,
         label=f'2.5mM Ca (n={len(PCA_Data_WT_Pooled)})')
plt.fill_between(pulse_numbers, np.array(means_standard) - np.array(sems_standard),
                 np.array(means_standard) + np.array(sems_standard), color='black', alpha=0.15)

# Format plot
plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.xlabel('Pulse Number')
plt.ylabel('PPR (A_n/A_1)')
plt.title('PPR Profiles: Calcium Concentration Effects')
plt.xticks(pulse_numbers)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save figure and data
output_fig = OUTPUT_DIR / "ppr_profiles_calcium_comparison.pdf"
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

# Save numerical data
output_data = OUTPUT_DIR / "ppr_profiles_calcium_data.txt"
with open(output_data, "w") as f:
    f.write("Pulse\t1.5mM_Mean\t1.5mM_SEM\t4mM_Mean\t4mM_SEM\t2.5mM_Mean\t2.5mM_SEM\n")
    for i, pulse in enumerate(pulse_numbers):
        f.write(f"{pulse}\t{means_low_ca[i]:.4f}\t{sems_low_ca[i]:.4f}\t"
                f"{means_high_ca[i]:.4f}\t{sems_high_ca[i]:.4f}\t"
                f"{means_standard[i]:.4f}\t{sems_standard[i]:.4f}\n")

print(f"✓ Saved PPR profiles to {output_fig}")
print(f"✓ Saved numerical data to {output_data}")

### F Prelude – Calcium Trace Morphology

Average traces under high and low calcium highlight kinetic consequences of altering release probability.


### F.6 Calcium Trace Morphology

Mean traces from high- and low-calcium experiments are compared over the response window, highlighting kinetic differences attributable to calcium availability.


In [ ]:
# Compare mean traces between calcium concentrations (0.5-2.0s window)

# Extract traces for calcium conditions
low_ca_traces = []
high_ca_traces = []

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    if row['Condition'] == 'Theo_1_5Ca':
        low_ca_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_4Ca':
        high_ca_traces.append(row['Avg'])

if not low_ca_traces or not high_ca_traces:
    print("Calcium trace conditions not found in resampled data")
    available_conditions = NORM_TRACES_DATAFRAME['Condition'].unique()
    print(f"Available conditions: {list(available_conditions)}")
else:
    # Calculate means and SEMs
    low_ca_mean = np.nanmean(low_ca_traces, axis=0)
    low_ca_sem = np.nanstd(low_ca_traces, axis=0, ddof=1) / np.sqrt(len(low_ca_traces))
    
    high_ca_mean = np.nanmean(high_ca_traces, axis=0)
    high_ca_sem = np.nanstd(high_ca_traces, axis=0, ddof=1) / np.sqrt(len(high_ca_traces))
    
    # Plot comparison (0.5-2.0s window)
    plt.figure(figsize=(10, 5))
    
    plt.plot(COMMON_TIME, low_ca_mean, color='blue', linewidth=2, 
             label=f'1.5mM Ca (n={len(low_ca_traces)})')
    plt.fill_between(COMMON_TIME, low_ca_mean - low_ca_sem, low_ca_mean + low_ca_sem, 
                     color='blue', alpha=0.25)
    
    plt.plot(COMMON_TIME, high_ca_mean, color='red', linewidth=2,
             label=f'4mM Ca (n={len(high_ca_traces)})')
    plt.fill_between(COMMON_TIME, high_ca_mean - high_ca_sem, high_ca_mean + high_ca_sem, 
                     color='red', alpha=0.25)

    # Add stimulus markers (every 100ms from 1.0s)
    stim_times = [1.0 + 0.1*i for i in range(10)]
    for stim_time in stim_times:
        if stim_time <= 2.0:
            plt.axvline(stim_time, color='gray', linestyle='--', alpha=0.4, linewidth=1)
    
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.xlim(0.5, 2.0)
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.title('Mean Traces: 1.5mM vs 4mM Calcium')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "calcium_mean_traces_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Calcium trace comparison: 1.5mM (n={len(low_ca_traces)}) vs 4mM (n={len(high_ca_traces)})")
    print(f"✓ Saved to {output_file}")

### F Prelude – Calcium Scatter Analyses

Scatterplots relate amplitude and failure metrics to PPR ratios, clarifying how calcium availability reshapes multivariate relationships.


### F.7 Calcium Correlation Structure

We correlate PPR ratios with amplitude and failure metrics under each calcium condition to see how release probability and synaptic reliability interact with plasticity when calcium is limiting.


In [ ]:
# Correlation analysis: AMP1 and %Fail1 vs PPR2/1 across calcium conditions
from scipy.stats import pearsonr, t

def calcium_scatter_analysis(x_param, y_param='PPR2/1'):
    """Create scatter plot with regression analysis for calcium conditions."""
    
    # Extract data for both conditions
    low_ca_data = PCA_Data_WT_Low_Ca[[x_param, y_param]].dropna()
    high_ca_data = PCA_Data_WT_High_Ca[[x_param, y_param]].dropna()
    
    x_low, y_low = low_ca_data[x_param].values, low_ca_data[y_param].values
    x_high, y_high = high_ca_data[x_param].values, high_ca_data[y_param].values
    
    # Calculate separate correlations
    r_low, p_low = pearsonr(x_low, y_low) if len(x_low) > 1 else (float('nan'), float('nan'))
    r_high, p_high = pearsonr(x_high, y_high) if len(x_high) > 1 else (float('nan'), float('nan'))
    
    # Pooled analysis
    x_pool = np.concatenate([x_low, x_high])
    y_pool = np.concatenate([y_low, y_high])
    
    if len(x_pool) > 2:
        slope, intercept = np.polyfit(x_pool, y_pool, 1)
        r_pool, p_pool = pearsonr(x_pool, y_pool)
        
        # Calculate confidence intervals
        x_grid = np.linspace(x_pool.min(), x_pool.max(), 100)
        y_fit = intercept + slope * x_grid
        
        # Simplified CI calculation
        residuals = y_pool - (intercept + slope * x_pool)
        mse = np.sum(residuals**2) / (len(x_pool) - 2)
        se = np.sqrt(mse)
        
        t_crit = t.ppf(0.975, len(x_pool) - 2)
        margin = t_crit * se
        
    else:
        r_pool = p_pool = float('nan')
        x_grid = y_fit = margin = None
    
    # Create plot
    plt.figure(figsize=(7, 5))
    plt.scatter(x_low, y_low, c='blue', alpha=0.7, edgecolor='black', s=60, 
               label=f'1.5mM Ca (n={len(x_low)})')
    plt.scatter(x_high, y_high, c='red', alpha=0.7, edgecolor='black', s=60,
               label=f'4mM Ca (n={len(x_high)})')
    
    # Add regression line and confidence band
    if x_grid is not None:
        plt.plot(x_grid, y_fit, color='black', linewidth=2, label='Pooled regression')
        plt.fill_between(x_grid, y_fit - margin, y_fit + margin, 
                        color='black', alpha=0.15, label='95% CI')
    
    # Format plot
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.title(f'{x_param} vs {y_param}: Calcium Comparison')
    plt.grid(True, alpha=0.3)
    
    # Set reasonable axis limits
    if x_param == 'AMP1':
        plt.xlim(0, max(3, x_pool.max() * 1.1))
    plt.ylim(0, max(3, y_pool.max() * 1.1))
    
    # Add correlation statistics
    stats_text = (f"1.5mM: r={r_low:.2f}, p={p_low:.2g}\n"
                  f"4mM: r={r_high:.2f}, p={p_high:.2g}\n"
                  f"Pooled: r={r_pool:.2f}, p={p_pool:.2g}")
    plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, 
             va='top', fontsize=9, 
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.legend()
    plt.tight_layout()
    
    # Save results
    safe_param = x_param.replace('%', 'pct').replace('/', '_')
    output_file = OUTPUT_DIR / f"scatter_{safe_param}_vs_ppr2_1_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Save statistics
    stats_file = OUTPUT_DIR / f"correlation_{safe_param}_vs_ppr2_1_stats.txt"
    with open(stats_file, 'w') as f:
        f.write(f"Correlation: {x_param} vs {y_param}\n")
        f.write(f"1.5mM Ca: r={r_low:.4f}, p={p_low:.6g}, n={len(x_low)}\n")
        f.write(f"4mM Ca: r={r_high:.4f}, p={p_high:.6g}, n={len(x_high)}\n")
        f.write(f"Pooled: r={r_pool:.4f}, p={p_pool:.6g}, n={len(x_pool)}\n")
    
    print(f"✓ {x_param} vs {y_param}: {stats_text.replace(chr(10), ' | ')}")
    return output_file

# Run both analyses
amp1_output = calcium_scatter_analysis('AMP1')
fail1_output = calcium_scatter_analysis('%Fail1')

print(f"\n✓ Scatter analyses complete:")
print(f"  AMP1 vs PPR2/1: {amp1_output}")
print(f"  %Fail1 vs PPR2/1: {fail1_output}")

### F Prelude – Trial-Level Amplitude Tests

We probe trial-by-trial amplitude distributions to separate changes in success amplitudes from shifts in failure prevalence at different calcium levels.


### F.8 Trial-Level Amplitude Distributions

Per-trial amplitude histograms are modeled to dissect how success and failure amplitudes diverge under different calcium concentrations, offering a granular view of release variability.


In [ ]:
# Analyze trial-level amplitude distributions using existing trials data
from scipy.stats import norm

def analyze_trial_amplitudes(fit_gaussian=False):
    """Analyze trial amplitude distributions from PPR_TRIALS_FILENAME."""
    
    # Load trials data using existing path structure
    trials_file = BASE_DIR / PPR_TRIALS_FILENAME
    
    try:
        trials = pd.read_excel(trials_file)
    except FileNotFoundError:
        print(f"Trials file not found: {trials_file}")
        return
    
    # Set column names based on the structure you provided
    trials.columns = ['AMP1', 'status', 'file', 'folder', 'trial']
    
    # Clean data
    trials['AMP1'] = pd.to_numeric(trials['AMP1'], errors='coerce')
    trials['status'] = trials['status'].astype(str).str.lower().str.strip()
    trials['folder'] = trials['folder'].astype(str).str.strip()
    
    # Filter for calcium conditions
    calcium_conditions = {'Theo_4Ca', 'Theo_1_5Ca'}
    trials_filtered = trials[
        trials['folder'].isin(calcium_conditions) &
        trials['status'].isin(['success', 'failure'])
    ].dropna(subset=['AMP1']).copy()
    
    if len(trials_filtered) == 0:
        print("No calcium trial data found")
        available_conditions = trials['folder'].unique()
        print(f"Available conditions: {list(available_conditions)}")
        return
    
    # Create categories
    def categorize_trial(row):
        if row['status'] == 'failure':
            return 'Failures (both Ca)'
        return f"Success {row['folder']}"
    
    trials_filtered['Category'] = trials_filtered.apply(categorize_trial, axis=1)
    
    # Set up plotting
    categories = ['Success Theo_4Ca', 'Success Theo_1_5Ca', 'Failures (both Ca)']
    colors = {'Success Theo_4Ca': 'red', 'Success Theo_1_5Ca': 'blue', 'Failures (both Ca)': 'green'}
    
    # Create bins
    amp_range = trials_filtered['AMP1']
    bins = np.linspace(amp_range.min(), amp_range.max(), 81)
    bin_width = bins[1] - bins[0]
    
    plt.figure(figsize=(8, 6))
    fit_results = []
    
    for category in categories:
        subset = trials_filtered[trials_filtered['Category'] == category]
        if len(subset) == 0:
            continue
        
        amplitudes = subset['AMP1'].values
        weights = np.ones(len(amplitudes)) * (100.0 / len(amplitudes))
        
        # Plot histogram
        plt.hist(amplitudes, bins=bins, weights=weights, alpha=0.6, 
                color=colors[category], label=f"{category} (n={len(amplitudes)})",
                edgecolor='black', linewidth=0.3)
        
        # Add Gaussian fit if requested
        if fit_gaussian and len(amplitudes) > 1:
            mu, sigma = norm.fit(amplitudes)
            bin_centers = (bins[:-1] + bins[1:]) / 2
            pdf_scaled = norm.pdf(bin_centers, mu, sigma) * (bin_width * 100)
            plt.plot(bin_centers, pdf_scaled, color=colors[category], linewidth=2)
            
            # Mark peak
            y_peak = norm.pdf(mu, mu, sigma) * (bin_width * 100)
            plt.text(mu, y_peak * 1.02, f"{mu:.2f}", color='black',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
            
            fit_results.append({
                'Category': category,
                'mu': mu,
                'sigma': sigma,
                'n': len(amplitudes)
            })
    
    # Format plot
    plt.xlabel('AMP1 (Trial Amplitude)')
    plt.ylabel('Proportion (%)')
    title = 'Trial Amplitude Distributions: Calcium Conditions'
    if fit_gaussian:
        title += ' + Gaussian Fits'
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    suffix = "_gaussian_fits" if fit_gaussian else ""
    output_file = OUTPUT_DIR / f"trial_amplitudes_calcium{suffix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Display results
    print(f"Trial amplitude analysis:")
    for category in categories:
        subset = trials_filtered[trials_filtered['Category'] == category]
        if len(subset) > 0:
            mean_amp = subset['AMP1'].mean()
            print(f"  {category}: {len(subset)} trials (mean: {mean_amp:.3f})")
    
    if fit_gaussian and fit_results:
        print("\nGaussian fit parameters:")
        for result in fit_results:
            print(f"  {result['Category']}: μ={result['mu']:.3f}, σ={result['sigma']:.3f}")
    
    print(f"✓ Saved plot to {output_file}")
    return trials_filtered

# Run analyses
trials_basic = analyze_trial_amplitudes(fit_gaussian=False)
trials_fitted = analyze_trial_amplitudes(fit_gaussian=True)

### F Prelude – Stability Cluster Attribution

The stability experiment reuses PCA-based boundaries to assign before and after recordings to established clusters.


### F.9 Stability Analysis Configuration

Parameter knobs and helper structures are established to analyze before-versus-after stability experiments. These controls govern how strictly clusters are defined in subsequent comparisons.


In [ ]:
# ==== Cell 0 — Config + tiny helpers (set your "edge" controls here) ====

ELLIPSE_ALPHA   = 0.95     # ellipse containment level
ALPHA_EXPANSION = 0.50     # alpha-shape expansion factor (0.0 → off)
KNN_K           = None     # k-NN neighbors (None → auto √N)

import numpy as np
from scipy.stats import chi2

def build_ellipse_models(X, y, n_clusters, alpha=0.95, ridge=1e-6):
    """Mean/cov/inv and chi2 threshold for each cluster."""
    thr = chi2.ppf(alpha, df=2)
    models = []
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts) > 2:
            mu  = pts.mean(axis=0)
            cov = np.cov(pts.T) + np.eye(2)*ridge
            inv = np.linalg.pinv(cov)
            models.append({'cluster': k, 'center': mu, 'cov': cov, 'inv_cov': inv})
    return models, thr

def mahalanobis_sq(x, mu, inv):
    d = x - mu
    return float(d.T @ inv @ d)

def ellipses_containing(x, models, thr):
    return {m['cluster'] for m in models if mahalanobis_sq(x, m['center'], m['inv_cov']) <= thr}

def nearest_ellipse_edge(x, models, thr):
    """Return (cluster_id, euclid_dist_to_edge)."""
    best = (None, np.inf)
    for m in models:
        md2 = mahalanobis_sq(x, m['center'], m['inv_cov'])
        if md2 <= thr:
            return m['cluster'], 0.0
        s = np.sqrt(thr/md2)
        x_proj = m['center'] + s*(x - m['center'])
        dist = float(np.linalg.norm(x - x_proj))
        if dist < best[1]:
            best = (m['cluster'], dist)
    return best


### F.10 Stability Trajectories

This visualization tracks how individual boutons move through PCA space from the baseline to the post-manipulation state, summarizing trajectory lengths and directionality.


In [ ]:
# ==== Cell 1 — Before/After trajectories + summary ====

import matplotlib.pyplot as plt

# Data
before_coords = np.asarray(pca_data['stab_before'])
after_coords  = np.asarray(pca_data['stab_after'])
n_pairs = int(min(len(before_coords), len(after_coords)))
assert n_pairs > 0, "No paired before/after points."

# Plot
plt.figure(figsize=(8,6))
plt.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=cluster_assignments, s=20, alpha=0.4, label='WT background')
plt.scatter(before_coords[:,0], before_coords[:,1], c='orange', s=60, edgecolors='darkorange', label=f'Before (n={len(before_coords)})')
plt.scatter(after_coords[:,0],  after_coords[:,1],  c='brown',  s=60, edgecolors='darkred',   label=f'After  (n={len(after_coords)})')

# Pair links + mean arrow
moves = []
for i in range(n_pairs):
    plt.plot([before_coords[i,0], after_coords[i,0]],
             [before_coords[i,1], after_coords[i,1]], color='gray', alpha=0.6, lw=1)
    moves.append(float(np.linalg.norm(after_coords[i] - before_coords[i])))

diffs = after_coords[:n_pairs] - before_coords[:n_pairs]
mean_vec = diffs.mean(axis=0)
center   = np.vstack([before_coords[:n_pairs], after_coords[:n_pairs]]).mean(axis=0)
plt.arrow(center[0], center[1], mean_vec[0], mean_vec[1], color='black',
          width=0.05, head_width=0.25, head_length=0.25, length_includes_head=True, label='Mean trajectory')

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%})'); plt.ylabel(f'PC2 ({pc2_variance:.1%})')
plt.title('Stability: Before vs After'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "stability_before_after_trajectories.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print("=== STABILITY TRAJECTORY ANALYSIS ===")
print(f"Pairs: {n_pairs}")
print(f"Movement (mean±SD): {np.mean(moves):.3f} ± {np.std(moves):.3f}")
print(f"Mean trajectory |mag|: {np.linalg.norm(mean_vec):.3f}  dir=({mean_vec[0]:.3f}, {mean_vec[1]:.3f})")
print(f"✓ Saved {out}")


### F.11 Stability Movement Histogram

We compile a histogram of bouton displacements to quantify how much synaptic properties drift between the before and after conditions.


In [ ]:
# ==== Cell 2 — Movement distance histogram (auto bins) ====

# Freedman–Diaconis binning with fallback
md = np.linalg.norm(diffs, axis=1).astype(float)
md = md[np.isfinite(md)]
q25,q75 = np.percentile(md,[25,75]); iqr=float(q75-q25); n=len(md)
bw = (2*iqr)/(n**(1/3)) if iqr>0 else 0.0
bins = max(5, int(np.ceil((md.max()-md.min())/bw))) if bw>0 else max(5, int(np.ceil(np.sqrt(n))))

plt.figure(figsize=(6,4))
plt.hist(md, bins=bins, edgecolor='black', alpha=0.85)
plt.axvline(md.mean(), ls='--', lw=2, label=f'Mean = {md.mean():.2f}')
plt.xlabel('Distance in PCA space'); plt.ylabel('Count'); plt.title('Before→After distances'); plt.legend(); plt.tight_layout()
out = OUTPUT_DIR / "stability_movement_distances.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print(f"n={n}  mean={md.mean():.3f}  sd={md.std(ddof=1):.3f}  median={np.median(md):.3f}  min={md.min():.3f}  max={md.max():.3f}  bins={bins}")
print(f"✓ Saved {out}")


### F.12 Stability Trace Evolution

Mean traces with confidence intervals are plotted for before and after recordings, revealing how waveform kinetics change with the stability manipulation.


In [ ]:
# ==== Cell 3 — Mean traces (±SEM) for Stability_Before/_05 vs Stability_After/_05 ====

assert 'NORM_TRACES_DATAFRAME' in locals() and 'COMMON_TIME' in locals()

import pandas as pd

def stack_traces(df, cond):
    rows = df[df['Condition']==cond]
    X = [np.asarray(r['Avg'], float) for _,r in rows.iterrows()]
    X = [t for t in X if np.isfinite(t).all() and len(t)==len(COMMON_TIME)]
    return (np.vstack(X) if len(X)>0 else np.empty((0,len(COMMON_TIME)))), len(X)

def mean_sem(X):
    if X.size==0: 
        z = np.zeros(len(COMMON_TIME)); return z,z
    m = np.nanmean(X, axis=0)
    s = np.nanstd(X, axis=0, ddof=1)/np.sqrt(max(1,X.shape[0]))
    return m,s

conds = ["Stability_Before","Stability_Before_05","Stability_After","Stability_After_05"]
stacked = {c: stack_traces(NORM_TRACES_DATAFRAME,c) for c in conds}
stats   = {c: mean_sem(stacked[c][0]) for c in conds}
counts  = {c: stacked[c][1] for c in conds}

colors = {"Stability_Before":"#1f77b4","Stability_Before_05":"#1f77b4","Stability_After":"#d62728","Stability_After_05":"#d62728"}
styles = {"Stability_Before":('-',2.0),"Stability_Before_05":('--',1.8),"Stability_After":('-',2.0),"Stability_After_05":('--',1.8)}

plt.figure(figsize=(8.5,5.0))
for c in conds:
    mean,sem = stats[c]; ls,lw = styles[c]
    plt.plot(COMMON_TIME, mean, color=colors[c], ls=ls, lw=lw, label=f"{c} (n={counts[c]})")
    plt.fill_between(COMMON_TIME, mean-sem, mean+sem, color=colors[c], alpha=0.15, lw=0)
plt.axhline(0, color='gray', ls=':', lw=1.0)
plt.axvspan(0.5, 2.0, color='gray', alpha=0.08, label='0.5–2.0 s')
plt.xlabel('Time (s)'); plt.ylabel('ΔF/F'); plt.title('Stability conditions: mean traces (±SEM)')
plt.legend(ncol=2, fontsize=9, frameon=True); plt.tight_layout()
out = OUTPUT_DIR / "stability_mean_traces_before_after.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print("Counts:", {c:counts[c] for c in conds}); print(f"✓ Saved {out}")


### F.13 Stability Ellipse Boundaries

Elliptical decision boundaries are applied in PCA space to evaluate which boutons remain within the WT tolerance zone after the manipulation, providing a geometric perspective on stability.


In [ ]:
# ===== Cell 4 — Ellipses: PCA with hard edge + tolerance zone, and pie =====
# knobs
ELLIPSE_ALPHA = 0.95        # hard edge level
ELLIPSE_TOL   = 0.20        # inflate ellipse threshold by (1 + ELLIPSE_TOL)

import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# --- build models ---
def _ellipse_models(X, y, n_clusters, ridge=1e-6):
    models=[]
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts)>2:
            mu=pts.mean(axis=0); cov=np.cov(pts.T)+np.eye(2)*ridge; inv=np.linalg.pinv(cov)
            models.append({'cluster':k,'center':mu,'cov':cov,'inv_cov':inv})
    return models
def _md2(x, m): 
    d=x-m['center']; return float(d.T @ m['inv_cov'] @ d)

ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol  = thr_hard*(1.0 + ELLIPSE_TOL)

# --- plot PCA with hard + tolerance ---
cmap = plt.get_cmap('Set2'); uniq=np.unique(cluster_assignments); lut={cid:i for i,cid in enumerate(uniq)}
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=cluster_assignments, s=20, alpha=0.35, label='WT')

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3: continue
    m = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    # tolerance ring: draw tol (filled light), then hard (outline)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  cmap(lut[cid]%8), 0.10, 0.0)    # tol zone
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, cmap(lut[cid]%8), 0.00, 2.0)    # hard edge

# overlay pairs
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before = np.asarray(pca_data['stab_before'])[:n_pairs]
after  = np.asarray(pca_data['stab_after'])[:n_pairs]

# Calculate stability for line colors
def _label_all_for_plot(x, thr):
    """Return all clusters that contain point x (for plotting)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _label_all_for_plot(before[i], thr_hard)
    a_clusters = _label_all_for_plot(after[i], thr_hard)
    is_stable = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}% (Conservative)')

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "ellipses_pca_hard_tol.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability approach (hard-edge only) ---
def _inside_any(x, thr): 
    return any(_md2(x,m) <= thr for m in ellipse_models)

def _label_all(x, thr):
    """Return all clusters that contain point x (conservative approach)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Get all cluster memberships for each point
b_labs = [_label_all(before[i], thr_hard) for i in range(n_pairs)]
a_labs = [_label_all(after[i], thr_hard) for i in range(n_pairs)]

# Conservative stability: stable if any overlap between before and after cluster sets
stable = sum(1 for i in range(n_pairs) if len(b_labs[i] & a_labs[i]) > 0)
unstable = n_pairs - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title('Conservative Ellipse Stability (hard-edge)'); plt.tight_layout()
out = OUTPUT_DIR / "ellipses_stability_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Ellipses] Conservative: hard α={ELLIPSE_ALPHA:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_labs[i] & a_labs[i]) > 0 and (len(b_labs[i]) > 1 or len(a_labs[i]) > 1):
        overlap_cases.append((i, b_labs[i], a_labs[i], b_labs[i] & a_labs[i]))

if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with ellipse overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")

### F.14 Stability Alpha Shapes

Alpha-shape contours supply a non-parametric boundary around WT clusters, allowing us to test whether post-manipulation boutons exit the original manifold.


In [ ]:
# ===== Cell — Alpha-shapes: PCA with hard polygon + tolerance zone, and pie =====
# knobs
ALPHA_EXPANSION = 0.50      # tolerance zone: expanded by sqrt(1+exp)
ALPHA_KNN_Q     = 0.5       # alpha heuristic quantile (0.8 for very small n)

import numpy as np, matplotlib.pyplot as plt, alphashape
from shapely.affinity import scale as shp_scale
from shapely.geometry import MultiPoint, Polygon as ShapelyPolygon, MultiPolygon, Point

def _alpha_for(pts):
    n=len(pts); d2=np.sum((pts[:,None,:]-pts[None,:,:])**2, axis=2); np.fill_diagonal(d2, np.inf)
    kth=np.partition(d2,1,axis=1)[:,1]; base=np.sqrt(kth)
    return float(1.5*np.quantile(base, ALPHA_KNN_Q if n>=10 else 0.8))

def _make_alpha_shapes(X,y,expansion):
    s = float(np.sqrt(1.0+expansion)) if expansion>0 else 1.0
    res={}
    for cid in np.unique(y):
        pts = X[y==cid]
        if len(pts)<3: res[cid]=None; continue
        a=_alpha_for(pts)
        poly = alphashape.alphashape([tuple(r) for r in pts], a)
        if poly is None or getattr(poly,'is_empty',True): poly = MultiPoint([tuple(r) for r in pts]).convex_hull
        res[cid]={'hard':poly, 'tol': (shp_scale(poly, xfact=s, yfact=s, origin='centroid') if expansion>0 else poly)}
    return res

shapes = _make_alpha_shapes(pca_coordinates, cluster_assignments, ALPHA_EXPANSION)
cmap = plt.get_cmap('Set2'); uniq=np.unique(cluster_assignments); lut={cid:i for i,cid in enumerate(uniq)}

# --- PCA: draw tol (light fill) + hard (outline) ---
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=cluster_assignments, s=20, alpha=0.35, label='WT')
for cid,sh in shapes.items():
    if not sh: continue
    for tag, (fa, ls, lw) in dict(tol=(0.10,'-',0.0), hard=(0.00,'-',2.0)).items():
        g = sh[tag]
        geoms=[g] if g.geom_type=='Polygon' else (list(g.geoms) if g.geom_type=='MultiPolygon' else [])
        for gg in geoms:
            X,Y = np.array(gg.exterior.coords).T
            if fa>0: ax.fill(X,Y,color=cmap(lut[cid]%8),alpha=fa)
            ax.plot(X,Y,color=cmap(lut[cid]%8),ls=ls,lw=lw)

# pairs with conservative coloring
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before = np.asarray(pca_data['stab_before'])[:n_pairs]
after  = np.asarray(pca_data['stab_after'])[:n_pairs]

# Conservative approach: find ALL clusters that contain each point
def _in_hard_all(x):
    """Return set of all clusters that contain point x (conservative approach)"""
    p = Point(float(x[0]), float(x[1]))
    clusters = set()
    for cid, sh in shapes.items():
        if not sh: continue
        g = sh['hard']
        if g.geom_type == 'Polygon' and g.contains(p):
            clusters.add(cid)
        elif g.geom_type == 'MultiPolygon' and any(gg.contains(p) for gg in g.geoms):
            clusters.add(cid)
    return clusters if clusters else None

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _in_hard_all(before[i])
    a_clusters = _in_hard_all(after[i])
    
    # Check stability: stable if both points are in some shapes and they share at least one cluster
    is_stable = (b_clusters is not None and a_clusters is not None and 
                len(b_clusters & a_clusters) > 0)
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Alpha-shapes: hard + tol (exp={ALPHA_EXPANSION:.2f}) (Conservative)')

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()

out = OUTPUT_DIR / "alphashapes_pca_hard_tol.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability (hard polygon only) ---
b_clusters_all = [_in_hard_all(before[i]) for i in range(n_pairs)]
a_clusters_all = [_in_hard_all(after[i]) for i in range(n_pairs)]

# Count pairs where both points are inside some shapes
valid_pairs = [(b, a) for b, a in zip(b_clusters_all, a_clusters_all) if b is not None and a is not None]
total = len(valid_pairs)

# Conservative stability: stable if any overlap between before and after cluster sets
stable = sum(1 for b, a in valid_pairs if len(b & a) > 0)
unstable = total - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title('Conservative Alpha-shape Stability (hard edge)'); plt.tight_layout()
out = OUTPUT_DIR / "alphashapes_stability_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Alpha] Conservative: exp={ALPHA_EXPANSION:.2f}  stable={stable}/{total} ({100*stable/max(1,total):.1f}%)  (pairs inside any hard shape: {total}/{n_pairs})  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i, (b, a) in enumerate(zip(b_clusters_all, a_clusters_all)):
    if b is not None and a is not None and len(b & a) > 0 and (len(b) > 1 or len(a) > 1):
        overlap_cases.append((i, b, a, b & a))

if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with alpha-shape overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")

### F.15 Stability k-NN Classification

A k-nearest neighbors classifier assigns before and after boutons to WT clusters while quantifying ambiguity zones, offering a probabilistic lens on stability.


In [ ]:
# ===== Cell — kNN: PCA with hard decision + tolerance band, and pie =====
# knobs
KNN_K      = None   # None → auto √N (odd)
KNN_TOLMAX = 0.15   # tolerance zone where max class prob < 1 - KNN_TOLMAX
KNN_PROB_THRESHOLD = 0.20  # consider all classes with prob >= this threshold for conservative approach

import numpy as np, matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from matplotlib.colors import ListedColormap

# train
k = KNN_K
if k is None:
    k = max(3, int(np.sqrt(len(cluster_assignments)))); 
    if k % 2 == 0: k += 1
knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
knn.fit(pca_coordinates, cluster_assignments)

# grid
x,y = pca_coordinates[:,0], pca_coordinates[:,1]
pad = 0.07*((x.max()-x.min())+(y.max()-y.min()))
xlim=(x.min()-pad, x.max()+pad); ylim=(y.min()-pad, y.max()+pad)
xx,yy = np.meshgrid(np.linspace(*xlim, 350), np.linspace(*ylim, 350))
XY = np.c_[xx.ravel(), yy.ravel()]
proba = knn.predict_proba(XY)                     # shape (N, n_classes)
pmax  = proba.max(axis=1).reshape(xx.shape)       # max class prob grid
Z     = knn.predict(XY).reshape(xx.shape)         # hard decision

# --- plot PCA with tolerance band (low confidence) + hard boundaries ---
uniq = np.unique(cluster_assignments); cmap=plt.get_cmap('Set2'); cm = ListedColormap([cmap(i%8) for i in range(len(uniq))])

plt.figure(figsize=(8,6))
plt.contourf(xx,yy,Z, levels=np.append(uniq, uniq[-1]+1), cmap=cm, alpha=0.12, antialiased=True, zorder=1)  # hard regions (light)
plt.contour(xx,yy,Z, levels=uniq, colors='k', linewidths=0.5, alpha=0.7, zorder=2)                           # hard edges
# tolerance band where classifier is less than (1 - tol) confident
tol_mask = pmax < (1.0 - KNN_TOLMAX)
plt.contourf(xx,yy,tol_mask, levels=[0.5,1.5], colors=['#999999'], alpha=0.18, zorder=1)                     # tol zone

plt.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=cluster_assignments, s=20, edgecolors='k', lw=0.3, alpha=0.35, label='WT', zorder=3)

# pairs with conservative coloring
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before = np.asarray(pca_data['stab_before'])[:n_pairs]
after  = np.asarray(pca_data['stab_after'])[:n_pairs]

# Conservative approach: find all classes with significant probability
def _get_likely_classes(points, threshold=KNN_PROB_THRESHOLD):
    """Return sets of likely classes for each point based on probability threshold"""
    probas = knn.predict_proba(points)  # shape (n_points, n_classes)
    likely_classes = []
    
    for i in range(len(points)):
        # Get all classes with probability >= threshold, or at least the top class
        probs = probas[i]
        above_threshold = set(knn.classes_[probs >= threshold])
        
        # If no class meets threshold, use the top class
        if not above_threshold:
            top_class = knn.classes_[np.argmax(probs)]
            above_threshold = {top_class}
            
        likely_classes.append(above_threshold)
    
    return likely_classes

b_likely = _get_likely_classes(before)
a_likely = _get_likely_classes(after)

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    is_stable = len(b_likely[i] & a_likely[i]) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1.0 if is_stable else 1.2
    
    plt.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
             color=line_color, alpha=line_alpha, lw=line_width, zorder=4)

plt.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before', zorder=5)
plt.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After',  zorder=5)

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1:.1%})'); plt.ylabel(f'PC2 ({pc2:.1%})')
plt.title(f'kNN: hard (k={k}) + tolerance (max p < {1-KNN_TOLMAX:.2f}) (Conservative)')

# Combine existing legend with line color legend
handles, labels = plt.gca().get_legend_handles_labels()
handles.extend(legend_elements)
plt.legend(handles=handles); plt.grid(alpha=0.3); plt.tight_layout()

out = OUTPUT_DIR / f"knn_pca_hard_tol_k{k}.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability ---
stable = sum(1 for i in range(n_pairs) if len(b_likely[i] & a_likely[i]) > 0)
unstable = n_pairs - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title('Conservative kNN Stability (hard decision)'); plt.tight_layout()
out = OUTPUT_DIR / f"knn_stability_pie_k{k}.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[kNN] Conservative: k={k}  tol(maxP)<{1-KNN_TOLMAX:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_likely[i] & a_likely[i]) > 0 and (len(b_likely[i]) > 1 or len(a_likely[i]) > 1):
        overlap_cases.append((i, b_likely[i], a_likely[i], b_likely[i] & a_likely[i]))

if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with kNN class overlaps:")
    for i, before_classes, after_classes, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_classes}, After={after_classes}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")

### F.16 Stability Bootstrap Analysis

Bootstrap resampling compares observed bouton shifts to random expectations and visualizes representative random pairs, strengthening conclusions about genuine remodeling.


In [ ]:
# ==== Cell 7 — Random bootstrap + (NEW) distance comparison + PCA overlay + random-pair mean traces ====
# knobs
BOOT_ITERS       = 2000     # bootstrap iterations
RANDOM_SEED      = 42       # RNG seed
SHOW_PAIRS_PLOT  = 50       # max random pairs to draw on PCA

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

# --- ellipse models (reuse from earlier cells or rebuild with Cell 0 helper) ---
def _ensure_ellipse_models():
    if 'ellipse_models' in locals() and 'chi2_thr' in locals():
        return ellipse_models, chi2_thr
    return build_ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS, alpha=ELLIPSE_ALPHA)

ellipse_models, chi2_thr = _ensure_ellipse_models()

def _ellipses_containing(x):
    # mahalanobis containment check
    hit = set()
    for m in ellipse_models:
        d = x - m['center']
        if float(d.T @ m['inv_cov'] @ d) <= chi2_thr:
            hit.add(m['cluster'])
    return hit

def _bootstrap_random_pairs(n_pairs_in, iters=2000, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(pca_coordinates, float); N=len(X)
    n_eff = int(min(n_pairs_in, N//2))
    if n_eff <= 0: raise RuntimeError("[bootstrap] Not enough WT points to form random pairs.")
    props = np.empty(iters, float)
    # keep one pairing for visualization & distance comparison
    perm = rng.permutation(N)
    idx1_vis = perm[:n_eff]; idx2_vis = perm[n_eff:2*n_eff]
    for t in range(iters):
        perm = rng.permutation(N)
        idx1 = perm[:n_eff]; idx2 = perm[n_eff:2*n_eff]
        shared = 0
        for i1, i2 in zip(idx1, idx2):
            if _ellipses_containing(X[i1]).intersection(_ellipses_containing(X[i2])):
                shared += 1
        props[t] = shared / n_eff
    return props, n_eff, (idx1_vis, idx2_vis)

# observed stability (ellipses, hard)
before_coords = np.asarray(pca_data['stab_before'])
after_coords  = np.asarray(pca_data['stab_after'])
n_pairs = int(min(len(before_coords), len(after_coords)))

def _hard_label(x):
    inside=[(m['cluster'], float((x-m['center']).T @ m['inv_cov'] @ (x-m['center'])))
            for m in ellipse_models if float((x-m['center']).T @ m['inv_cov'] @ (x-m['center'])) <= chi2_thr]
    if inside: return min(inside, key=lambda t:t[1])[0]
    # nearest-edge proxy by boundary gap
    gaps=[(m['cluster'], np.sqrt(float((x-m['center']).T @ m['inv_cov'] @ (x-m['center']))/chi2_thr)-1.0)
          for m in ellipse_models]
    return min(gaps, key=lambda t:t[1])[0]

b_h = np.array([_hard_label(before_coords[i]) for i in range(n_pairs)])
a_h = np.array([_hard_label(after_coords[i])  for i in range(n_pairs)])
observed_prop = float(np.sum(b_h==a_h)) / n_pairs

# --- run bootstrap ---
props, n_eff, (idx1_vis, idx2_vis) = _bootstrap_random_pairs(n_pairs, iters=BOOT_ITERS, seed=RANDOM_SEED)
p_ge  = float(np.mean(props >= observed_prop))
p_two = float(np.mean(np.abs(props - props.mean()) >= abs(observed_prop - props.mean())))

# (A0) NEW — Distance comparison: observed before→after vs random-pair distances
obs_dists = np.linalg.norm(after_coords[:n_pairs] - before_coords[:n_pairs], axis=1).astype(float)
rand_dists = np.linalg.norm(pca_coordinates[idx2_vis[:n_eff]] - pca_coordinates[idx1_vis[:n_eff]], axis=1).astype(float)

# Freedman–Diaconis bins over combined set
combined = np.concatenate([obs_dists, rand_dists])
q25,q75 = np.percentile(combined,[25,75]); iqr = float(q75-q25); N = len(combined)
bw = (2*iqr)/(N**(1/3)) if iqr>0 else 0.0
n_bins = max(8, int(np.ceil((combined.max()-combined.min())/bw))) if bw>0 else max(8, int(np.ceil(np.sqrt(N))))
bins = np.linspace(combined.min(), combined.max(), n_bins + 1)

plt.figure(figsize=(8.4,5.6))
plt.hist(rand_dists, bins=bins, alpha=0.45, edgecolor='black', label=f'Random pairs (n={len(rand_dists)})')
plt.hist(obs_dists,  bins=bins, alpha=0.45, edgecolor='black', label=f'Observed before→after (n={len(obs_dists)})')
plt.axvline(rand_dists.mean(), color='C0', ls='--', lw=2, label=f'Random mean={rand_dists.mean():.2f}')
plt.axvline(obs_dists.mean(),  color='C1', ls='--', lw=2, label=f'Observed mean={obs_dists.mean():.2f}')
plt.xlabel('Distance in PCA space'); plt.ylabel('Count'); plt.title('Distance distributions: Random vs Observed')
plt.legend(); plt.tight_layout()
out_dist = OUTPUT_DIR / "random_vs_observed_distances.pdf"
plt.savefig(out_dist, dpi=300, bbox_inches='tight'); plt.show()

# (A) PCA overlay of a subset of random pairs
max_show = min(SHOW_PAIRS_PLOT, n_eff)
x = pca_coordinates[:,0]; y = pca_coordinates[:,1]
plt.figure(figsize=(8,6))
plt.scatter(x, y, c=cluster_assignments, s=20, alpha=0.35, label='WT background')
for i1, i2 in zip(idx1_vis[:max_show], idx2_vis[:max_show]):
    plt.plot([x[i1], x[i2]], [y[i1], y[i2]], color='tab:blue', alpha=0.6, lw=1.2)
plt.scatter(x[idx1_vis[:max_show]], y[idx1_vis[:max_show]], c='tab:blue', s=35, edgecolors='k', lw=0.3, label='Random pair A')
plt.scatter(x[idx2_vis[:max_show]], y[idx2_vis[:max_show]], c='tab:cyan',  s=35, edgecolors='k', lw=0.3, label='Random pair B', marker='s')
pc1, pc2 = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1:.1%})'); plt.ylabel(f'PC2 ({pc2:.1%})')
plt.title(f'Random WT pairs overlay (showing {max_show} of {n_eff})')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
out_pca = OUTPUT_DIR / "random_pairs_pca_overlay.pdf"
plt.savefig(out_pca, dpi=300, bbox_inches='tight'); plt.show()

# (B) Random-pair mean traces (if index→trace mapping is available)
def _get_trace_by_index(idx):
    if 'WT_IDS' in locals() and 'resampled_trace_lookup' in locals():
        _id = WT_IDS[idx]
        if _id in resampled_trace_lookup and 'Avg' in resampled_trace_lookup[_id]:
            tr = np.asarray(resampled_trace_lookup[_id]['Avg'], float)
            return tr if np.isfinite(tr).all() and len(tr)==len(COMMON_TIME) else None
    if 'WT_TRACES_ARRAY' in locals() and len(WT_TRACES_ARRAY) == len(pca_coordinates):
        tr = np.asarray(WT_TRACES_ARRAY[idx], float)
        return tr if np.isfinite(tr).all() and len(tr)==len(COMMON_TIME) else None
    return None

pair_means = []
for i1, i2 in zip(idx1_vis[:n_eff], idx2_vis[:n_eff]):
    t1 = _get_trace_by_index(i1); t2 = _get_trace_by_index(i2)
    if t1 is not None and t2 is not None:
        pair_means.append(0.5*(t1+t2))
pair_means = np.vstack(pair_means) if len(pair_means)>0 else np.empty((0, len(COMMON_TIME)))

if pair_means.size > 0:
    m = np.nanmean(pair_means, axis=0)
    s = np.nanstd(pair_means, axis=0, ddof=1)/np.sqrt(max(1, pair_means.shape[0]))
    plt.figure(figsize=(8.5,5.0))
    plt.plot(COMMON_TIME, m, lw=2.0, label=f'Random-pair mean (n={pair_means.shape[0]})')
    plt.fill_between(COMMON_TIME, m-s, m+s, alpha=0.2, lw=0)
    if 'stats' in locals():  # overlay observed means from Cell 3
        for c, col, ls in [("Stability_Before","#1f77b4","-"), ("Stability_After","#d62728","-")]:
            if c in stats:
                mm, ss = stats[c]; plt.plot(COMMON_TIME, mm, color=col, ls=ls, lw=1.5, alpha=0.9, label=f'{c} mean')
    plt.axhline(0, color='gray', ls=':', lw=1.0)
    plt.xlabel('Time (s)'); plt.ylabel('ΔF/F'); plt.title('Random-pair mean traces (±SEM)')
    plt.legend(); plt.tight_layout()
    out_tr = OUTPUT_DIR / "random_pairs_mean_traces.pdf"
    plt.savefig(out_tr, dpi=300, bbox_inches='tight'); plt.show()
else:
    print("Random-pair traces: skipped (no index→trace mapping).")

# (C) Box vs observed (assignment sharing)
plt.figure(figsize=(8.2,6.0))
plt.boxplot(props, positions=[1], patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.7),
            medianprops=dict(color='navy', linewidth=2))
plt.scatter([2], [observed_prop], color='red', s=110, zorder=5, label='Observed stability')
plt.axhline(props.mean(), color='blue', linestyle='--', alpha=0.7, label='Random mean')
plt.xticks([1,2], [f'Random pairs\n({len(props)} iters)', 'Observed'])
plt.ylabel('Proportion sharing ≥1 ellipse'); plt.title('Random vs Observed stability (ellipses)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
out_box = OUTPUT_DIR / "ellipse_random_vs_observed.pdf"
plt.savefig(out_box, dpi=300, bbox_inches='tight'); plt.show()

# --- summary ---
print(f"DISTANCES: random mean={rand_dists.mean():.3f} (n={len(rand_dists)}) | observed mean={obs_dists.mean():.3f} (n={len(obs_dists)})  ✓ {out_dist}")
print(f"ASSIGNMENTS: random mean={props.mean():.3f} ± {props.std(ddof=1):.3f}, observed={observed_prop:.3f}, p>=obs={p_ge:.4f}, p(two)={p_two:.4f}")
print(f"PLOTS: PCA overlay ✓ {out_pca} | Distances ✓ {out_dist} | Box ✓ {out_box}" )


### F Prelude – Alternative Cluster Boundaries

Exploring alpha-shape and related boundaries tests the robustness of stability conclusions to different geometric definitions of clusters.


### F Prelude – Supplemental Stability Figures

Additional figures summarize stability-related plasticity metrics beyond PCA space.


#### F Focus – Additional Comparisons

We conclude the stability section with targeted comparisons of plasticity metrics that complement the PCA-based views.


### F.17 Stability Plasticity Profiles

Paired-pulse and amplitude profiles are recalculated for before and after datasets with statistical annotations to detect systematic shifts in short-term plasticity.


In [ ]:
# Compare PPR and amplitude profiles before vs after treatment
from scipy.stats import ttest_rel

def plot_stability_profiles():
    """Plot PPR and amplitude profiles with statistical testing."""
    
    # PPR analysis
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_Stability_Before.columns]
    pulse_numbers = list(range(1, len(ppr_cols) + 2))
    
    # Calculate PPR profiles
    ppr_before = [1.0] + PCA_Data_Stability_Before[ppr_cols].mean().tolist()
    ppr_before_sem = [0.0] + PCA_Data_Stability_Before[ppr_cols].sem().tolist()
    ppr_after = [1.0] + PCA_Data_Stability_After[ppr_cols].mean().tolist()
    ppr_after_sem = [0.0] + PCA_Data_Stability_After[ppr_cols].sem().tolist()
    
    # Amplitude analysis
    amp_cols = [f'AMP{i}' for i in range(1, 11) if f'AMP{i}' in PCA_Data_Stability_Before.columns]
    amp_before = PCA_Data_Stability_Before[amp_cols].mean().values
    amp_before_sem = PCA_Data_Stability_Before[amp_cols].sem().values
    amp_after = PCA_Data_Stability_After[amp_cols].mean().values
    amp_after_sem = PCA_Data_Stability_After[amp_cols].sem().values
    
    # Create plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # PPR plot
    ax1.plot(pulse_numbers, ppr_before, marker='o', color='orange', linewidth=2, label='Before')
    ax1.fill_between(pulse_numbers, np.array(ppr_before) - np.array(ppr_before_sem),
                     np.array(ppr_before) + np.array(ppr_before_sem), color='orange', alpha=0.2)
    ax1.plot(pulse_numbers, ppr_after, marker='s', color='brown', linewidth=2, label='After')
    ax1.fill_between(pulse_numbers, np.array(ppr_after) - np.array(ppr_after_sem),
                     np.array(ppr_after) + np.array(ppr_after_sem), color='brown', alpha=0.2)
    ax1.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    ax1.set_xlabel('Pulse Number')
    ax1.set_ylabel('PPR (A_n/A_1)')
    ax1.set_title('PPR Profiles: Before vs After')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Amplitude plot
    amp_pulse_numbers = list(range(1, len(amp_cols) + 1))
    ax2.plot(amp_pulse_numbers, amp_before, marker='o', color='orange', linewidth=2, label='Before')
    ax2.fill_between(amp_pulse_numbers, amp_before - amp_before_sem, amp_before + amp_before_sem, 
                     color='orange', alpha=0.2)
    ax2.plot(amp_pulse_numbers, amp_after, marker='s', color='brown', linewidth=2, label='After')
    ax2.fill_between(amp_pulse_numbers, amp_after - amp_after_sem, amp_after + amp_after_sem,
                     color='brown', alpha=0.2)
    ax2.set_xlabel('Pulse Number')
    ax2.set_ylabel('Amplitude')
    ax2.set_title('Amplitude Profiles: Before vs After')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    output_file = OUTPUT_DIR / "stability_ppr_amplitude_profiles.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Statistical testing for PPR
    print("=== PPR STATISTICAL ANALYSIS ===")
    for i, col in enumerate(ppr_cols, start=2):
        t_stat, p_val = ttest_rel(PCA_Data_Stability_Before[col], PCA_Data_Stability_After[col], nan_policy='omit')
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        print(f"Pulse {i}: t={t_stat:.3f}, p={p_val:.3e} ({sig})")
    
    return output_file

plot_output = plot_stability_profiles()
print(f"✓ Saved profiles to {plot_output}")

### F.18 Stability Parameter Comparisons

Detailed paired statistical tests are run for amplitudes, failure rates, and key ratios, with significance markers highlighting which parameters change reliably.


In [ ]:
# Detailed statistical comparisons for key parameters
def plot_stability_comparisons():
    """Create paired boxplots with statistical tests."""
    
    def add_significance_bar(ax, p_value, positions=[0, 1]):
        """Add significance bar above boxplot."""
        y_max = ax.get_ylim()[1]
        y_min = ax.get_ylim()[0]
        y_sig = y_max + 0.05 * (y_max - y_min)
        
        # Significance stars
        if p_value < 0.001:
            sig_text = '***'
        elif p_value < 0.01:
            sig_text = '**'
        elif p_value < 0.05:
            sig_text = '*'
        else:
            sig_text = 'ns'
        
        # Draw bar and text
        ax.plot(positions, [y_sig, y_sig], color='black', linewidth=1.5)
        ax.text(np.mean(positions), y_sig + 0.01 * (y_max - y_min), sig_text,
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # AMP1 comparison
    amp1_data = pd.DataFrame({
        'AMP1': pd.concat([PCA_Data_Stability_Before['AMP1'], PCA_Data_Stability_After['AMP1']]),
        'Condition': ['Before'] * len(PCA_Data_Stability_Before) + ['After'] * len(PCA_Data_Stability_After)
    })
    
    sns.boxplot(data=amp1_data, x='Condition', y='AMP1', ax=axes[0], palette=['orange', 'brown'])
    sns.stripplot(data=amp1_data, x='Condition', y='AMP1', ax=axes[0], color='black', size=3, alpha=0.6)
    
    # Add connecting lines for paired data
    n_pairs = min(len(PCA_Data_Stability_Before), len(PCA_Data_Stability_After))
    for i in range(n_pairs):
        axes[0].plot([0, 1], [PCA_Data_Stability_Before['AMP1'].iloc[i], PCA_Data_Stability_After['AMP1'].iloc[i]],
                     color='gray', alpha=0.4, linewidth=1)
    
    t_stat_amp1, p_val_amp1 = ttest_rel(PCA_Data_Stability_Before['AMP1'], PCA_Data_Stability_After['AMP1'])
    add_significance_bar(axes[0], p_val_amp1)
    axes[0].set_title('AMP1: Before vs After')
    
    # PPR2/1 comparison
    ppr_data = pd.DataFrame({
        'PPR2/1': pd.concat([PCA_Data_Stability_Before['PPR2/1'], PCA_Data_Stability_After['PPR2/1']]),
        'Condition': ['Before'] * len(PCA_Data_Stability_Before) + ['After'] * len(PCA_Data_Stability_After)
    })
    
    sns.boxplot(data=ppr_data, x='Condition', y='PPR2/1', ax=axes[1], palette=['orange', 'brown'])
    sns.stripplot(data=ppr_data, x='Condition', y='PPR2/1', ax=axes[1], color='black', size=3, alpha=0.6)
    
    for i in range(n_pairs):
        axes[1].plot([0, 1], [PCA_Data_Stability_Before['PPR2/1'].iloc[i], PCA_Data_Stability_After['PPR2/1'].iloc[i]],
                     color='gray', alpha=0.4, linewidth=1)
    
    t_stat_ppr, p_val_ppr = ttest_rel(PCA_Data_Stability_Before['PPR2/1'], PCA_Data_Stability_After['PPR2/1'])
    add_significance_bar(axes[1], p_val_ppr)
    axes[1].set_title('PPR2/1: Before vs After')
    
    # Baseline fluorescence (F0) analysis
    if 'stab_before_traces' in locals() and 'stab_after_traces' in locals():
        # Calculate baseline from first 0.5s of traces
        baseline_before = [np.mean(trace[:int(0.5 * len(COMMON_TIME))]) for trace in stab_before_traces]
        baseline_after = [np.mean(trace[:int(0.5 * len(COMMON_TIME))]) for trace in stab_after_traces]
        
        f0_data = pd.DataFrame({
            'F0': baseline_before + baseline_after,
            'Condition': ['Before'] * len(baseline_before) + ['After'] * len(baseline_after)
        })
        
        sns.boxplot(data=f0_data, x='Condition', y='F0', ax=axes[2], palette=['orange', 'brown'])
        sns.stripplot(data=f0_data, x='Condition', y='F0', ax=axes[2], color='black', size=3, alpha=0.6)
        
        # Connect paired points
        for i in range(min(len(baseline_before), len(baseline_after))):
            axes[2].plot([0, 1], [baseline_before[i], baseline_after[i]],
                         color='gray', alpha=0.4, linewidth=1)
        
        t_stat_f0, p_val_f0 = ttest_rel(baseline_before, baseline_after)
        add_significance_bar(axes[2], p_val_f0)
        axes[2].set_title('Baseline F0: Before vs After')
        axes[2].set_ylabel('F0 (ΔF/F)')
    else:
        axes[2].text(0.5, 0.5, 'F0 data\nnot available', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_title('Baseline F0')
    
    plt.tight_layout()
    output_file = OUTPUT_DIR / "stability_statistical_comparisons.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print results
    print("=== STABILITY STATISTICAL RESULTS ===")
    print(f"AMP1: t={t_stat_amp1:.3f}, p={p_val_amp1:.4g}")
    print(f"PPR2/1: t={t_stat_ppr:.3f}, p={p_val_ppr:.4g}")
    
    return output_file

comparison_output = plot_stability_comparisons()
print(f"✓ Saved comparisons to {comparison_output}")

#### F Focus – Stability Amplitude and PPR Comparisons

Histograms and paired statistics highlight how Amp1 and PPR2/1 values evolve between the stability experiment's before and after states.


##### F Detail – Histogram Perspective

Viewing the paired distributions clarifies whether shifts arise from global scaling or from specific subpopulations of boutons.


### F.19 Stability Synthesis

A textual summary consolidates sample sizes, significant metrics, and interpretation from the stability analysis, ensuring the narrative captures the most impactful findings.


In [ ]:
# Final stability analysis summary
def stability_summary():
    """Generate comprehensive stability analysis summary."""
    
    print("=" * 60)
    print("COMPREHENSIVE STABILITY ANALYSIS SUMMARY")
    print("=" * 60)
    
    # Sample sizes
    n_before = len(PCA_Data_Stability_Before)
    n_after = len(PCA_Data_Stability_After)
    n_pairs = min(n_before, n_after)
    
    print(f"\nSample sizes:")
    print(f"  Before: {n_before} boutons")
    print(f"  After: {n_after} boutons")
    print(f"  Paired: {n_pairs} boutons")
    
    # Key parameter changes
    print(f"\nKey parameter changes (Before → After):")
    
    # AMP1
    amp1_before_mean = PCA_Data_Stability_Before['AMP1'].mean()
    amp1_after_mean = PCA_Data_Stability_After['AMP1'].mean()
    amp1_change = ((amp1_after_mean - amp1_before_mean) / amp1_before_mean) * 100
    print(f"  AMP1: {amp1_before_mean:.3f} → {amp1_after_mean:.3f} ({amp1_change:+.1f}%)")
    
    # PPR2/1
    ppr_before_mean = PCA_Data_Stability_Before['PPR2/1'].mean()
    ppr_after_mean = PCA_Data_Stability_After['PPR2/1'].mean()
    ppr_change = ((ppr_after_mean - ppr_before_mean) / ppr_before_mean) * 100
    print(f"  PPR2/1: {ppr_before_mean:.3f} → {ppr_after_mean:.3f} ({ppr_change:+.1f}%)")
    
    # PCA movement analysis (if available)
    if 'movement_distances' in locals():
        print(f"\nPCA space movement:")
        print(f"  Mean distance: {np.mean(movement_distances):.3f} ± {np.std(movement_distances):.3f}")
        print(f"  Max distance: {np.max(movement_distances):.3f}")
        print(f"  Boutons with large movement (>mean): {np.sum(movement_distances > np.mean(movement_distances))}/{len(movement_distances)}")
    
    # Cluster stability (if available)
    if 'stable_pairs' in locals() and 'unstable_pairs' in locals():
        total_analyzed = stable_pairs + unstable_pairs
        stability_pct = (stable_pairs / total_analyzed) * 100
        print(f"\nCluster assignment stability:")
        print(f"  Stable pairs: {stable_pairs}/{total_analyzed} ({stability_pct:.1f}%)")
        print(f"  Unstable pairs: {unstable_pairs}/{total_analyzed} ({100-stability_pct:.1f}%)")
    
    print(f"\n" + "=" * 60)
    print("Analysis complete - all figures saved to OUTPUT_DIR")
    print("=" * 60)

# Run summary
stability_summary()

### Appendix – Interactive Utilities

The remaining cells provide environment safeguards and exploratory widgets that complement the primary analysis.


### F.20 Visualization Backend Safeguards

Matplotlib backends are configured with fallbacks to guarantee that interactive and static plots render correctly across execution environments.


In [ ]:
# Fix matplotlib display issues
import matplotlib
import warnings

# Configure matplotlib for proper display
try:
    # Try Qt5Agg first (best for interactive features)
    matplotlib.use('Qt5Agg')
    print("✓ Using Qt5Agg backend for interactive features")
except:
    try:
        # Fallback to inline for basic display
        matplotlib.use('inline')
        print("✓ Using inline backend for basic display")
    except:
        print("⚠ Using default backend")

# Configure for notebook
%matplotlib inline
plt.ion()  # Interactive mode
plt.close('all')  # Clear any existing figures

# Suppress common warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', message='.*GUI.*')

print(f"Current backend: {matplotlib.get_backend()}")
print("Figures should now display properly")

### E.12 Interactive Fiber Exploration

Interacting tools are prepared to let users probe bouton clusters along individual fibers within the PCA space. Buttons and callbacks provide an exploratory interface to inspect structural-functional relationships.


In [ ]:
# Interactive fiber analysis in PCA space
from matplotlib.widgets import Button

def extract_fiber_id(bouton_id, dataset_type):
    """Extract fiber ID based on dataset-specific rules."""
    id_lengths = {
        'SynII': 13,
        'WT_Anthime': 22, 
        'WT_Theo': 18,
        'WT_pooled': 22  # Default
    }
    length = id_lengths.get(dataset_type, 22)
    return str(bouton_id)[:length]

def create_fiber_dataset():
    """Create unified fiber dataset from all available data."""
    fiber_data = []
    
    # Add WT pooled data (main dataset)
    for i, row in PCA_Data_WT_Pooled_clustered.iterrows():
        if i < len(pca_coordinates):
            fiber_data.append({
                'ID': row['ID'],
                'PC1': pca_coordinates[i, 0],
                'PC2': pca_coordinates[i, 1],
                'cluster': cluster_assignments[i],
                'dataset': 'WT_pooled',
                'fiber_id': extract_fiber_id(row['ID'], 'WT_pooled'),
                'color': cluster_assignments[i],
                'marker': 'o'
            })
    
    # Group by fiber ID
    fibers = {}
    for point in fiber_data:
        fid = point['fiber_id']
        if fid not in fibers:
            fibers[fid] = []
        fibers[fid].append(point)
    
    # Filter for fibers with multiple boutons
    multi_bouton_fibers = {k: v for k, v in fibers.items() if len(v) > 1}
    
    return multi_bouton_fibers

class FiberNavigator:
    def __init__(self, fibers_dict):
        self.fibers = fibers_dict
        self.fiber_list = list(fibers_dict.keys())
        self.current_index = 0
        
        if len(self.fiber_list) == 0:
            print("No multi-bouton fibers found")
            return
        
        # Create figure
        self.fig, self.ax = plt.subplots(figsize=(12, 8))
        self.fig.subplots_adjust(bottom=0.15)
        
        # Navigation buttons
        ax_prev = plt.axes([0.3, 0.02, 0.1, 0.06])
        ax_next = plt.axes([0.6, 0.02, 0.1, 0.06])
        self.btn_prev = Button(ax_prev, 'Previous')
        self.btn_next = Button(ax_next, 'Next')
        
        self.btn_prev.on_clicked(self.prev_fiber)
        self.btn_next.on_clicked(self.next_fiber)
        
        # Keyboard navigation
        self.fig.canvas.mpl_connect('key_press_event', self.on_key)
        
        self.update_plot()
    
    def prev_fiber(self, event=None):
        if self.current_index > 0:
            self.current_index -= 1
            self.update_plot()
    
    def next_fiber(self, event=None):
        if self.current_index < len(self.fiber_list) - 1:
            self.current_index += 1
            self.update_plot()
    
    def on_key(self, event):
        if event.key == 'left':
            self.prev_fiber()
        elif event.key == 'right':
            self.next_fiber()
    
    def update_plot(self):
        self.ax.clear()
        
        # Background points
        self.ax.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], c=cluster_assignments, alpha=0.3, s=20, label='WT background')
        
        # Current fiber
        fiber_id = self.fiber_list[self.current_index]
        fiber_boutons = self.fibers[fiber_id]
        
        # Sort by ID for consistent ordering
        fiber_boutons.sort(key=lambda x: x['ID'])
        
        # Extract coordinates and clusters
        x_coords = [b['PC1'] for b in fiber_boutons]
        y_coords = [b['PC2'] for b in fiber_boutons]
        clusters = [b['cluster'] for b in fiber_boutons]
        
        # Connection line
        self.ax.plot(x_coords, y_coords, 'red', linewidth=2.5, alpha=0.8, 
                    label=f'Fiber connection')
        
        # Boutons with cluster colors and numbers
        cluster_colormap = plt.get_cmap('Set2')
        for i, bouton in enumerate(fiber_boutons):
            color = cluster_colormap(bouton['cluster'] - 1)
            self.ax.scatter(bouton['PC1'], bouton['PC2'], 
                           c=color, s=120, edgecolors='black', linewidth=2, 
                           zorder=5, marker='o')
            
            # Number annotation
            self.ax.annotate(f"{i+1}\nC{bouton['cluster']}", 
                           (bouton['PC1'], bouton['PC2']),
                           xytext=(8, 8), textcoords='offset points',
                           fontsize=9, fontweight='bold', ha='center',
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                                   edgecolor='black', alpha=0.9))
        
        # Cluster diversity analysis
        unique_clusters = set(clusters)
        diversity_info = f"Clusters: {sorted(unique_clusters)} ({len(unique_clusters)} types)"
        
        self.ax.set_xlabel('PC1')
        self.ax.set_ylabel('PC2')
        self.ax.set_title(f'Fiber {self.current_index + 1}/{len(self.fiber_list)}: {fiber_id}\n'
                         f'Boutons: {len(fiber_boutons)} | {diversity_info}')
        
        self.ax.legend()
        self.ax.grid(True, alpha=0.3)
        
        # Navigation instruction
        self.fig.suptitle('Use ← → arrows or buttons to navigate', fontsize=10, y=0.02)
        
        self.fig.canvas.draw()

# Create fiber dataset and navigator
fiber_dataset = create_fiber_dataset()

if len(fiber_dataset) > 0:
    print(f"Found {len(fiber_dataset)} fibers with multiple boutons")
    
    # Create navigator
    navigator = FiberNavigator(fiber_dataset)
    
    # Show plot
    plt.show()
    
    # Fiber diversity summary
    fiber_diversity = {}
    for fid, boutons in fiber_dataset.items():
        clusters = set(b['cluster'] for b in boutons)
        n_clusters = len(clusters)
        if n_clusters not in fiber_diversity:
            fiber_diversity[n_clusters] = 0
        fiber_diversity[n_clusters] += 1
    
    print(f"\nFiber cluster diversity:")
    for n_types, count in sorted(fiber_diversity.items()):
        print(f"  {n_types} cluster type(s): {count} fibers")

else:
    print("No multi-bouton fibers found in dataset")